# Suite DIA — The Python3, TypeScript5, MATLAB, Excel, LaTeX, C, Cpp and SystemVerilog dialects

Each dialect models a language's expressions: its kinds, its vocabulary and the domains of its values, `render`
to its source text, an evaluator with the language's own rules, and a scope that resolves names, imports and
references as the language does. Python evaluates by Python's rules (modeled over JavaScript values here);
MATLAB is two-valued over doubles, logicals and strings, with functions from the path and packages; Excel coerces,
computes with error values, and reads cells from workbooks. LaTeX writes notation, and has no evaluator. C evaluates by C's
rules over its arithmetic types (and Cpp, which shares its kinds, by C++'s), and SystemVerilog by IEEE 1800's over
4-state vectors and reals. Each dialect's install adds its union of its kinds, named plainly, to the framework's
`Term`, as the branch named after it, and its terms are written in a store through its writers; it reads only its own
terms. What every dialect shares is suite FRM; the Patterns dialect is suites EXP and EVL. Reading Python
source (`parse`) and NumPy are the Python implementation's only.

In [ ]:
import { Framework } from "@mbse/expressions";
import * as IEEE1164 from "@mbse/expressions/IEEE1164/Expressions";
import * as IEEE754 from "@mbse/expressions/IEEE754/Expressions";
import * as ISOIEC11404 from "@mbse/expressions/ISOIEC11404/Expressions";
import * as Unicode from "@mbse/expressions/Unicode/Expressions";
import * as Patterns from "@mbse/expressions/Patterns/Expressions";
import * as CDialect from "@mbse/expressions/C/Expressions";
import * as CppDialect from "@mbse/expressions/Cpp/Expressions";
import * as Excel from "@mbse/expressions/Excel/Expressions";
import * as Latex from "@mbse/expressions/LaTeX/Expressions";
import * as Matlab from "@mbse/expressions/MATLAB/Expressions";
import * as Python3 from "@mbse/expressions/Python/Python3";
import * as AAPCS64 from "@mbse/expressions/Arm/AAPCS64";
import * as AMD64 from "@mbse/expressions/SystemV/AMD64";
import * as SystemVerilog from "@mbse/expressions/SystemVerilog/Expressions";
import * as SVAssertions from "@mbse/expressions/SystemVerilog/Assertions";
import * as SVConstraints from "@mbse/expressions/SystemVerilog/Constraints";
import * as SVPatterns from "@mbse/expressions/SystemVerilog/Patterns";
import * as SVPreprocessor from "@mbse/expressions/SystemVerilog/Preprocessor";
import * as SVTypes from "@mbse/expressions/SystemVerilog/Types";
import * as TypeScript5 from "@mbse/expressions/TypeScript/TypeScript5";
import * as CPreprocessor from "@mbse/expressions/C/Preprocessor"; // the retrofit's new dialects
import * as CTypes from "@mbse/expressions/C/Types";
import * as CppTypes from "@mbse/expressions/Cpp/Types";
import * as PythonPatterns from "@mbse/expressions/Python/Patterns";
import * as PythonTypes from "@mbse/expressions/Python/Types";
import * as TypeScriptPatterns from "@mbse/expressions/TypeScript/Patterns";
import * as TypeScriptTypes from "@mbse/expressions/TypeScript/Types";
import { Domains as D, Errors as FE, Symbolics as SY, Terms as F } from "@mbse/expressions/Framework";
import { Builders, Errors, JSON, Repr, YAML } from "@mbse/schemas/Framework";
import { assert, equal, map, raises, same } from "./support.js";

const { AttributeError, KeyError, NotImplementedError, ValueError } = Errors;
const { ImportError, IndexError, NameError, OverflowError, ZeroDivisionError } = FE;

const [P, M, X] = [Python3.Expressions, Matlab.Expressions, Excel.Expressions];
const [PV, MV, XV] = [Python3.Evaluators, Matlab.Evaluators, Excel.Evaluators];
const [PD, MD, XD] = [Python3.Domains, Matlab.Domains, Excel.Domains];
const [C, CV, CD] = [CDialect.Expressions, CDialect.Evaluators, CDialect.Domains];
const [SV, SVV, SVD] = [SystemVerilog.Expressions, SystemVerilog.Evaluators, SystemVerilog.Domains];
const SVT = SVTypes.Expressions;
const [TS, TV, TSText] = [TypeScript5.Expressions, TypeScript5.Evaluators, TypeScript5.Text];
const [TSP, TST] = [TypeScriptPatterns.Expressions, TypeScriptTypes.Expressions];
const [PText, MText, XText, LText, CText, SVText] = [Python3.Text, Matlab.Text, Excel.Text, Latex.Text, CDialect.Text, SystemVerilog.Text];

const store: any = new Builders.Store(); // every dialect's kinds, in one store
for (const install of [Framework.install, ISOIEC11404.install, IEEE754.install, IEEE1164.install, Unicode.install, // the standard
  Patterns.install, Python3.install, Matlab.install, Excel.install, Latex.install, // dialects first
  CDialect.install, CppDialect.install, AMD64.install, AAPCS64.install, SystemVerilog.install, SVTypes.install,
  SVPatterns.install, SVAssertions.install, SVConstraints.install, SVPreprocessor.install, TypeScript5.install,
  PythonTypes.install, PythonPatterns.install, TypeScriptTypes.install, TypeScriptPatterns.install, // each after its
  CTypes.install, CppTypes.install, CPreprocessor.install]) { // family's dialect
  store.transaction(install);
}
store.transaction(Patterns.define); // Patterns' operations, which its validation reads
for (const define of [CDialect.define, CppDialect.define, AMD64.define, AAPCS64.define]) { // C's, C++'s and the ABIs' types,
  store.transaction(define); // which their evaluation, inference and validation read through a term's holder
}
for (const define of [CTypes.define, CppTypes.define, CPreprocessor.define]) store.transaction(define); // their new dialects' data
store.transaction(SystemVerilog.define); // SystemVerilog's types, which its evaluation, inference and validation read
const Term: any = store.registered("Expressions.Framework.Term");
const [AMD64_, AAPCS64_] = [store.Instances.Expressions.SystemV.AMD64, store.Instances.Expressions.Arm.AAPCS64]; // the ABIs

/** The term value `spec` writes, a native value as `dialect`'s literal, the root of a holder written in a transaction;
 * a C or C++ term's holder names the ABI its terms are read under, `abi` (x86-64's System V ABI unless given; null,
 * none). */
function term(spec: unknown, dialect: F.Dialect, abi: unknown = AMD64_): any {
  return F.term(store.transaction(() => store.Expressions.Framework.Expression((e: any) =>
    (["c", "cpp"].includes(dialect.tag) && abi !== null ? e.root(dialect.spec(spec)).abi([abi]) : e.root(dialect.spec(spec))))));
}

/** A dialect's terms, from their specs. */
const termsOf = (dialect: F.Dialect) => (spec: unknown): any => term(spec, dialect);
const [py, mat, xl, tex, cc, sv] = [termsOf(P.DIALECT), termsOf(M.DIALECT), termsOf(X.DIALECT), termsOf(Latex.Expressions.DIALECT),
  termsOf(C.DIALECT), termsOf(SV.DIALECT)];

/** A type name of keywords (`unsigned int`), written through the builders: a spec of C's or C++'s `Type`. */
const keywords = (...words: string[]) => (t: any) => t.typeid((i: any) => i.specifiers(
  words.map((word) => (s: any) => s.primitivetypespecifier((p: any) => p.keyword(word)))));

/** A type name that is a type's name (`uint8_t`), written through the builders: a spec of C's or C++'s `Type`. */
const named_type = (name: string) => (t: any) => t.typeid((i: any) => i.specifiers([(s: any) => s.namedtypespecifier(
  (n: any) => n.name((m: any) => m.identifier((d: any) => d.spelling(name))))]));

/** Whether a term reads back the same from JSON and from YAML. */
function roundTrips(value: unknown): boolean {
  return ([[JSON.ToJSON(store), JSON.FromJSON(store)], [YAML.ToYAML(store), YAML.FromYAML(store)]] as [any, any][]).every(
    ([encode, decode]) => F.same(decode(Term, encode(Term, value)), value));
}

/** The data the expressions read: people, and objects with lists. */
function data(txn: any): any {
  const S = txn.Schemas;
  const ints = () => S.OfContainer((c: any) => c.index([S.OfInt]).element([S.OfInt]));
  return txn.uid("dia.data/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Person: S.OfObject, Squad: S.OfObject, Board: S.OfObject, Bus: S.OfObject })
    .spec([
      (_: any) => _.Person.name("Person").ref(true).properties({ name: S.OfStr, age: S.OfInt, email: S.OfStr }),
      (_: any) => _.Squad.name("Squad").ref(true).properties({ ranks: ints() }),
      (_: any) => _.Board.name("Board").ref(true).properties({ name: S.OfStr, grid: S.OfContainer(
        (c: any) => c.index([S.OfInt]).element([S.OfContainer((d: any) => d.index([S.OfInt]).element([S.OfInt]))])) }),
      (_: any) => _.Bus.name("Bus").ref(true).properties({ name: S.OfStr, ports: ints() }),
    ])]))]);
}

store.transaction(data);
const [ann, bob, squad, board, bus] = store.transaction(() => [
  store.Person((p: any) => p.name("Ann").age(30n).email("ann@example.com")),
  store.Person((p: any) => p.name("Bob").age(70n)), // no email
  store.Squad((s: any) => s.ranks([1n, 2n, 3n])), // a list property is an array of its values
  store.Board((b: any) => b.name("b").grid([(e: any) => e.as_container([(x: any) => x.as_scalar(1n), (x: any) => x.as_scalar(2n)]),
    (e: any) => e.as_container([(x: any) => x.as_scalar(3n)])])),
  store.Bus((b: any) => b.name("b").ports([4n, 5n])),
]);

<a id="DIA-01"></a>
## DIA-01 · Python: expressions as `ast` has them, with their imports; rendered and parsed as source

In [ ]:
{
  const [t, a, b, c, d, x, y, z] = [..."tabcdxyz"].map((n) => P.name(n));
  for (const [built, source] of [ // written with constructors, as source
    [P.import_("numpy", P.importfrom("numpy.ma", "getmaskarray", P.call(
      "np.logical_and", P.call("np.greater_equal", P.subscript(t, "age"), 18n),
      P.call("np.logical_not", P.call("getmaskarray", P.subscript(t, "email"))))), "np"),
    "import numpy as np\nfrom numpy.ma import getmaskarray\n"
      + "np.logical_and(np.greater_equal(t['age'], 18), np.logical_not(getmaskarray(t['email'])))"],
    [P.let_("a", t, P.boolop("and", P.compare(">", P.attribute(a, "b"), 1n), P.unaryop("not", P.attribute(a, "c")))),
      "(lambda a: a.b > 1 and not a.c)(t)"],
    [P.ifexp(P.compare("<", a, b), x, P.unaryop("-", P.binop("**", y, 2n))), "x if a < b else -y ** 2"],
    [P.ifexp(b, P.ifexp(a, x, y), z), "(x if a else y) if b else z"],
    [P.ifexp(a, x, P.ifexp(b, y, z)), "x if a else y if b else z"],
    [P.binop("-", P.binop("-", P.binop("**", 2n, P.unaryop("-", 1n)), P.binop("-", a, b)), c), "2 ** -1 - (a - b) - c"],
    [P.boolop("or", P.boolop("and", a, b), P.boolop("and", c, P.unaryop("not", d))), "a and b or c and not d"],
    [P.boolop("and", P.boolop("or", a, b), P.boolop("or", c, d)), "(a or b) and (c or d)"],
    [P.boolop("and", a, b, c), "a and b and c"], [P.boolop("and", P.boolop("and", a, b), c), "(a and b) and c"], // n values
    [P.compare("!=", P.unaryop("not", P.compare("==", a, b)), P.compare("<=", c, d)), "(not a == b) != (c <= d)"],
    [P.compare("<", a, b, ["<=", c], ["is not", d]), "a < b <= c is not d"], // a chain
    [P.call(P.call("f", P.name("g")), P.subscript(P.attribute(x, "y"), "z"), P.unaryop("+", 1n), new Uint8Array([0]), true, "it's"),
      "f(g)(x.y['z'], +1, b'\\x00', True, \"it's\")"],
    [P.call("f", a, P.starred(b), P.keyword("k", 1n), P.keyword(null, c)), "f(a, *b, k=1, **c)"], // keywords, unpacked
    [P.import_("a", P.import_("b", P.importfrom("d", "e", P.importfrom("d", "f", 1n, "g")), "c")),
      "import a\nimport b as c\nfrom d import e\nfrom d import f as g\n1"],
    [P.call("all", P.generatorexp(P.compare(">", P.attribute(P.name("p"), "pin"), 0n), P.comprehension("p", P.attribute(t, "ports")))),
      "all(p.pin > 0 for p in t.ports)"],
    [P.call("sum", P.generatorexp(1n, P.comprehension("p", x, [P.compare("in", P.name("p"), y), P.compare("not in", P.name("p"), z)]))),
      "sum(1 for p in x if p in y if p not in z)"],
    [P.binop("+", P.subscript(x, P.binop("-", a, 1n)), P.call("len", P.call("set", x))), "x[a - 1] + len(set(x))"],
    [P.call("max", P.generatorexp(P.ifexp(b, P.name("v"), z), P.comprehension("v", P.ifexp(a, x, y), [P.boolop("or", a, b)])), 1n),
      "max((v if b else z for v in (x if a else y) if a or b), 1)"],
    [P.subscript(t, P.tuple_(P.slice_(1n, null, 2n), P.slice_(), P.constant(P.ELLIPSIS), P.constant(null), P.starred(x))),
      "t[1::2, :, ..., None, *x]"], // a tensor's indices: slices, ..., a new axis, unpacked
    // every other kind: displays, comprehensions, lambdas, named expressions, await and yield, `@`, constants
    [P.listcomp(x, P.comprehension("x", y, [x]), P.comprehension("z", x)), "[x for x in y if x for z in x]"],
    [P.dictcomp(P.name("k"), P.name("v"), P.comprehension(P.tuple_(P.name("k"), P.name("v")), P.call(P.attribute(d, "items")))),
      "{k: v for k, v in d.items()}"],
    [P.setcomp(x, P.comprehension("x", y)), "{x for x in y}"], [P.listcomp(x, P.comprehension("x", y, [], true)), "[x async for x in y]"],
    [P.set_(1n, P.starred(a)), "{1, *a}"],
    [P.dict_([null, a], ["k", 1n]), "{**a, 'k': 1}"], [P.tuple_(), "()"], [P.tuple_(1n), "(1,)"], [P.tuple_(1n, 2n), "(1, 2)"],
    [P.subscript(x, P.tuple_()), "x[()]"], [P.subscript(x, P.tuple_(1n)), "x[1,]"], [P.subscript(x, P.slice_()), "x[:]"],
    [P.lambda_(0n), "lambda: 0"], [P.lambda_(P.name("k"), [], { kwonlyargs: ["k"] }), "lambda *, k: k"],
    [P.lambda_(a, ["b", P.arg("c", 1n)], { posonlyargs: ["a"], vararg: "d", kwonlyargs: ["e", P.arg("f", 2n)], kwarg: "g" }),
      "lambda a, /, b, c=1, *d, e, f=2, **g: a"],
    [P.namedexpr("y", 10n), "(y := 10)"], [P.await_(x), "await x"], [P.yield_(), "(yield)"], [P.yield_(x), "(yield x)"],
    [P.yieldfrom(x), "(yield from x)"], [P.binop("@", a, b), "a @ b"], [P.compare("is", a, b), "a is b"],
    [P.unaryop("-", P.binop("**", x, 2n)), "-x ** 2"], [P.attribute(1n, "real"), "(1).real"],
    [P.constant(new P.Complex(0, 2)), "2j"], [P.constant(null), "None"], [P.constant(P.ELLIPSIS), "..."],
    [P.concatenatedstring("a", "b"), "'a' 'b'"],
    [P.call("f", P.generatorexp(x, P.comprehension("x", y))), "f(x for x in y)"],
    [P.call("f", P.generatorexp(x, P.comprehension("x", y)), 1n), "f((x for x in y), 1)"],
  ] as [unknown, string][]) {
    const written = py(built);
    assert(PText.ToText(written) === source && roundTrips(written), PText.ToText(written));
  }
  for (const [built, source] of [ // printed as written, what a parser keeps beside its terms: an f-string's text and fields, parentheses
    [P.joinedstr(["a{{b}}", P.formattedvalue(x, "r", [">", P.formattedvalue(P.name("w"))]), " ", P.formattedvalue(y, null, null, true)]),
      "f'a{{b}}{x!r:>{w}} {y=}'"],
    [P.templatestr(["x", P.interpolation(y), "z"]), "t'x{y}z'"], [P.joinedstr([P.formattedvalue(P.set_(1n))], "rf", "\""), "rf\"{ {1}}\""],
    [P.concatenatedstring(P.constant("a", "'a'"), P.constant("b", "\"b\"")), "'a' \"b\""],
    [P.parenthesized(P.binop("+", a, b)), "(a + b)"], [P.binop("*", P.parenthesized(a), b), "(a) * b"],
    [P.dictcomp(x, null, P.comprehension("x", y)), "{**x for x in y}"], // Python 3.15's, which 3.14 does not parse
    [P.listcomp(P.starred(x), P.comprehension("x", y)), "[*x for x in y]"],
  ] as [unknown, string][]) {
    assert(PText.ToText(py(built)) === source && roundTrips(py(built)), PText.ToText(py(built)));
  }
  assert(!("FromText" in PText)); // reading Python source needs Python's parser: the Python implementation's only
  assert(same([NaN, Infinity, -Infinity, -2n].map((v) => PText.ToText(py(P.constant(v)))), [
    "float('nan')", "float('inf')", "-float('inf')", "-2"]));
  assert(same([null, P.ELLIPSIS, new P.Complex(0, 2), new P.Complex(1, -2), new P.Complex(1, 2.5), new P.Complex(0, -2.5)].map(
    (v) => PText.ToText(py(P.constant(v)))), ["None", "...", "2j", "(1-2j)", "(1+2.5j)", "-2.5j"])); // Python3's own
  assert(PText.ToText(py(P.binop("**", P.constant(new P.Complex(0, -2.5)), 2n))) === "(-2.5j) ** 2"); // its precedence, as its spelling's
  assert(PText.ToText(py(P.binop("**", -2n, 2n))) === "(-2) ** 2" && PText.ToText(py(P.unaryop("-", P.binop("**", 2n, 2n)))) === "-2 ** 2");
  assert(PText.ToText(py(P.call("np.ma.getmaskarray", P.name("x")))) === "np.ma.getmaskarray(x)"); // a dotted function
  assert(PText.ToText(py(P.parenthesized(P.binop("+", a, b)))) === "(a + b)" && PText.ToText(py(P.binop("*", P.parenthesized(a), b))) === "(a) * b");
  assert(PText.ToText(py(P.slice_(a, b))) === "a:b" && PText.ToText(py(P.tuple_(a))) === "(a,)" && PText.ToText(py(P.yield_(a))) === "(yield a)"); // alone
  assert(PText.ToText(py(P.call("f", P.namedexpr("y", 1n)))) === "f((y := 1))" && PText.ToText(py(P.starred(P.binop("+", a, b)))) === "*a + b");
  assert(PText.ToText(py(P.starred(P.compare("<", a, b)))) === "*(a < b)"); // unpacked: a bitwise or, or what binds tighter
  assert(PText.ToText(py(P.attribute(P.binop("+", 1n, 2n), "real"))) === "(1 + 2).real");
  const counted = py(P.generatorexp(P.name("p"), P.comprehension("p", P.name("ps"), [P.name("q")]))); // p is bound in its clause and the element
  assert(same([...SY.free(counted, P.DIALECT)].sort(), ["ps", "q"]) && P.DIALECT.validate(counted, ["ps", "q"]).length === 0);
  assert(same(P.DIALECT.validate(py(P.generatorexp(1n, P.comprehension("1p", P.name("ps")))), ["ps"]), [
    "generators 0 target: a name must be an identifier, got '1p'"]));
  assert(same(P.DIALECT.validate(py(P.generatorexp(1n, P.comprehension("p", P.name("p"))))), ["generators 0 iter: variable 'p' is not bound"]));
  const chain = py(P.compare("<", a, b, ["<=", c])); // one comparison chain: b written once, the operators of its links
  assert(PText.ToText(chain) === "a < b <= c" && same(F.form(chain).arguments.map((operand) => F.nameOf(operand)), ["a", "b", "c"]));
  assert(same((F.form(chain).attributes.get("comparisons") as F.Part[]).map((part) => part.attributes.get("op")), ["<", "<="]));
  raises(ValueError, () => PText.ToText(py(P.binop("+", 1n, P.import_("math", 2n)))), "an import can only enclose the whole expression");
  // Displays: a list display (the framework's container constructor), and a dict display, evaluated as Python's.
  assert(PText.ToText(py(P.list_(1n, x))) === "[1, x]" && PText.ToText(py(P.dict_(["a", 1n], [x, P.list_()]))) === "{'a': 1, x: []}"
    && PText.ToText(py(P.dict_())) === "{}");
  assert(equal(PV.OfAny(py(P.dict_(["a", 1n], ["a", 2n], ["b", P.list_(x)])), { x: 3n }), new Map<unknown, unknown>([["a", 2n], ["b", [3n]]]))); // a later key replaces an earlier's
  const kinds = P.DIALECT.kinds(); // an import binds the names its kind reads from its attributes
  assert(same(kinds.get(P.IMPORT)!.names!(F.form(py(P.import_("numpy.ma", 1n))).attributes), ["numpy"]));
  assert(same(kinds.get(P.IMPORTFROM)!.names!(F.form(py(P.importfrom("m", "n", 1n, "k"))).attributes), ["k"]));
  assert(equal(SY.imports(py(P.import_("numpy", P.importfrom("m", "n", 1n), "np")), P.DIALECT).map((i) => F.form(i).attributes),
    map([{ module: "numpy", alias: "np" }, { module: "m", name: "n" }])));
  assert(same(P.DIALECT.validate(py(P.import_("not a module", 1n))), ["an import's module must be a dotted name, got 'not a module'"]));
  assert(same(P.DIALECT.validate(py(P.import_("m", 1n, "a b"))), ["an import's alias must be an identifier, got 'a b'"]));
  assert(same(P.DIALECT.validate(py(P.importfrom("m.", "x y", 1n))), ["an importfrom's module must be a dotted name, got 'm.'",
    "an importfrom's name must be an identifier, got 'x y'"]));
  assert(same(P.DIALECT.validate(py(P.importfrom("m", "n", 1n, ""))), ["an importfrom needs an alias"])); // empty, not absent
  assert(same(P.DIALECT.validate(py(P.let_("a b", 1n, 2n))), ["a let's name must be an identifier, got 'a b'"]));
  assert(same(P.DIALECT.validate(py(P.attribute(P.name("x"), "1")), ["x"]), ["an attribute must be an identifier, got '1'"]));
  for (const [spec, problems] of [
    [P.importfrom("m", "n", P.name("n")), []],
    [(u: any) => u[P.IMPORT]((r: any) => r.module(7n).body(P.constant(1n))), ["an import's module must be a str, got int"]],
    [(u: any) => u[P.IMPORT]((r: any) => r.module("numpy")), ["an import needs a body"]],
    [(u: any) => u[P.LITERAL]((r: any) => r), ["a literal needs a value"]],
    [(u: any) => u[P.COMPARE]((r: any) => r.left(P.constant(1n))), ["a compare needs a comparison"]],
  ] as [unknown, string[]][]) {
    assert(same(P.DIALECT.validate(py(spec)), problems), Repr.repr(P.DIALECT.validate(py(spec))));
  }
  assert(P.DIALECT.validate(py(P.binop("@", 1n, 2n)), [], true).length === 0 && P.DIALECT.validate(py(P.compare("is", 1n, 2n)), [], true).length === 0);
  assert(same(P.DIALECT.validate(py(P.binop("@@", 1n, 2n)), [], true), ["'@@' is not a core operation"]));
  assert(same(P.DIALECT.validate(py(P.compare("<", 1n, 2n, ["===", 3n])), [], true), ["'===' is not a core operation"]));
  assert(P.DIALECT.validate(py(P.compare("<", 1n, 2n, ["===", 3n]))).length === 0); // an extension, unless the core is asked for
  assert(P.DIALECT.validate(py(P.binop("+", 1n, 2n)), [], true).length === 0);
  assert(P.DIALECT.validate(py(P.call("hasattr", P.name("x"), "k")), ["x"]).length === 0); // builtins are ambient
  assert(P.DIALECT.infer(py(P.binop("+", 1n, 2.5))) === PD.Float && P.DIALECT.infer(py(P.binop("+", true, 1n))) === PD.Int);
  assert(P.DIALECT.infer(py(P.binop("+", "a", "b"))) === PD.Str && P.DIALECT.infer(py(P.unaryop("-", 2.0))) === PD.Float);
  assert(P.DIALECT.infer(py(P.binop("**", 2n, 3n))).name() === "int | float" && P.DIALECT.infer(py(P.unaryop("not", 1n))) === PD.Bool);
  assert(P.DIALECT.infer(py(P.compare("<", 1n, 2.5))) === PD.Bool && P.DIALECT.infer(py(P.attribute(P.name("x"), "a")), { x: D.Anything }) === D.Anything);
  assert(P.DIALECT.infer(py(P.boolop("and", 1n, "a"))).name() === "int | str"); // Python's and gives an operand
  assert(P.DIALECT.infer(py(P.let_("a", 1n, P.binop("*", P.name("a"), 2.5)))) === PD.Float
    && P.DIALECT.infer(py(P.call("f", 1n)), { f: D.Anything }) === D.Anything);
  assert(P.DIALECT.infer(py(P.generatorexp(P.name("p"), P.comprehension("p", P.name("v")))), { v: D.Anything }) === D.Anything);
  assert(P.DIALECT.infer(py(P.compare("<", 1n, 2n, ["<=", 3.5]))) === PD.Bool); // a chain
  assert(P.DIALECT.infer(py(P.boolop("and", 1n, "a", 2.5))).name() === "int | str | float");
  assert(P.DIALECT.infer(py(P.compare("<", 1n, 2n, ["===", 3n]))) === D.Anything); // an extension's operator: not known
  assert(P.DIALECT.infer(py(P.constant(null))) === PD.NoneType && P.DIALECT.infer(py(P.constant(P.ELLIPSIS))) === PD.Ellipsis
    && P.DIALECT.infer(py(P.constant(new P.Complex(0, 2)))) === PD.Complex); // Python3's constants
  assert(P.DIALECT.infer(py(P.joinedstr([P.formattedvalue(x)])), { x: D.Anything }) === PD.Str
    && P.DIALECT.infer(py(P.subscript(x, P.slice_(1n))), { x: D.Anything }) === D.Anything);
  raises(TypeError, () => P.DIALECT.infer(py(P.compare("<", "a", 1n))), "< cannot take (str, int)");
  raises(TypeError, () => P.DIALECT.infer(py(P.compare("<", 1n, 2n, ["<=", "a"]))), "<= cannot take (int, str)");
  assert(PD.of(new Uint8Array()) === PD.Bytes && PD.Numeric.includes(PD.Bool));
  raises(TypeError, () => P.DIALECT.spec([1n]), "expected a term, a writer or a native value, got [1]"); // no Python3 constant holds a list
  raises(ValueError, () => PV.OfAny(term(Patterns.Expressions.operation("add", 1n, 2n), Patterns.Expressions.DIALECT)),
    "a Patterns call is not a Python3 term"); // a term of another dialect: Python3 reads its own
}

<a id="DIA-02"></a>
## DIA-02 · Python evaluates by Python's rules, in a scope that allowlists what an expression may reach

In [ ]:
{
  const ev = (expression: unknown, variables: Record<string, unknown> = {}, options: any = {}) =>
    PV.OfAny(py(expression), new PV.Scope(variables, options));
  const [a, t] = [P.name("a"), P.name("t")];
  assert(ev(P.boolop("or", 0n, "x")) === "x" && ev(P.boolop("and", "", 1n)) === "" && ev(P.boolop("and", 1n, 2n)) === 2n); // an operand, not a bool
  assert(ev(P.compare("==", 1n, 1.0)) === true && ev(P.binop("+", true, 1n)) === 2n);
  assert(ev(P.binop("-", P.binop("%", P.binop("//", 7n, 2n), 3n), P.binop("/", P.binop("**", 2n, 2n), 4n))) === -1.0);
  assert(ev(P.ifexp(P.compare(">", a, 1n), a, P.name("missing")), { a: 2n }) === 2n
    && ev(P.let_("a", 3n, P.binop("*", a, a))) === 9n);
  assert(ev(P.attribute(t, "age"), { t: ann }) === 30n && ev(P.call("hasattr", t, "email"), { t: bob }) === false); // properties are attributes
  assert(ev(P.call("getattr", t, "name"), { t: ann }) === "Ann" && ev(P.subscript(t, "x"), { t: { x: 1n } }) === 1n);
  assert(ev(P.call("hasattr", t, "_secret"), { t: ann }) === false && ev(P.call("hasattr", t, "email"), { t: ann }) === true);
  assert(ev(P.binop("+", P.binop("+", P.call("abs", P.unaryop("-", 2n)), P.call("len", "abc")), P.call("max", 1n, 2n))) === 7n
    && ev(P.call("f", 2n), { f: (v: bigint) => v + 1n }) === 3n);
  raises(AttributeError, () => ev(P.attribute(t, "email"), { t: bob }), "'Person' object has no attribute 'email'");
  raises(AttributeError, () => ev(P.attribute(t, "__class__"), { t: ann }), "attribute '__class__' is private");
  raises(NameError, () => ev(P.name("x")), "name 'x' is not defined");
  raises(NameError, () => ev(P.call("len", "a"), {}, { builtins: {} }), "name 'len' is not defined");
  function shout(value: string): string {
    return value.toUpperCase();
  }
  raises(TypeError, () => ev(P.call(P.subscript(t, "f"), "a"), { t: { f: shout } }),
    "calling 'shout' is not allowed by the scope"); // only what the scope provides, not what values hold
  raises(TypeError, () => ev(P.call(t), { t: 3n }), "calling 'int' is not allowed by the scope");

  // NumPy is Python's; a module of the scope stands in for it here.
  const masked = { masked: true };
  const numpy = { numpy: new PV.Module("numpy", {
    ma: new PV.Module("numpy.ma", { masked }), pi: Math.PI, _core: new PV.Module("numpy._core", {}) }) };
  assert(ev(P.import_("numpy.ma", P.attribute(P.attribute(P.name("numpy"), "ma"), "masked")), {}, { modules: numpy }) === masked); // binds numpy
  assert(ev(P.import_("numpy.ma", P.attribute(P.name("ma"), "masked"), "ma"), {}, { modules: numpy }) === masked
    && ev(P.import_("numpy", P.attribute(P.name("numpy"), "pi")), {}, { modules: numpy }) === Math.PI);
  raises(ImportError, () => ev(P.import_("numpy", 1n, "np")), "import of 'numpy' is not allowed by the scope"); // nothing is imported unless the scope allows it
  raises(ImportError, () => ev(P.import_("os", P.call("os.getcwd")), {}, { modules: numpy }), "import of 'os' is not allowed by the scope");
  raises(ImportError, () => ev(P.import_("numpy.nope", 1n), {}, { modules: numpy }), "No module named 'numpy.nope'");
  raises(ImportError, () => ev(P.import_("numpy._core", 1n), {}, { modules: numpy }), "No module named 'numpy._core'");
  raises(ImportError, () => ev(P.importfrom("numpy", "nope", 1n), {}, { modules: numpy }), "cannot import name 'nope' from 'numpy'");
  raises(ImportError, () => ev(P.importfrom("numpy", "_core", 1n), {}, { modules: numpy }), "cannot import name '_core' from 'numpy'");
  assert(PV.OfAny(py(P.binop("+", P.name("x"), 1n)), { x: 1n }) === 2n); // an object is the variables of a scope
  const adult = store.transaction(() => store.Expressions.Framework.Expression( // an expression, its root written by a writer
    (e: any) => e.root(P.compare(">=", P.attribute(P.name("t"), "age"), 18n))));
  assert(PV.OfAny(adult, { t: ann }) === true && PText.ToText(adult) === "t.age >= 18");

  // Python's rules, case by case. The Python suite runs this table against Python itself; here it runs against the
  // model of them.
  const n = P.name;
  const bytes = (...values: number[]) => new Uint8Array(values);
  const modules = { math: new PV.Module("math", { floor: (v: number) => BigInt(Math.floor(v)) }), ...numpy };
  const VALUES: [unknown, Record<string, unknown>, unknown][] = [ // (expression, variables, value): the value and its type
    [P.unaryop("not", n("v")), { v: null }, true], [P.unaryop("not", 0.0), {}, true], [P.unaryop("not", ""), {}, true],
    [P.unaryop("not", bytes()), {}, true], [P.unaryop("not", n("v")), { v: [] }, true], [P.unaryop("not", n("v")), { v: {} }, true],
    [P.unaryop("not", NaN), {}, false], [P.unaryop("not", n("v")), { v: { k: 1n } }, false],
    [P.compare("<", 1n, 1.5), {}, true], [P.compare(">", 2n, 1.5), {}, true], [P.compare("==", 1n, 1.5), {}, false],
    [P.compare("==", P.binop("+", P.binop("**", 2n, 60n), 1n), P.call("float", P.binop("**", 2n, 60n))), {}, false],
    [P.compare("<", 1n, Infinity), {}, true], [P.compare(">", 1n, -Infinity), {}, true], [P.compare("<", 1n, NaN), {}, false],
    [P.compare("<", 1.5, 2n), {}, true], [P.compare("==", NaN, NaN), {}, false], [P.compare("==", 3n, 3.0), {}, true],
    [P.compare("<", 3n, 3.5), {}, true], [P.compare("<", 2.5, 1.5), {}, false], [P.compare("<", true, 2n), {}, true],
    [P.compare("<", "a", "b"), {}, true], [P.compare("<", bytes(97), bytes(98)), {}, true], [P.compare("==", bytes(97), bytes(97)), {}, true],
    [P.compare("!=", "a", 1n), {}, true],
    [P.binop("+", "ab", "c"), {}, "abc"], [P.binop("+", bytes(97), bytes(98)), {}, bytes(97, 98)], [P.binop("*", "ab", 2n), {}, "abab"],
    [P.binop("*", 2n, bytes(120)), {}, bytes(120, 120)], [P.binop("*", "ab", -1n), {}, ""], [P.binop("//", -7n, 2n), {}, -4n],
    [P.binop("%", -7n, 2n), {}, 1n], [P.binop("%", 7n, -2n), {}, -1n], [P.binop("%", -7.5, 2n), {}, 0.5], [P.binop("%", 7.5, 2n), {}, 1.5],
    [P.binop("//", 7.5, 2n), {}, 3.0], [P.binop("**", 2n, -1n), {}, 0.5], [P.binop("**", 2.0, 3n), {}, 8.0],
    [P.binop("+", 1.5, 1n), {}, 2.5], [P.binop("*", 1.5, 2n), {}, 3.0], [P.binop("-", 3n, 1.5), {}, 1.5], [P.binop("/", 1n, 2n), {}, 0.5],
    [P.unaryop("-", true), {}, -1n], [P.unaryop("+", true), {}, 1n], [P.unaryop("-", 1.5), {}, -1.5], [P.unaryop("+", 1.5), {}, 1.5],
    [P.call("abs", true), {}, 1n], [P.call("abs", -1.5), {}, 1.5], [P.call("bool", ""), {}, false], [P.call("float", 1n), {}, 1.0],
    [P.call("float", " 1e3 "), {}, 1000.0], [P.call("float", "inf"), {}, Infinity], [P.call("float", "-inf"), {}, -Infinity],
    [P.call("str", P.call("float", "nan")), {}, "nan"], [P.call("int", " 42 "), {}, 42n], [P.call("int", 2.7), {}, 2n],
    [P.call("int", -2.7), {}, -2n], [P.call("int", true), {}, 1n], [P.call("len", bytes(97, 98)), {}, 2n],
    [P.call("len", n("v")), { v: [1n, 2n] }, 2n], [P.call("len", n("v")), { v: { a: 1n } }, 1n], [P.call("len", "héllo"), {}, 5n],
    [P.call("max", 1n, 2.5), {}, 2.5], [P.call("min", 3n, 1n, 2n), {}, 1n], [P.call("round", 2.5), {}, 2n],
    [P.call("round", 3.5), {}, 4n], [P.call("round", -0.5), {}, 0n], [P.call("round", 7n), {}, 7n], [P.call("str", 1.0), {}, "1.0"],
    [P.call("str", bytes(120)), {}, "b'x'"], [P.call("str", n("v")), { v: null }, "None"], [P.call("str", true), {}, "True"],
    [P.call("str", 7n), {}, "7"], [P.call("str", "s"), {}, "s"], [P.import_("math", P.call("math.floor", 2.5)), {}, 2n],
    [P.unaryop("-", false), {}, 0n], [P.unaryop("not", n("v")), { v: ann }, false], [P.compare("==", 1.5, 1.5), {}, true],
    [P.binop("/", 1.5, 2n), {}, 0.75], [P.binop("&", 12n, 10n), {}, 8n], [P.binop("|", 12n, true), {}, 13n],
    [P.binop("^", true, true), {}, false], [P.binop("&", true, false), {}, false], [P.binop("|", false, true), {}, true],
    [P.binop("<<", true, 3n), {}, 8n], [P.binop(">>", -7n, 1n), {}, -4n], [P.unaryop("~", 5n), {}, -6n],
    [P.binop("|", P.binop("**", 2n, 70n), 1n), {}, 2n ** 70n + 1n],
    [P.compare("<", 1n, 2n, ["<", 3n]), {}, true], [P.compare("<", 1n, 3n, ["<", 2n]), {}, false], // a chain, link by link
    [P.compare("==", 1n, 1.0, ["in", n("v")]), { v: [1.0] }, true], [P.compare(">", 1n, 2n, ["<", "a"]), {}, false], // the first false link decides, its later operands not compared
    [P.boolop("and", 1n, 2n, 3n), {}, 3n], [P.boolop("and", 1n, 0n, n("missing")), {}, 0n], [P.boolop("or", 0n, "", 5n), {}, 5n],
    [P.boolop("or", 0n, ""), {}, ""], // n values: the first that decides, else the last
    [P.call("sum", P.generatorexp(P.binop("*", n("p"), n("q")), P.comprehension("p", n("v")),
      P.comprehension("q", n("v"), [P.compare(">", n("q"), n("p"))]))), { v: [1n, 2n, 3n] }, 11n],
    [P.call("all", P.generatorexp(P.compare(">", n("p"), 0n), P.comprehension("p", n("v")))), { v: [1n, 2n] }, true],
    [P.call("all", P.generatorexp(n("p"), P.comprehension("p", n("v")))), { v: [] }, true], [P.call("any", n("v")), { v: [0n, 3n] }, true],
    [P.call("any", P.generatorexp(n("p"), P.comprehension("p", n("v")))), { v: [0n, ""] }, false],
    [P.call("sum", P.generatorexp(1n, P.comprehension("p", n("v"), [P.compare(">", n("p"), 0n)]))), { v: [1n, -1n, 2n] }, 2n],
    [P.call("sum", P.generatorexp(1n, P.comprehension("c", "abc", [P.compare("!=", n("c"), "b"), P.compare("!=", n("c"), "c")]))), {}, 1n],
    [P.call("sum", n("v")), { v: Array(10).fill(0.1) }, 1.0], [P.call("sum", n("v")), { v: [1n, 0.1, 0.2, 0.3] }, 1.6],
    [P.call("sum", n("v")), { v: [1e+16, 1n, 1n] }, 1.0000000000000002e+16],
    [P.call("sum", n("v"), true), { v: Array(10).fill(0.1) }, 2.000000000000001], [P.call("sum", n("v")), { v: [true, true] }, 2n],
    [P.call("sum", n("v")), { v: [] }, 0n], [P.call("sum", n("v"), 1.5), { v: [] }, 1.5],
    [P.call("sum", n("v")), { v: [2n ** 62n, 2n ** 62n, 0.5] }, 9.223372036854776e+18],
    [P.call("sum", n("v")), { v: [(2n ** 63n) - 1n, 1n] }, 2n ** 63n],
    [P.call("sum", n("v")), { v: [-654.25, (-(2n ** 63n)) - 1n, -9n, (2n ** 63n) + 1n] }, -663.25],
    [P.call("sum", n("v"), 2n ** 64n), { v: [1n, 0.5] }, 1.8446744073709552e+19], [P.call("min", n("v")), { v: [3n, 1n, 2n] }, 1n],
    [P.call("max", P.generatorexp(P.binop("*", n("p"), 2n), P.comprehension("p", n("v")))), { v: [1n, 2.5] }, 5.0], [P.call("max", "abc"), {}, "c"],
    [P.call("min", P.call("set", n("v"))), { v: [2n, 1n] }, 1n], [P.call("len", P.call("set", n("v"))), { v: [1n, 1.0, true, 2n] }, 2n],
    [P.call("len", P.call("set")), {}, 0n], [P.unaryop("not", P.call("set")), {}, true],
    [P.call("all", n("v")), { v: [1n, 0n] }, false], [P.ifexp(0n, 1n, 2n), {}, 2n], [P.binop("-", 7n, 10n), {}, -3n],
    [P.compare("in", 2n, n("v")), { v: [1n, 2n] }, true], [P.compare("in", "b", "abc"), {}, true],
    [P.compare("in", "k", n("v")), { v: new Map([["k", 1n]]) }, true], [P.compare("not in", 3n, n("v")), { v: [1n, 2n] }, true],
    [P.compare("in", 1.0, n("v")), { v: [1n] }, true], [P.compare("in", 98n, new Uint8Array([97, 98])), {}, true],
    [P.compare("in", new Uint8Array([98]), new Uint8Array([97, 98])), {}, true],
    [P.compare("in", new Uint8Array([98, 97]), new Uint8Array([97, 98])), {}, false],
    [P.compare("in", n("w"), n("v")), { v: [[1n]], w: [1n] }, true], [P.compare("in", 1n, P.call("set", n("v"))), { v: [1n] }, true],
    [P.compare("in", 3n, P.generatorexp(n("p"), P.comprehension("p", n("v")))), { v: [1n, 2n] }, false], [P.subscript(n("v"), 1n), { v: [10n, 20n] }, 20n],
    [P.subscript(n("v"), -1n), { v: [10n, 20n] }, 20n], [P.subscript("abc", 1n), {}, "b"], [P.subscript(new Uint8Array([97, 98]), 0n), {}, 97n],
    [P.subscript(n("v"), true), { v: [10n, 20n] }, 20n], [P.subscript(n("v"), 1.0), { v: new Map([[1n, "x"]]) }, "x"],
    [P.subscript(n("v"), "k"), { v: new Map([["k", 1n]]) }, 1n], [P.call("all", P.subscript(n("v"), 0n)), { v: [[1n, 1n]] }, true],
    [P.call("sum", bytes(1, 2)), {}, 3n], [P.call("max", bytes(97, 98)), {}, 98n],
    [P.call("len", P.call("set", n("v"))), { v: { a: 1n, b: 2n } }, 2n], [P.subscript(n("v"), "b"), { v: { a: 1n, b: 2n } }, 2n],
    [P.call("sum", P.attribute(n("v"), "ranks")), { v: squad }, 6n], [P.subscript(P.attribute(n("v"), "ranks"), -1n), { v: squad }, 3n],
  ];
  const pyEqual = (a: unknown, b: unknown) => Repr.typeName(a) === Repr.typeName(b)
    && (a instanceof Uint8Array ? F.sameNative(a, b) : a === b);
  for (const [expression, variables, expected] of VALUES) {
    const value = PV.OfAny(py(expression), new PV.Scope(variables, { modules }));
    assert(pyEqual(value, expected), `${PText.ToText(py(expression))}: ${Repr.repr(value)}, not ${Repr.repr(expected)}`);
  }
  const ERRORS: [unknown, Record<string, unknown>, Function, string][] = [ // (expression, variables, error, message)
    [P.compare("<", "a", 1n), {}, TypeError, "'<' not supported between instances of 'str' and 'int'"],
    [P.binop("*", "a", 1.5), {}, TypeError, "can't multiply sequence by non-int of type 'float'"],
    [P.binop("-", "a", 1n), {}, TypeError, "unsupported operand type(s) for -: 'str' and 'int'"],
    [P.binop("&", 1.5, 1n), {}, TypeError, "unsupported operand type(s) for &: 'float' and 'int'"],
    [P.binop("<<", 1n, -1n), {}, ValueError, "negative shift count"], [P.binop(">>", 1n, -1n), {}, ValueError, "negative shift count"],
    [P.unaryop("~", 1.5), {}, TypeError, "bad operand type for unary ~: 'float'"],
    [P.binop("/", 1n, 0n), {}, ZeroDivisionError, "division by zero"],
    [P.binop("//", 1.5, 0n), {}, ZeroDivisionError, "division by zero"],
    [P.binop("%", 1n, 0n), {}, ZeroDivisionError, "division by zero"],
    [P.binop("**", 0.0, -1n), {}, ZeroDivisionError, "zero to a negative power"],
    [P.unaryop("-", "a"), {}, TypeError, "bad operand type for unary -: 'str'"],
    [P.call("abs", "a"), {}, TypeError, "bad operand type for abs(): 'str'"],
    [P.call("float", "x"), {}, ValueError, "could not convert string to float: 'x'"],
    [P.call("int", "x"), {}, ValueError, "invalid literal for int() with base 10: 'x'"],
    [P.call("int", NaN), {}, ValueError, "cannot convert float NaN to integer"],
    [P.call("int", Infinity), {}, OverflowError, "cannot convert float infinity to integer"],
    [P.call("len", 1n), {}, TypeError, "object of type 'int' has no len()"],
    [P.call("max", 1n, "a"), {}, TypeError, "'>' not supported between instances of 'str' and 'int'"],
    [P.call("round", "x"), {}, TypeError, "type str doesn't define __round__ method"],
    [P.call("float", n("v")), { v: null }, TypeError, "float() argument must be a string or a real number, not 'NoneType'"],
    [P.call("int", n("v")), { v: null }, TypeError,
      "int() argument must be a string, a bytes-like object or a real number, not 'NoneType'"],
    [P.import_("numpy", P.attribute(n("numpy"), "nope")), {}, AttributeError, "module 'numpy' has no attribute 'nope'"],
    [P.subscript(n("t"), "y"), { t: { x: 1n } }, KeyError, "'y'"],
    [P.subscript(n("t"), "x"), { t: 3n }, TypeError, "'int' object is not subscriptable"],
    [P.call(P.subscript(n("t"), "f")), { t: { f: [() => 1n][0] } }, TypeError, "calling '<lambda>' is not allowed by the scope"],
    [P.call("len", P.generatorexp(n("p"), P.comprehension("p", n("v")))), { v: [1n] }, TypeError, "object of type 'generator' has no len()"],
    [P.call("sum", n("v")), { v: ["a"] }, TypeError, "unsupported operand type(s) for +: 'int' and 'str'"],
    [P.call("sum", n("v")), { v: [0.5, "a"] }, TypeError, "unsupported operand type(s) for +: 'float' and 'str'"],
    [P.call("sum", n("v"), "a"), { v: [1n] }, TypeError, "sum() can't sum strings [use ''.join(seq) instead]"],
    [P.call("sum", n("v"), new Uint8Array([])), { v: [1n] }, TypeError, "sum() can't sum bytes [use b''.join(seq) instead]"],
    [P.call("sum", n("v")), { v: [0.5, 10n ** 400n] }, OverflowError, "int too large to convert to float"],
    [P.call("min", n("v")), { v: [] }, ValueError, "min() iterable argument is empty"],
    [P.call("max", P.generatorexp(n("p"), P.comprehension("p", n("v")))), { v: [] }, ValueError, "max() iterable argument is empty"],
    [P.call("max"), {}, TypeError, "max expected at least 1 argument, got 0"],
    [P.call("min", 1n), {}, TypeError, "'int' object is not iterable"],
    [P.call("set", n("v")), { v: [[1n]] }, TypeError, "cannot use 'list' as a set element (unhashable type: 'list')"],
    [P.call("all", 5n), {}, TypeError, "'int' object is not iterable"],
    [P.generatorexp(n("p"), P.comprehension("p", 5n)), {}, TypeError, "'int' object is not iterable"],
    [P.compare("in", 1n, 5n), {}, TypeError, "argument of type 'int' is not a container or iterable"],
    [P.compare("in", 1n, "abc"), {}, TypeError, "'in <string>' requires string as left operand, not int"],
    [P.compare("in", "a", new Uint8Array([97, 98])), {}, TypeError, "a bytes-like object is required, not 'str'"],
    [P.compare("in", 256n, new Uint8Array([97, 98])), {}, ValueError, "byte must be in range(0, 256)"],
    [P.compare("in", n("w"), n("v")), { v: new Map([[1n, 2n]]), w: [1n] }, TypeError, "cannot use 'list' as a dict key (unhashable type: 'list')"],
    [P.compare("in", n("w"), P.call("set", n("v"))), { v: [1n], w: [1n] }, TypeError, "cannot use 'list' as a set element (unhashable type: 'list')"],
    [P.subscript(n("v"), 5n), { v: [1n] }, IndexError, "list index out of range"],
    [P.subscript(n("v"), -3n), { v: [1n, 2n] }, IndexError, "list index out of range"],
    [P.subscript("abc", 5n), {}, IndexError, "string index out of range"],
    [P.subscript(new Uint8Array([97]), 3n), {}, IndexError, "index out of range"],
    [P.subscript(n("v"), P.binop("**", 2n, 70n)), { v: [1n] }, IndexError, "cannot fit 'int' into an index-sized integer"],
    [P.subscript(n("v"), 1.5), { v: [1n] }, TypeError, "list indices must be integers or slices, not float"],
    [P.subscript("abc", 1.5), {}, TypeError, "string indices must be integers, not 'float'"],
    [P.subscript(new Uint8Array([97, 98]), 1.5), {}, TypeError, "byte indices must be integers or slices, not float"],
    [P.subscript(5n, 0n), {}, TypeError, "'int' object is not subscriptable"],
    [P.subscript(n("v"), 5n), { v: new Map([["k", 1n]]) }, KeyError, "5"],
    [P.subscript(n("v"), n("w")), { v: new Map([["k", 1n]]), w: ["k"] }, TypeError, "cannot use 'list' as a dict key (unhashable type: 'list')"],
    [P.binop("@@", 1n, 2n), {}, NotImplementedError, "'@@' is not a core operation"],
    [(u: any) => u[P.LITERAL]((r: any) => r), {}, ValueError, "a literal needs a value"],
    // The evaluator is shallow: what it does not evaluate it refuses by name, before evaluating anything
    [P.binop("@", 1n, 2n), {}, NotImplementedError, "Python3 binop '@' is not evaluated"],
    [P.compare("<", 1n, 2n, ["is", 3n]), {}, NotImplementedError, "Python3 compare 'is' is not evaluated"],
    [P.compare("<", 1n, 2n, ["===", 3n]), {}, NotImplementedError, "'===' is not a core operation"],
    [P.lambda_(1n), {}, NotImplementedError, "Python3 lambda is not evaluated"],
    [P.constant(null), {}, NotImplementedError, "Python3 constant is not evaluated"],
    [P.tuple_(1n, 2n), {}, NotImplementedError, "Python3 tuple is not evaluated"],
    [P.subscript(n("v"), P.slice_(1n)), { v: [1n] }, NotImplementedError, "Python3 slice is not evaluated"],
    [P.call("abs", P.keyword("x", 1n)), {}, NotImplementedError, "Python3 keyword is not evaluated"],
    [P.call("abs", P.starred(n("v"))), { v: [1n] }, NotImplementedError, "Python3 starred is not evaluated"],
    [P.dict_([null, n("v")]), { v: {} }, NotImplementedError, "Python3 dict's ** item is not evaluated"],
    [P.generatorexp(1n, P.comprehension("p", n("v"), [], true)), { v: [1n] }, NotImplementedError,
      "Python3 async comprehension is not evaluated"],
    [P.generatorexp(1n, P.comprehension(P.tuple_(n("p")), n("v"))), { v: [1n] }, NotImplementedError,
      "Python3 tuple target is not evaluated"],
    [(u: any) => u[P.GENERATOREXP]((r: any) => r.body(P.constant(1n)).generators([(a: any) => a.as_object((o: any) => o.iter(n("v")))])),
      { v: [1n] }, NotImplementedError, "Python3 missing target is not evaluated"],
    [(u: any) => u[P.GENERATOREXP]((r: any) => r.body(P.constant(1n))), {}, ValueError, "a generatorexp needs a for clause"],
    [(u: any) => u[P.IMPORT]((r: any) => r.module("m")), {}, ValueError, "an import needs a body"],
  ];
  for (const [expression, variables, error, message] of ERRORS) {
    raises(error, () => PV.OfAny(py(expression), new PV.Scope(variables, { modules })), message);
  }
  assert(PV.OfAny(py(P.importfrom("numpy.ma", "masked", n("masked"))), new PV.Scope({}, { modules })) === masked);
  // A mapping is a Map or a plain object.
  assert(ev(P.subscript(t, "x"), { t: new Map([["x", 1n]]) }) === 1n && ev(P.unaryop("not", t), { t: new Map() }) === true);
  raises(KeyError, () => ev(P.subscript(t, "y"), { t: new Map() }), "'y'");
  assert(ev(P.call("len", t), { t: new Map([["x", 1n]]) }) === 1n);
}

<a id="DIA-03"></a>
## DIA-03 · MATLAB: operators, fields and `isfield`, rendered as MATLAB source

In [ ]:
const s = M.identifier("s");
const matlabRule = mat(M.binary("&&", M.binary(">=", M.field(s, "age"), 18n), M.call("isfield", s, "email")));
{
  assert(MText.ToText(matlabRule) === 's.age >= 18 && isfield(s, "email")' && roundTrips(matlabRule));
  assert(MText.ToText(mat(M.binary("-", 1n, M.binary("-", 2n, 3n)))) === "1 - (2 - 3)"
    && MText.ToText(mat(M.binary("-", M.binary("-", 1n, 2n), 3n))) === "1 - 2 - 3");
  assert(MText.ToText(mat(M.binary(".*", M.binary("+", 1n, 2n), M.unary("-", M.unary("~", s))))) === "(1 + 2) .* -~s");
  assert(MText.ToText(mat(M.unary("-", M.binary("+", 1n, 2n)))) === "-(1 + 2)" && MText.ToText(mat(M.unary("~", -1n))) === "~-1");
  assert(MText.ToText(mat(M.binary("||", M.binary("&&", true, false), M.binary("&&", 1n, 0n)))) === "true && false || 1 && 0");
  assert(same([NaN, Infinity, -Infinity, -2.5, 'say "hi"', 7n].map((v) => MText.ToText(mat(M.constant(v)))), [
    "NaN", "Inf", "-Inf", "-2.5", '"say ""hi"""', "7"]));
  assert(MText.ToText(mat(M.field(M.field(s, "a"), "b"))) === "s.a.b");
  assert(M.DIALECT.infer(mat(matlabRule), { s: MD.Struct }) === MD.Logical);
  assert(M.DIALECT.infer(mat(M.binary("+", "a", 1n))) === MD.String && M.DIALECT.infer(mat(M.binary("+", true, 1n))) === MD.Double);
  assert(MD.Double.contains(1n) && MD.Struct.contains({}) && !MD.Struct.contains("s") && MD.of(1n) === MD.Double);
  assert(MD.Struct.contains(ann) && !MD.Struct.contains([1n])); // a data object is a struct
  assert(same(M.DIALECT.validate(mat(M.binary("^", 1n, 2n)), [], true), ["'^' is not a core operation"]));
  assert(same(M.DIALECT.validate(mat((u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(new Uint8Array([120])))))),
    ["a literal cannot hold a bytes"]));
  assert(same(M.DIALECT.validate(mat(M.binary("+", null, 1n))), ["a binary needs a left"]));
  assert(equal(F.form(mat(M.call("f", 1n))).attributes, map({ callable: "f" }))); // MATLAB's call is the framework's
  raises(TypeError, () => M.constant(new Uint8Array([120])), "a literal cannot hold a bytes");
  raises(TypeError, () => M.binary("+", new Uint8Array([120]), 1n), "expected a term, a writer or a native value, got b'x'");
  raises(Error, () => term((u: any) => u.let((r: any) => r.name("x")), M.DIALECT), "'let' is not a member of a union value"); // MATLAB's union has no let
  const notMatlab = term(Patterns.Expressions.let_("x", 1n, 2n), Patterns.Expressions.DIALECT);
  assert(same(M.DIALECT.validate(notMatlab), ["a Patterns let is not a MATLAB term"]));
  raises(ValueError, () => M.DIALECT.kind(notMatlab), "a Patterns let is not a MATLAB term");
}

<a id="DIA-04"></a>
## DIA-04 · MATLAB evaluates two-valued, converts numbers and logicals, and has no unknown

In [ ]:
{
  assert(MV.OfAny(mat(matlabRule), { s: ann }) === true && MV.OfAny(mat(matlabRule), { s: bob }) === false);
  assert(MV.OfAny(mat(matlabRule), { s: { age: 20n, email: "x" } }) === true); // a mapping is a struct
  assert(MV.OfAny(mat(M.field(s, "age")), { s: ann }) === 30 && typeof MV.OfAny(mat(18n)) === "number"); // numbers are doubles
  assert(MV.OfAny(mat(M.binary("==", 1n, true))) === true && MV.OfAny(mat(M.binary("+", true, 1n))) === 2);
  assert(MV.OfAny(mat(M.binary("<", "apple", "banana"))) === true && MV.OfAny(mat(M.binary("==", "5", 5n))) === true);
  assert(MV.OfAny(mat(M.binary("==", "true", true))) === true && MV.OfAny(mat(M.binary("~=", "1.5", 1.5))) === false);
  assert(MV.OfAny(mat(M.binary("+", "a", 1n))) === "a1" && MV.OfAny(mat(M.binary("+", "x", 0.125))) === "x0.125");
  assert(same([NaN, Infinity, -Infinity, 12345.6, 1.5e-7, 123456.7, true, false].map((v) => MV.OfAny(mat(M.binary("+", "x", v)))), [
    "xNaN", "xInf", "x-Inf", "x12346", "x1.5e-07", "x1.2346e+05", "xtrue", "xfalse"])); // numbers as MATLAB writes them
  assert(MV.OfAny(mat(matlabRule), { s: new Map<string, unknown>([["age", 20n], ["email", "x"]]) }) === true); // a Map is a struct too
  assert(same(["~=", "<=", ">", ">=", "-", ".*"].map((op) => MV.OfAny(mat(M.binary(op, 2n, 3n)))), [true, true, false, false, -1, 6]));
  assert(MV.OfAny(mat(M.binary("&&", false, M.identifier("missing")))) === false); // short-circuits
  assert(MV.OfAny(mat(M.binary("||", 2n, M.identifier("missing")))) === true && MV.OfAny(mat(M.binary("||", 0n, 0n))) === false);
  assert(MV.OfAny(mat(M.unary("~", 0n))) === true && MV.OfAny(mat(M.unary("-", true))) === -1);
  assert(MV.OfAny(mat(M.call("isfield", 3n, "x"))) === false && MV.OfAny(mat(M.call("isfield", s, 1n)), { s: ann }) === false);
  raises(KeyError, () => MV.OfAny(mat(M.field(s, "email")), { s: bob }), 'Unrecognized field name "email".');
  raises(TypeError, () => MV.OfAny(mat(M.field(s, "age")), { s: 3n }), "Dot indexing is not supported for variables of this type.");
  raises(NameError, () => MV.OfAny(mat(M.identifier("missing"))), "Unrecognized function or variable 'missing'.");
  raises(TypeError, () => MV.OfAny(mat(M.binary("&&", "yes", true))), "Operands to the && operator must be convertible to logical scalar values.");
  raises(ValueError, () => MV.OfAny(mat(M.unary("~", NaN))), "NaN's cannot be converted to logicals.");
  raises(TypeError, () => MV.OfAny(mat(M.binary("-", "a", 1n))), "Operator '-' is not supported for operands of type 'string' and 'double'.");
  raises(TypeError, () => MV.OfAny(mat(M.binary(".*", s, true)), { s: ann }), "Operator '.*' is not supported for operands of type 'struct' and 'logical'.");
  raises(TypeError, () => MV.OfAny(mat(M.unary("-", "a"))), "Operator '-' is not supported for operands of type 'string'.");
  const adult = store.transaction(() => store.Expressions.Framework.Expression( // an expression, its root written by a writer
    (e: any) => e.root(M.binary(">", M.field(s, "age"), 18n))));
  assert(MText.ToText(adult) === "s.age > 18" && MV.OfAny(adult, { s: ann }) === true);
}

<a id="DIA-05"></a>
## DIA-05 · MATLAB resolves functions from imports, the path and packages that the scope provides

In [ ]:
{
  const geo = { dist: (a: number, b: number) => (a * a + b * b) ** 0.5, area: (r: number) => 3 * r * r };
  const scope = new MV.Scope({ r: 2n }, { functions: { twice: (v: number) => 2 * v },
    packages: { geo, "geo.more": { area: () => 0n } } });
  const wild = mat(M.import_("geo.*", M.binary("+", M.call("dist", 3n, 4n), M.call("area", M.identifier("r")))));
  assert(MText.ToText(mat(wild)) === "import geo.*\ndist(3, 4) + area(r)" && M.DIALECT.validate(mat(wild), ["r"]).length === 0);
  assert(same(M.DIALECT.validate(mat(wild), ["r"], true), ["body: left: 'dist' is not a core operation",
    "body: right: 'area' is not a core operation"]));
  assert(equal(SY.imports(wild, M.DIALECT).map((i) => F.form(i).attributes), [map({ name: "geo.*" })]));
  assert(MV.OfAny(mat(wild), scope) === 17);
  assert(MV.OfAny(mat(M.import_("geo.dist", M.call("dist", 6n, 8n))), scope) === 10);
  assert(MV.OfAny(mat(M.call("geo.more.area", 1n)), scope) === 0 && MV.OfAny(mat(M.call("twice", 4n)), scope) === 8); // an int result is a double
  assert(MV.OfAny(mat(M.import_("geo.*", M.import_("geo.more.area", M.call("area", 5n)))), scope) === 0); // innermost wins
  raises(NameError, () => MV.OfAny(mat(M.call("dist", 3n, 4n)), scope), "Unrecognized function or variable 'dist'."); // not imported
  raises(ImportError, () => MV.OfAny(mat(M.import_("os.system", 1n)), scope), "Import argument 'os.system' cannot be found or cannot be imported.");
  raises(ImportError, () => MV.OfAny(mat(M.import_("geo.nope", 1n)), scope), "Import argument 'geo.nope' cannot be found or cannot be imported.");
  raises(ImportError, () => MV.OfAny(mat(M.import_("nope.*", 1n)), scope), "Import argument 'nope.*' cannot be found or cannot be imported.");
  raises(ValueError, () => MText.ToText(mat(M.binary("+", 1n, M.import_("geo.*", 2n)))), "an import can only enclose the whole expression");
  assert(same(M.DIALECT.validate(mat(M.import_("geo", 1n))), ["an import's name must be pkg.name or pkg.*, got 'geo'"]));
  assert(same(M.DIALECT.validate(mat(M.import_("geo.*.x", 1n))), ["an import's name must be pkg.name or pkg.*, got 'geo.*.x'"]));
  assert(same(M.DIALECT.validate(mat(M.import_(7n as any, M.constant(1n)))), ["an import's name must be a str, got int"]));
}

<a id="DIA-06"></a>
## DIA-06 · Excel: formulas with `LET`, functions, operators and fields

In [ ]:
const r = X.name("r");
const excelRule = xl(X.let_("age", X.field(r, "age"), X.function("AND", X.infix(">=", X.name("age"), 18n),
  X.function("NOT", X.function("ISERROR", X.field(r, "email"))))));
{
  assert(XText.ToText(excelRule) === "=LET(age, r.age, AND(age >= 18, NOT(ISERROR(r.email))))" && roundTrips(excelRule));
  assert(XText.ToText(xl(X.infix("*", X.infix("+", 1n, 2n), X.prefix("-", X.infix("-", 3n, 4n))))) === "=(1 + 2) * -(3 - 4)");
  assert(XText.ToText(xl(X.infix("=", X.infix("-", 1n, X.infix("-", 2n, 3n)), -1n))) === "=1 - (2 - 3) = -1");
  assert(XText.ToText(xl(X.prefix("-", X.field(r, "first name")))) === "=-r.[first name]");
  assert(same([true, false, 'say "hi"', Infinity, 2.5, 3n].map((v) => XText.ToText(xl(X.constant(v)))), [
    "=TRUE", "=FALSE", '="say ""hi"""', "=#NUM!", "=2.5", "=3"]));
  assert(X.DIALECT.validate(xl(excelRule), ["r"], true).length === 0
    && same(X.DIALECT.validate(xl(X.function("VLOOKUP", 1n, 2n)), [], true), ["'VLOOKUP' is not a core operation"]));
  assert(X.DIALECT.infer(xl(excelRule), { r: XD.Record }) === XD.Logical && X.DIALECT.infer(xl(X.infix("+", "1", true))) === XD.Number);
  assert(XD.of(1n) === XD.Number && XD.Errors.contains(XD.Error.of("#N/A")) && String(XD.Error.of("#N/A")) === "#N/A");
  assert(XD.Record.contains(ann) && !XD.Record.contains(1n));
  assert([true, 'say "hi"', 2.5, 3n, -0, NaN].every((v) => roundTrips(xl(X.constant(v)))));
  assert(same([...SY.free(excelRule, X.DIALECT)], ["r"]));
  raises(TypeError, () => X.constant(new Uint8Array([0])), "an Excel literal cannot hold a bytes");
  for (const [spec, problems] of [
    [(u: any) => u.literal((c: any) => c.value((v: any) => v.as_scalar(new Uint8Array([1])))), ["an Excel literal cannot hold a bytes"]],
    [(u: any) => u.cell((c: any) => c.sheet("S")), ["a cell needs an address"]],
    [(u: any) => u.call((c: any) => c.arguments([])), ["a call needs a callable"]],
    [(u: any) => u.infix((c: any) => c.operator("+").left(X.constant(1n))), ["an infix needs a right"]],
    [X.infix("+", null, 1n), ["an infix needs a left"]], // a writer leaves out a slot given null
    [(u: any) => u.literal((c: any) => c), ["a literal needs a value"]],
    [(u: any) => u.map((c: any) => c.name("p")), ["a map needs a value", "a map needs a body"]],
    [X.name("u"), ["variable 'u' is not bound"]],
    [X.function("NOT", 1n, 2n), ["NOT takes 1 arguments, got 2"]],
  ] as [unknown, string[]][]) {
    assert(same(X.DIALECT.validate(xl(spec)), problems), Repr.repr(X.DIALECT.validate(xl(spec))));
  }
  assert(same(X.DIALECT.validate(xl(X.infix("^", 1n, 2n)), [], true), ["'^' is not a core operation"]));
  const quantified = term(Patterns.Expressions.variable("xs").all("p", true), Patterns.Expressions.DIALECT); // a Patterns kind Excel does not read
  assert(same(X.DIALECT.validate(quantified), ["a Patterns quantifier is not an Excel term"]));
  raises(ValueError, () => XText.ToText(quantified), "a Patterns variable is not an Excel term"); // printing reads its terms first
  raises(ValueError, () => XText.ToText(xl(Patterns.Expressions.operation("SUM", Patterns.Expressions.variable("x"), 1n))),
    "a Patterns variable is not an Excel term"); // the same kinds, of another dialect
}

<a id="DIA-07"></a>
## DIA-07 · Excel coerces, orders values of every type, and computes with errors

In [ ]:
{
  const Error = XV.Error;
  assert(XV.OfAny(xl(excelRule), { r: ann }) === true && XV.OfAny(xl(excelRule), { r: bob }) === false); // ISERROR finds the #FIELD!
  assert(XV.OfAny(xl(X.field(r, "email")), { r: bob }) === Error.of("#FIELD!") && XV.OfAny(xl(X.name("nope"))) === Error.of("#NAME?"));
  assert(XV.OfAny(xl(X.field(r, "age")), { r: { age: 5n } }) === 5 && XV.OfAny(xl(X.field(r, "x")), { r: 3n }) === Error.of("#VALUE!"));
  assert(XV.OfAny(xl(X.field(X.field(r, "a"), "b")), { r: {} }) === Error.of("#FIELD!")); // errors propagate
  assert(XV.OfAny(xl(X.infix("+", "1", true))) === 2 && XV.OfAny(xl(X.infix("*", "x", 1n))) === Error.of("#VALUE!")); // coercion
  assert(XV.OfAny(xl(X.infix("-", 1n, X.infix("+", X.name("u"), 1n)))) === Error.of("#NAME?"));
  assert(XV.OfAny(xl(X.prefix("-", "2"))) === -2 && XV.OfAny(xl(X.prefix("-", X.name("u")))) === Error.of("#NAME?"));
  assert(XV.OfAny(xl(X.infix(">", "a", 1n))) === true && XV.OfAny(xl(X.infix(">", true, "z"))) === true); // number < text < logical
  assert(XV.OfAny(xl(X.infix("=", "a", "A"))) === true && XV.OfAny(xl(X.infix("<>", 1n, "1"))) === true); // case; no coercion
  assert(same(["<", "<=", ">=", "-"].map((op) => XV.OfAny(xl(X.infix(op, 2n, 3n)))), [true, true, false, -1]));
  assert(XV.OfAny(xl(X.infix("=", 2n, 2n))) === true && XV.OfAny(xl(X.infix("+", false, 1n))) === 1);
  assert(XV.OfAny(xl(X.infix("+", "inf", 1n))) === Error.of("#VALUE!") && XV.OfAny(xl(X.infix("+", " 1e3 ", 1n))) === 1001); // decimal text
  assert(XV.OfAny(xl(X.infix("+", r, 1n)), { r: {} }) === Error.of("#VALUE!"));
  assert(XV.OfAny(xl(X.field(r, "age")), { r: new Map([["age", 5n]]) }) === 5); // a Map is a record too
  assert(XV.OfAny(xl(X.infix("=", X.name("u"), 1n))) === Error.of("#NAME?") && XV.OfAny(xl(X.infix("=", r, 1n)), { r: {} }) === Error.of("#VALUE!"));
  assert(XV.OfAny(xl(X.function("AND", 1n, true))) === true && XV.OfAny(xl(X.function("OR", 0n, false))) === false);
  assert(XV.OfAny(xl(X.function("AND", false, X.name("u")))) === Error.of("#NAME?")); // AND evaluates every argument
  assert(XV.OfAny(xl(X.function("OR", "yes", true))) === Error.of("#VALUE!") && XV.OfAny(xl(X.function("NOT", 0n))) === true);
  assert(XV.OfAny(xl(X.function("NOT", X.name("u")))) === Error.of("#NAME?"));
  assert(XV.OfAny(xl(X.function("IF", true, 1n, X.name("u")))) === 1 && XV.OfAny(xl(X.function("IF", 0n, 1n, "no"))) === "no");
  assert(XV.OfAny(xl(X.function("IF", X.name("u"), 1n, 2n))) === Error.of("#NAME?")); // IF evaluates only the branch it takes
  assert(XV.OfAny(xl(X.field(r, "age")), { r: ann }) === 30);
  assert(XV.OfAny(xl(X.function("ISERROR", 1n))) === false && XV.OfAny(xl(X.let_("x", 2n, X.infix("*", X.name("x"), 3n)))) === 6);
  const adult = store.transaction(() => store.Expressions.Framework.Expression( // an expression, its root written by a writer
    (e: any) => e.root(X.function("AND", X.infix(">=", X.field(r, "age"), 18n)))));
  assert(XV.OfAny(adult, { r: ann }) === true && XText.ToText(adult) === "=AND(r.age >= 18)");
}

<a id="DIA-08"></a>
## DIA-08 · Excel reads cells from workbooks, and add-in functions from the workbook's add-ins

In [ ]:
{
  const Error = XV.Error;
  const rates = new XV.Workbook({}, { Rates: { B2: 0.5 } }, { name: "Rates.xlsx" });
  const book = new XV.Workbook({ base: 10n }, { Sheet1: { a1: 2n, $A$2: 3n }, "My Sheet": { C3: "x" } },
    { books: { "Rates.xlsx": rates }, add_ins: { double: (v: number) => v * 2 } });
  const total = xl(X.infix("+", X.infix("*", X.cell("A1"), X.cell("B2", "Rates", "Rates.xlsx")), X.function("DOUBLE", X.name("base"))));
  assert(XText.ToText(xl(total)) === "=A1 * [Rates.xlsx]Rates!B2 + DOUBLE(base)" && XV.OfAny(xl(total), book) === 21 && roundTrips(total));
  assert(X.DIALECT.validate(xl(total), ["base"]).length === 0
    && same(X.DIALECT.validate(xl(total), ["base"], true), ["right: 'DOUBLE' is not a core operation"]));
  assert(XText.ToText(xl(X.cell("C3", "My Sheet"))) === "='My Sheet'!C3" && XV.OfAny(xl(X.cell("C3", "My Sheet")), book) === "x");
  assert(XText.ToText(xl(X.cell("A1", "Bob's", "My Book.xlsx"))) === "='[My Book.xlsx]Bob''s'!A1");
  assert(XV.OfAny(xl(X.cell("$a$2")), book) === 3 && XV.OfAny(xl(X.cell("Z9")), book) === 0); // an empty cell is 0
  assert(XV.OfAny(xl(X.cell("A1", "Sheet1", "Book1")), book) === 2); // this book, named
  assert(XV.OfAny(xl(X.cell("A1", "Nope")), book) === Error.of("#REF!") && XV.OfAny(xl(X.cell("A1", "S", "Other.xlsx")), book) === Error.of("#REF!"));
  assert(XV.OfAny(xl(X.function("NOPE", 1n)), book) === Error.of("#NAME?")
    && XV.OfAny(xl(X.function("double", X.name("u"))), book) === Error.of("#NAME?")); // add-ins ignore case; errors in their arguments propagate
  assert(XV.OfAny(xl(X.cell("A1")), new XV.Workbook()) === Error.of("#REF!")); // a workbook has no sheet until given one
  assert(XV.OfAny(xl(X.name("x")), { x: 1n }) === 1 && new XV.Workbook().sheet === "Sheet1");
  raises(ValueError, () => new XV.Workbook({}, { Sheet1: { total: 1n } }), "not a cell address: 'total'");
  assert(same(X.DIALECT.validate(xl(X.cell("A0"))), ["a cell's address must be a column and a row, like A1, got 'A0'"]));
  assert(same(X.DIALECT.validate(xl(X.cell("A1", null, "B.xlsx"))), ["a cell in another book needs a sheet"]));
  assert(X.DIALECT.infer(xl(X.cell("A1"))) === D.Anything && X.address("$b$12") === "B12" && X.address("B") === null);
}

<a id="DIA-09"></a>
## DIA-09 · LaTeX: notation in math mode, rendered and checked, with no evaluator

In [ ]:
{
  const [L, LD] = [Latex.Expressions, Latex.Domains];
  const [a, person] = [L.symbol("a"), L.symbol("person")];
  const notation = tex(L.where("a", L.member(person, "age"), L.binary("\\land", L.binary("\\geq", a, 18n), L.binary(
    "\\lor", L.unary("\\lnot", L.function("has", person, "email")), L.binary(">", L.frac(a, 2n), 1.5)))));
  assert(LText.ToText(tex(notation)) === "a \\geq 18 \\land (\\lnot \\operatorname{has}(\\mathit{person}, \\text{email}) \\lor \\frac{a}{2} > 1.5) "
    + "\\quad \\text{where } a = \\mathit{person}.\\mathit{age}" && roundTrips(notation));
  assert(LText.ToText(tex(L.binary("\\cdot", L.binary("+", 1n, 2n), L.unary("-", L.binary("-", 3n, -4n))))) === "(1 + 2) \\cdot -(3 - -4)");
  assert(LText.ToText(tex(L.binary("\\implies", L.binary("\\implies", 1n, 2n), L.binary("\\implies", 3n, 4n))))
    === "(1 \\implies 2) \\implies 3 \\implies 4"); // right-associative
  assert(LText.ToText(tex(L.binary("<", L.binary("<", 1n, 2n), 3n))) === "(1 < 2) < 3"); // comparisons do not chain
  assert(LText.ToText(tex(L.member(L.binary("+", person, 1n), "x_y"))) === "(\\mathit{person} + 1).\\mathit{x\\_y}");
  assert(LText.ToText(tex(L.where("b", 1n, L.where("a", 2n, a)))) === "(a \\quad \\text{where } a = 2) \\quad \\text{where } b = 1");
  assert(same([NaN, Infinity, -Infinity, 1e-7, 1e16, -2n, true, false, "a_b {c}"].map((v) => LText.ToText(tex(L.constant(v)))), [
    "\\mathrm{NaN}", "\\infty", "-\\infty", "1 \\times 10^{-7}", "1 \\times 10^{16}", "-2", "\\mathrm{true}",
    "\\mathrm{false}", "\\text{a\\_b \\{c\\}}"]));
  assert(LText.ToText(tex(L.constant("\\$&#%~^"))) === "\\text{\\textbackslash{}\\$\\&\\#\\%\\textasciitilde{}\\textasciicircum{}}");
  assert(LText.ToText(tex(L.binary("\\cdot", L.constant(1e16), 2n))) === "1 \\times 10^{16} \\cdot 2");
  assert(L.DIALECT.validate(tex(notation), ["person"], true).length === 0
    && same(L.DIALECT.validate(tex(L.binary("\\times", 1n, 2n)), [], true), ["'\\\\times' is not a core operation"]));
  assert(same(L.DIALECT.validate(notation), ["value: value: variable 'person' is not bound",
    "body: right: left: operand: argument 0: variable 'person' is not bound"]));
  assert(same(L.DIALECT.validate(tex((u: any) => u.binary((r: any) => r.operator("+").left(L.DIALECT.spec(1n))))), ["a binary needs a right"]));
  assert(same(L.DIALECT.validate(tex((u: any) => u.variable((r: any) => r))), ["a variable needs a name"]));
  assert(same(L.DIALECT.validate(tex(L.constant(new Uint8Array([0])))), ["a LaTeX literal cannot hold a bytes"])); // the framework's literal, LaTeX's check
  assert(L.DIALECT.validate(tex(L.function("max", 1n, 2n))).length === 0 && same(L.DIALECT.validate(tex(L.function("has", 1n))), ["has takes 2 arguments, got 1"]));
  assert(L.DIALECT.infer(tex(notation), { person: D.Anything }) === LD.Truth && L.DIALECT.infer(tex(L.frac(1n, 2.5))) === LD.Number);
  raises(TypeError, () => L.DIALECT.infer(tex(L.binary("\\land", 1n, true))), "\\land cannot take (number, truth)");
  raises(TypeError, () => L.DIALECT.spec(new Uint8Array([120])), "expected a term, a writer or a native value, got b'x'");
  assert(F.same(F.form(tex(L.frac(notation, 1n))).arguments[0], notation)); // a term written into another is copied
  assert(LD.of("x") === LD.Text && !("Evaluators" in Latex)); // notation is not evaluated
}

<a id="DIA-10"></a>
## DIA-10 · Bitwise operations: Python's operators, and MATLAB's and Excel's bit functions

In [ ]:
{
  // Python: rendered with Python's precedence, and evaluated by its rules (DIA-02's tables have more).
  const [a, b, c, d, x] = ["a", "b", "c", "d", "x"].map((name) => P.name(name)) as [any, any, any, any, any];
  const sources: [unknown, string][] = [
    [P.binop("|", a, P.binop("^", b, P.binop("&", c, P.binop(">>", P.binop("<<", d, 1n), 2n)))), "a | b ^ c & d << 1 >> 2"],
    [P.binop("&", P.binop("|", a, b), c), "(a | b) & c"], [P.binop("+", P.unaryop("~", x), 1n), "~x + 1"],
    [P.binop("+", P.binop("<<", a, b), c), "(a << b) + c"], [P.compare("==", P.binop("&", x, 1n), 1n), "x & 1 == 1"],
    [P.unaryop("-", P.unaryop("~", x)), "-~x"],
  ];
  for (const [expression, source] of sources) assert(PText.ToText(py(expression)) === source, source);
  const mixed = py(P.binop("^", P.binop("|", P.binop("&", x, 12n), P.binop(">>", x, 2n)), P.binop("<<", x, 1n)));
  assert(PText.ToText(mixed) === "(x & 12 | x >> 2) ^ x << 1" && PV.OfAny(mixed, { x: 13n }) === 21n);
  assert(P.DIALECT.infer(py(P.binop("&", true, false))) === PD.Bool && P.DIALECT.infer(py(P.binop("<<", 1n, true))) === PD.Int);
  assert(P.DIALECT.infer(py(P.unaryop("~", x)), { x: PD.Bool }) === PD.Int);
  // MATLAB: the bit functions take doubles holding integers of 53 bits.
  assert(MV.OfAny(mat(M.call("bitand", 12n, 10n))) === 8 && MV.OfAny(mat(M.call("bitor", 12n, true))) === 13);
  assert(MV.OfAny(mat(M.call("bitxor", 12n, 10n))) === 6 && MV.OfAny(mat(M.call("bitshift", 1n, 3n))) === 8);
  assert(MV.OfAny(mat(M.call("bitshift", 16n, M.unary("-", 2n)))) === 4 && MV.OfAny(mat(M.call("bitshift", 2 ** 52, 1n))) === 0); // beyond 53 bits
  assert(MText.ToText(mat(M.call("bitshift", M.identifier("x"), M.unary("-", 2n)))) === "bitshift(x, -2)");
  assert(M.DIALECT.infer(mat(M.call("bitand", 1n, true))) === MD.Double);
  for (const call of [M.call("bitand", -1n, 1n), M.call("bitand", 1.5, 1n), M.call("bitor", 2 ** 54, 1n), M.call("bitshift", 1n, 0.5)]) {
    raises(ValueError, () => MV.OfAny(mat(call)), "Double inputs must have integer values in the range of ASSUMEDTYPE.");
  }
  raises(TypeError, () => MV.OfAny(mat(M.call("bitor", "a", 1n))), "Undefined function 'bitor' for input arguments of type 'string'.");
  // Excel: the bit functions take integers below 2^48, converted as arithmetic converts; otherwise #NUM!.
  assert(XV.OfAny(xl(X.function("BITAND", 12n, 10n))) === 8 && XV.OfAny(xl(X.function("BITOR", "12", true))) === 13);
  assert(XV.OfAny(xl(X.function("BITXOR", 12n, 10n))) === 6 && XV.OfAny(xl(X.function("BITLSHIFT", 1n, 3n))) === 8);
  assert(XV.OfAny(xl(X.function("BITRSHIFT", 16n, 2n))) === 4 && XV.OfAny(xl(X.function("BITRSHIFT", 1n, -3n))) === 8);
  assert(XV.OfAny(xl(X.function("BITLSHIFT", 16n, -2n))) === 4 && XText.ToText(xl(X.function("BITXOR", X.name("x"), 1n))) === "=BITXOR(x, 1)");
  for (const formula of [X.function("BITAND", -1n, 1n), X.function("BITAND", 1.5, 1n), X.function("BITOR", 2n ** 48n, 1n),
    X.function("BITLSHIFT", 1n, 54n), X.function("BITLSHIFT", 1n, 48n), X.function("BITRSHIFT", 1n, 0.5)]) {
    assert(XV.OfAny(xl(formula)) === XV.Error.of("#NUM!"), XText.ToText(xl(formula)));
  }
  assert(XV.OfAny(xl(X.function("BITAND", "a", 1n))) === XV.Error.of("#VALUE!"));
  assert(XV.OfAny(xl(X.function("BITXOR", 1n, X.name("nope")))) === XV.Error.of("#NAME?"));
  assert(X.DIALECT.infer(xl(X.function("BITAND", 1n, 2n))) === XD.Number);
}

<a id="DIA-11"></a>
## DIA-11 · C: C and C++ expressions over their arithmetic types, with C's precedence, typed literals and casts

In [ ]:
{
  const [x, y, device] = [C.identifier("x"), C.identifier("y"), C.identifier("device")];
  const cases: [unknown, string][] = [
    [C.binary("+", x, C.binary("*", y, 2n)), "x + y * 2"], [C.binary("*", C.binary("+", x, y), 2n), "(x + y) * 2"],
    [C.binary("-", x, C.binary("-", y, 1n)), "x - (y - 1)"], [C.binary("<<", C.binary("+", x, 1n), 2n), "x + 1 << 2"],
    [C.binary("&", C.binary("==", x, 1n), y), "x == 1 & y"], [C.binary("==", C.binary("&", x, 1n), y), "(x & 1) == y"],
    [C.binary("||", C.binary("&&", x, y), C.unary("!", x)), "x && y || !x"],
    [C.binary("|", C.binary("^", x, y), C.binary("%", x, y)), "x ^ y | x % y"],
    [C.binary("<", C.binary(">>", x, 1n), C.binary("/", y, 2n)), "x >> 1 < y / 2"],
    [C.conditional(x, y, C.conditional(y, 1n, 2n)), "x ? y : y ? 1 : 2"], [C.conditional(C.conditional(x, y, 1n), 2n, 3n), "(x ? y : 1) ? 2 : 3"],
    [C.unary("-", C.unary("-", x)), "- -x"], [C.unary("-", -1n), "- -1"], [C.unary("~", C.unary("!", x)), "~!x"],
    [C.cast(named_type("uint8_t"), C.binary("+", x, 1n)), "(uint8_t)(x + 1)"], [C.unary("-", C.cast(keywords("int"), x)), "-(int)x"],
    [C.member(C.member(device, "a"), "b", "->"), "device.a->b"], [C.subscript(C.member(device, "ports"), C.binary("+", x, 1n)), "device.ports[x + 1]"],
    [C.call("f", x, C.binary("+", y, 1n)), "f(x, y + 1)"], [C.member(C.binary("+", x, y), "a"), "(x + y).a"],
    [C.literal(5n, "u"), "5u"], [C.literal(5n, "ul"), "5ul"], [C.literal(5n, "ll"), "5ll"], [C.literal(5n, "ULL"), "5ULL"],
    [C.literal(5n, "l"), "5l"], [C.literal(5n), "5"], [C.literal(1.5, "f"), "1.5f"], [C.literal(1.5, "L"), "1.5L"],
    [C.literal(1.0, "L"), "1.0L"], [C.literal(Infinity, "L"), "INFINITY"], [C.cast(named_type("uint8_t"), 5n), "(uint8_t)5"],
    [C.cast(keywords("unsigned", "long", "long"), 5n), "(unsigned long long)5"], [C.literal(2.5), "2.5"],
    [C.literal(1.0), "1.0"], [C.literal(1e300), "1e+300"], [C.literal(NaN), "NAN"], [C.literal(-Infinity), "-INFINITY"], [C.literal(Infinity), "INFINITY"],
    [C.literal(true), "true"], [C.literal(false), "false"], [C.literal('a"\\\n\t\r'), '"a\\"\\\\\\n\\t\\r"'],
    [C.literal(-5n), "-5"], [C.binary("-", x, -5n), "x - -5"],
  ];
  for (const [expression, source] of cases) assert(CText.ToText(cc(expression)) === source, `${source}: ${CText.ToText(cc(expression))}`);
  const sample = cc(C.conditional(C.binary("&&", C.unary("!", C.member(device, "a")), C.binary(
    ">", C.subscript(C.member(device, "b", "->"), 0n), C.literal(1.5, "f"))), C.cast(named_type("uint8_t"), C.call("f", 1n)), C.literal(5n, "u")));
  assert(roundTrips(sample) && CText.ToText(sample) === "!device.a && device->b[0] > 1.5f ? (uint8_t)f(1) : 5u");
  // Validation: what a literal's text encodes, each its own vocabulary, and its value, of a type its suffix and base
  // allow; a member's operator; a callee is a term, its name bound as a variable's is.
  assert(same(C.DIALECT.validate(cc(C.literal(5n, "q"))), ["a literal's suffix cannot be 'q'"]));
  assert(same(C.DIALECT.validate(cc(C.literal(2n ** 70n))), ["integer literal 1180591620717411303424 does not fit any type"]));
  assert(same(C.DIALECT.validate(cc(C.literal(2n ** 70n, "u"))), ["integer literal 1180591620717411303424 does not fit 'unsigned long long'"]));
  assert(same(C.DIALECT.validate(cc(C.literal(-5n, "u"))), ["integer literal -5 does not fit 'unsigned long long'"]));
  assert(C.DIALECT.validate(cc(C.literal(2n ** 64n - 1n, "u"))).length === 0 && C.DIALECT.validate(cc(C.literal(2n ** 63n))).length === 0); // the last that holds it
  assert(same(C.DIALECT.validate(cc((u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(300n)).quote("'").prefix("u8")))), [
    "integer literal 300 does not fit 'unsigned char'"]));
  assert(same(C.DIALECT.validate(cc(C.literal("a", "u"))), ["a literal's suffix cannot be 'u'"])); // a string has none
  assert(same(C.DIALECT.validate(cc(C.literal(1.5, "u"))), ["a literal's suffix cannot be 'u'"])); // nor a float an integer's
  assert(["U", "lu", "LLU", "wb", "uwb"].every((suffix) => C.DIALECT.validate(cc(C.literal(5n, suffix))).length === 0));
  assert(["F", "l", "df", "DL", "f32", "F128x"].every((suffix) => C.DIALECT.validate(cc(C.literal(1.5, suffix))).length === 0));
  assert(same(C.DIALECT.validate(cc(C.member(x, "a", "=>")), ["x"]), ["a memberexpression's operator must be '.' or '->', got '=>'"]));
  assert(same(C.DIALECT.validate(cc(C.binary("**", x, 1n)), ["x"], true), ["'**' is not a core operation"]));
  assert(C.DIALECT.validate(cc(C.call("f", x)), ["x", "f"], true).length === 0); // the scope's functions, bound by name
  assert(same(C.DIALECT.validate(cc(C.call("f", x)), ["x"]), ["function: variable 'f' is not bound"]));
  assert(same(C.DIALECT.validate(cc((u: any) => u.castexpression((r: any) => r))), ["a castexpression needs a type", "a castexpression needs an operand"]));
  assert(same(C.DIALECT.validate(cc(C.literal(null))), ["a literal needs a value"]));
  raises(ValueError, () => CText.ToText(cc(C.literal(null))), "C literal has no value to print"); // an invalid term has no source
  raises(TypeError, () => C.DIALECT.spec(new Uint8Array([120])), "expected a term, a writer or a native value, got b'x'"); // no C literal holds bytes
  assert(same(C.DIALECT.validate(cc(C.binary("+", x, 1n))), ["left: variable 'x' is not bound"]));
  // A literal's range is its stored type's, read through its holder: C's own types, then its holder's ABI's; a holder
  // naming no ABI defines no `int`, and a lone term reaches no store
  assert(same(C.DIALECT.validate(term(C.literal(5n), C.DIALECT, null)), ["'int' is not a type C defines, and its expression names no ABI"]));
  assert(C.DIALECT.validate(term(C.literal(5n, "wb"), C.DIALECT, null)).length === 0 && C.DIALECT.validate(term(C.literal(1.5), C.DIALECT, null)).length === 0); // read none
  assert(C.DIALECT.validate(term(C.literal(300n), C.DIALECT, AAPCS64_)).length === 0 && same(C.DIALECT.validate(term(C.literal(2n ** 70n), C.DIALECT, AAPCS64_)), [
    "integer literal 1180591620717411303424 does not fit any type"]));
  const lone = store.transaction(() => store.Expressions.Framework.Term(C.literal(5n)));
  raises(TypeError, () => C.DIALECT.validate(lone), "the value does not belong to a store");
  // Types: each a stored domain, read through a term's holder, among C's own (`int32_t`) and then among the ABI's its
  // holder names (`int`, `size_t`): here x86-64's System V ABI, LP64. The integer promotions and the usual arithmetic
  // conversions.
  const T = new CD.Types(cc(0n));
  const N = (name: string) => T.named(name);
  assert(N("int32_t").equals(N("int")) && N("int32_t").name() === "int32_t" && String(N("size_t")) === "size_t");
  assert(N("size_t").equals(N("unsigned long")) && N("int").types === T && N("int") === N("int")); // read once
  assert(N("int").domain === D.domains(AMD64_).get("int") && N("int32_t").domain === D.domains(store.Instances.Expressions.C.Expressions).get("int32_t"));
  assert(CD.promoted(N("uint8_t")) === N("int") && CD.promoted(N("bool")) === N("int") && CD.promoted(N("long")) === N("long"));
  assert(CD.common(N("int"), N("unsigned int"))!.name() === "unsigned int" && CD.common(N("long"), N("unsigned int"))!.name() === "long");
  assert(CD.common(N("float"), N("long"))!.name() === "float" && CD.common(N("double"), N("long double"))!.name() === "long double");
  assert(CD.common(N("unsigned long"), N("long long"))!.equals(N("unsigned long")) && CD.common(N("short"), N("char")) === N("int"));
  assert(CD.common(N("int"), N("long long"))!.name() === "long long" && CD.common(N("long"), N("int"))!.name() === "long");
  assert(CD.common(N("_Decimal32"), N("double")) === null && CD.common(N("_Decimal64"), N("int"))!.name() === "_Decimal64"); // decimal
  assert(CD.common(N("_Decimal32"), N("_Decimal128"))!.name() === "_Decimal128" && N("_Decimal32").decimal && !N("double").decimal);
  assert(same([1n, 2n ** 40n, 2n ** 63n, true, 1.5, "a"].map((v) => CD.of(v, T)), ["int", "long", "unsigned long", "bool", "double", null]));
  assert(CD.of(2n ** 64n, T) === null && CD.of(null, T) === null);
  assert(N("uint8_t").contains(255n) && !N("uint8_t").contains(256n) && N("int8_t").contains(-128n) && !N("int8_t").contains(-129n));
  assert(N("float").contains(1.5) && N("long double").contains("1.5") && N("bool").contains(true) && !N("int").contains(true));
  assert(N("_Float16").contains(1.5) && N("_Float128").contains("1.5") && N("_Decimal32").contains("1.5")); // text, where not binary16 to 64
  assert(!N("int").contains(1.5) && N("int").includes(N("int32_t")) && N("int").includes(new D.OfUnion(N("int"))));
  assert(!N("int").includes(N("long")) && equal(N("long double").format, [2n, 64, 16383]) && N("long double").width === 80); // x87's
  assert(T.label(N("int32_t")).name() === "int" && T.label(N("_Float32x")).name() === "double" && T.label(N("_Float16")).name() === "_Float16");
  assert(N("long double").equals(N("_Float64x")) && !N("long double").equals(N("double"))); // by format and width
  assert(T.find("my_t") === null && T.find("nullptr_t") === null); // a program's own type; a type that is not arithmetic
  assert(C.untyped(2n ** 64n) === D.Anything); // a literal no type holds is not known (validation refuses it before inference)
  raises(ValueError, () => T.named("my_t"), "'my_t' is not a type C or AMD64 defines");
  // A type name's arithmetic type, its keywords in any order, cv-qualifiers aside, by its name; a type's name is itself
  for (const [names, named] of [[["unsigned"], "unsigned int"], [["long", "unsigned", "int"], "unsigned long"], [["const", "char"], "char"],
    [["signed", "char"], "signed char"], [["unsigned", "char"], "unsigned char"], [["short", "int"], "short"],
    [["long", "long"], "long long"], [["signed"], "int"], [["long", "double"], "long double"], [["_Bool"], "bool"],
    [["double"], "double"], [["float"], "float"], [["uint8_t"], "uint8_t"], [["volatile", "size_t"], "size_t"],
    [["bool"], "bool"], [["my_t"], "my_t"]] as [string[], string][]) {
    assert(CD.arithmetic(names) === named, names.join(" "));
  }
  for (const names of [[], ["void"], ["signed", "unsigned"], ["int", "int"], ["long", "long", "long"], ["short", "long"], ["long", "float"],
    ["unsigned", "double"], ["long", "char"], ["char", "int"], ["unsigned", "unsigned"], ["uint8_t", "int"],
    ["long", "long", "double"], ["unsigned", "float"], ["long", "bool"], [null]] as (string | null)[][]) {
    assert(CD.arithmetic(names) === null, String(names));
  }
  // A literal's type: what its text encodes, by C's rules, a stored domain found by its name
  const typed = (attributes: Record<string, unknown>) => CD.literal(map(attributes), "C", T);
  const CPT = new CD.Types(term(0n, CppDialect.Expressions.DIALECT), "Cpp.Expressions"); // a C++ term's
  const cppTyped = (attributes: Record<string, unknown>) => CD.literal(map(attributes), "C++", CPT);
  assert(typed({ value: 5n }) === N("int") && typed({ value: 1.5 }) === N("double") && typed({ value: true }) === N("bool")); // its value's
  assert(typed({}) === null && typed({ value: 2n ** 64n }) === null); // no value, and a bigint no type holds
  assert(typed({ value: 5n, suffix: "u" }) === N("unsigned int") && typed({ value: 2n ** 32n, suffix: "U" }) === N("unsigned long"));
  assert(typed({ value: 5n, suffix: "LL" }) === N("long long"));
  assert(typed({ value: 0xFFFFFFFFn, base: 16n }) === N("unsigned int") && typed({ value: 2n ** 31n, base: 10n, suffix: "l" }) === N("long"));
  assert(typed({ value: 5n, base: 8n }) === N("int"));
  assert(typed({ value: 2n ** 64n, suffix: "u" }) === N("unsigned long long")); // none holds it: the last, which evaluation refuses
  assert(typed({ value: 1.5, suffix: "f" }) === N("float") && typed({ value: 1.5, suffix: "L" }) === N("long double"));
  assert(typed({ value: 1.5, suffix: "f16" })!.name() === "_Float16");
  assert(typed({ value: 1.5, suffix: "DF" })!.name() === "_Decimal32" && typed({ value: 1.5, suffix: "f64x" })!.name() === "_Float64x");
  assert(typed({ value: 1.5, suffix: "q" }) === D.Anything);
  assert(cppTyped({ value: 1.5, suffix: "f16" })!.name() === "std::float16_t" && cppTyped({ value: 1.5, suffix: "bf16" })!.name() === "std::bfloat16_t");
  assert(cppTyped({ value: 1.5, suffix: "df" }) === D.Anything); // C++ writes no decimal floating literal: unknown
  assert(typed({ value: 5n, suffix: "wb" })!.name() === "_BitInt" && typed({ value: 5n, suffix: "uwb" })!.name() === "unsigned _BitInt");
  assert(typed({ value: 5n, suffix: "z" })!.name() === "long" && typed({ value: 5n, suffix: "q" }) === D.Anything); // a suffix no language writes: unknown
  assert(typed({ value: 97n, quote: "'" }) === N("int") && cppTyped({ value: 97n, quote: "'" })!.name() === "char"); // a character: its code
  assert(typed({ value: 97n, quote: "'", prefix: "u8" }) === N("unsigned char") && cppTyped({ value: 97n, quote: "'", prefix: "u8" })!.name() === "char8_t");
  assert(typed({ value: 97n, quote: "'", prefix: "L" }) === N("wchar_t") && typed({ value: "a" }) === CD.String);
  assert(typed({ value: "a", prefix: "u" })!.name() === "char16_t string");
  assert(typed({ value: "a", prefix: "Q" })!.name() === "Q string" && typed({ value: 97n, quote: "'", prefix: "Q" })!.name() === "Q"); // a prefix no language has
  raises(ValueError, () => typed({ value: 1.5, suffix: "f128x" }), "'_Float128x' is not a type C or AMD64 defines"); // neither C nor this ABI provides it
  const infer = (expression: unknown, environment?: Record<string, D.Domain>) => C.DIALECT.infer(cc(expression), environment);
  const nameOf = (domain: D.Domain) => domain.name();
  assert(nameOf(infer(C.binary("+", C.cast(named_type("uint8_t"), 1n), 1n))) === "int" && nameOf(infer(C.binary("+", 1n, C.literal(1n, "u")))) === "unsigned int");
  assert(nameOf(infer(C.binary("<<", C.cast(named_type("uint8_t"), 1n), C.literal(1n, "l")))) === "int" && nameOf(infer(C.binary("<", 1n, 2.0))) === "bool");
  assert(nameOf(infer(C.binary("&&", x, 1n), { x: D.Anything })) === "bool" && infer(C.binary("+", x, 1n), { x: D.Anything }) === D.Anything);
  assert(nameOf(infer(C.binary("==", x, y), { x: D.Anything, y: D.Anything })) === "bool"); // a comparison's type, whatever its operands'
  assert(N("uint16_t").equals(infer(C.cast(named_type("uint16_t"), x), { x: D.Anything })) && N("uint8_t").equals(infer(C.cast(named_type("uint8_t"), 5n))));
  assert(nameOf(infer(C.unary("~", C.cast(named_type("uint8_t"), 1n)))) === "int" && nameOf(infer(C.unary("-", 1.5))) === "double");
  assert(nameOf(infer(C.unary("!", 1.5))) === "bool" && nameOf(infer(C.unary("!", x), { x: D.Anything })) === "bool");
  assert(nameOf(infer(C.binary("%", C.cast(named_type("uint32_t"), 7n), C.cast(named_type("uint32_t"), 2n)))) === "unsigned int");
  assert(nameOf(infer(C.binary("+", C.cast(named_type("int32_t"), 7n), C.cast(named_type("int64_t"), 2n)))) === "long"); // named as C writes it
  assert(nameOf(infer(C.binary("*", C.literal(1.5, "f16"), C.literal(2.5, "f16")))) === "_Float16" && nameOf(infer(C.literal(1.5, "dd"))) === "_Decimal64");
  assert(infer(C.cast(named_type("my_t"), 1n)) === D.Anything); // a type a program declares: not arithmetic
  assert(infer(C.conditional(x, 1n, 2n), { x: D.Anything }) === D.Anything && infer(C.member(x, "a"), { x: D.Anything }) === D.Anything);
  assert(infer(C.call("f", 1n), { f: D.Anything }) === D.Anything && infer(C.literal("a")) === CD.String);
  const refused: [unknown, string][] = [[C.binary("%", 1.5, 1n), "% cannot take (double, int); it takes (integer, integer) -> their common type"],
    [C.binary("+", "a", 1n), "+ cannot take (string, int)"], [C.binary("<<", 1.5, 1n), "<< cannot take (double, int)"],
    [C.unary("~", 1.5), "~ cannot take (double); it takes (integer) -> its promoted type"],
    [C.binary("+", C.literal(1.5, "df"), 1.5), "+ cannot take (_Decimal32, double)"]]; // decimal and binary
  for (const [expression, message] of refused) raises(TypeError, () => infer(expression), message);
}

<a id="DIA-12"></a>
## DIA-12 · C evaluates by C's rules: promotions and conversions, wrapping unsigned arithmetic, and undefined behavior raising

In [ ]:
{
  /** The value of a C term, read under `abi` (x86-64's System V ABI unless given). */
  const ev = (expression: unknown, variables: Record<string, unknown> = {}, abi: unknown = AMD64_) =>
    CV.OfAny(term(expression, C.DIALECT, abi), variables) as any;
  const x = C.identifier("x");
  const U = (value: bigint) => C.literal(value, "u");
  assert(ev(C.binary("+", U(4294967295n), 1n)) === 0n && ev(C.binary("-", U(0n), 1n)) === 4294967295n && ev(C.binary("*", U(65536n), U(65536n))) === 0n);
  assert(ev(C.binary("<", -1n, U(0n))) === false && ev(C.binary("<", -1n, C.cast(keywords("unsigned", "short"), 0n))) === true); // promoted to int
  assert(ev(C.binary("+", C.cast(named_type("uint8_t"), 200n), C.cast(named_type("uint8_t"), 100n))) === 300n); // in int
  assert(ev(C.binary("/", -7n, 2n)) === -3n && ev(C.binary("%", -7n, 2n)) === -1n && ev(C.binary("/", 7n, -2n)) === -3n && ev(C.binary("%", 7n, -2n)) === 1n);
  assert(ev(C.binary("/", 7.0, 2n)) === 3.5 && ev(C.binary("/", 1.0, 0.0)) === Infinity && Number.isNaN(ev(C.binary("/", 0.0, 0.0))));
  assert(ev(C.binary("+", C.cast(keywords("float"), 1.5), C.cast(keywords("float"), 0.1))) === 1.600000023841858 && ev(C.binary("-", 1.5, 1n)) === 0.5);
  assert(ev(C.binary("*", C.cast(keywords("long", "double"), 1.5), 2n)) === "3");
  assert(ev(C.binary("&", 12n, 10n)) === 8n && ev(C.binary("^", 12n, 10n)) === 6n && ev(C.binary("|", -1n, U(0n))) === 4294967295n);
  assert(ev(C.binary("<<", U(1n), 31n)) === 2147483648n && ev(C.binary(">>", -8n, 1n)) === -4n && ev(C.binary("<<", C.cast(named_type("uint8_t"), 1n), 8n)) === 256n);
  assert(ev(C.unary("-", U(1n))) === 4294967295n && ev(C.unary("~", C.cast(named_type("uint8_t"), 0n))) === -1n && ev(C.unary("~", U(0n))) === 4294967295n);
  assert(ev(C.unary("+", true)) === 1n && ev(C.unary("-", 1.5)) === -1.5 && ev(C.unary("+", 1.5)) === 1.5);
  assert(ev(C.unary("!", 0.0)) === true && ev(C.unary("!", -0)) === true && ev(C.unary("!", NaN)) === false && ev(C.unary("!", 3n)) === false);
  assert(ev(C.binary("==", NaN, NaN)) === false && ev(C.binary("!=", NaN, 1.0)) === true && ev(C.binary("==", -0, 0.0)) === true);
  assert(ev(C.binary("<", 1n, 1.5)) === true && ev(C.binary(">=", C.cast(keywords("long", "double"), 2n), 1n)) === true && ev(C.binary(">", 2n, 1n)) === true);
  assert(ev(C.binary("<=", Infinity, Infinity)) === true && ev(C.binary("<", C.cast(keywords("long", "double"), NaN), 1n)) === false);
  assert(ev(C.binary("&&", 0n, C.binary("/", 1n, 0n))) === false && ev(C.binary("||", 2n, C.binary("/", 1n, 0n))) === true); // decided early
  assert(ev(C.binary("&&", 1n, 2.5)) === true && ev(C.binary("||", 0n, 0.0)) === false);
  assert(ev(C.conditional(C.binary(">", x, 0n), C.binary("/", 10n, x), 0n), { x: 0n }) === 0n && ev(C.conditional(1n, 2n, 3.5)) === 2n); // the chosen one
  assert(ev(C.cast(named_type("uint8_t"), 300n)) === 44n && ev(C.cast(named_type("int8_t"), 200n)) === -56n && ev(C.cast(keywords("int"), 2.9)) === 2n && ev(C.cast(keywords("int"), -2.9)) === -2n);
  assert(ev(C.cast(keywords("bool"), 0.5)) === true && ev(C.cast(keywords("bool"), 0n)) === false && ev(C.cast(keywords("bool"), -0)) === false);
  assert(ev(C.cast(keywords("double"), 2n ** 60n + 1n)) === 2 ** 60 && ev(C.cast(keywords("float"), 0.1)) === 0.10000000149011612 && ev(C.cast(keywords("int"), true)) === 1n);
  assert(ev(C.cast(keywords("long", "double"), 0.1)) === "0.10000000000000000555" && ev(C.cast(keywords("double"), C.cast(keywords("float"), 0.1))) === 0.10000000149011612);
  assert(ev(C.cast(named_type("uint8_t"), 5n)) === 5n && ev(C.literal("hi")) === "hi" && ev(C.literal(true)) === true);
  // A literal's type is what its text encodes: a float rounded to its type, a long double's as its text
  assert(ev(C.literal(0.1, "f")) === 0.10000000149011612 && ev(C.literal(1.5, "L")) === "1.5" && ev(C.literal(0xFFFFFFFFn, "u")) === 4294967295n);
  assert(ev((u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(97n)).quote("'"))) === 97n); // 'a', an int in C
  // What C leaves undefined throws.
  const undefinedBehavior: [unknown, Function, string][] = [
    [C.binary("+", 2147483647n, 1n), FE.OverflowError, "signed overflow in +: 2147483648 does not fit 'int'"],
    [C.unary("-", -2147483648n), FE.OverflowError, "signed overflow in -: 2147483648 does not fit 'int'"],
    [C.binary("/", -2147483648n, -1n), FE.OverflowError, "signed overflow in /: 2147483648 does not fit 'int'"],
    [C.binary("%", 1n, 0n), FE.ZeroDivisionError, "integer division by zero"],
    [C.binary("<<", 1n, 32n), ValueError, "shift by 32 is undefined for 'int'"], [C.binary(">>", 1n, -1n), ValueError, "shift by -1 is undefined for 'int'"],
    [C.binary("<<", -1n, 1n), ValueError, "left shift of a negative value is undefined"],
    [C.binary("<<", 1n, 31n), FE.OverflowError, "signed overflow in <<: 2147483648 does not fit 'int'"],
    [C.cast(named_type("uint8_t"), 300.0), FE.OverflowError, "300.0 is out of range of 'uint8_t'"],
    [C.cast(keywords("int"), Infinity), FE.OverflowError, "inf is out of range of 'int'"],
    [C.cast(keywords("int"), C.cast(keywords("long", "double"), 1e300)), FE.OverflowError, " is out of range of 'int'"],
    [C.cast(keywords("int"), NaN), ValueError, "NaN has no integer value"],
    [C.binary("%", 1.5, 1n), TypeError, "invalid operands to binary % ('double' and 'double')"],
    [C.binary("<<", 1.5, 1n), TypeError, "invalid operands to binary << ('double' and 'int')"],
    [C.unary("~", 1.5), TypeError, "invalid argument type 'double' to unary expression"],
    [C.binary("+", "a", 1n), TypeError, "invalid operand to +: 'str'"], [C.unary("!", "a"), TypeError, "invalid operand to !: 'str'"],
  ];
  for (const [expression, error, message] of undefinedBehavior) raises(error, () => ev(expression), message);
  // A variable's type is its value's (a value carries no domain: one of another type waits for domain inference).
  raises(FE.OverflowError, () => ev(x, { x: 2n ** 64n }), "integer 18446744073709551616 is too large for any type");
  assert(ev(C.cast(named_type("uint8_t"), C.binary("+", x, 1n)), { x: 255n }) === 0n && ev(C.binary("+", x, x), { x: 1.5 }) === 3.0);
  assert(ev(C.binary("+", x, 1n), { x: 2n ** 40n }) === 2n ** 40n + 1n);
  assert(ev(C.binary("*", x, 2n), { x: -5n }) === -10n && ev(C.literal(5n)) === 5n); // no suffix: its value's
  // Members, elements and calls.
  assert(ev(C.member(x, "age"), { x: ann }) === 30n && ev(C.member(x, "age", "->"), { x: { age: 5n } }) === 5n);
  assert(ev(C.member(x, "age"), { x: new Map([["age", 7n]]) }) === 7n);
  assert(ev(C.subscript(x, 1n), { x: [10n, 20n] }) === 20n && ev(C.subscript(C.subscript(C.member(x, "grid"), 1n), 0n), { x: board }) === 3n);
  const refused: [unknown, unknown, Function, string][] = [
    [C.member(x, "phone"), ann, KeyError, "no member named 'phone'"],
    [C.member(x, "a"), 1n, TypeError, "member reference base type 'int' is not a structure"],
    [C.subscript(x, 2n), [10n, 20n], ValueError, "index 2 is out of bounds of an array of 2"],
    [C.subscript(x, 0n), 1n, TypeError, "subscripted value is not an array: 'int'"],
    [C.subscript(x, 0.5), [1n], TypeError, "array subscript is not an integer: 'double'"],
    [C.call("f", 1n), null, FE.NameError, "use of undeclared identifier 'f'"], [C.identifier("z"), null, FE.NameError, "use of undeclared identifier 'z'"],
    [C.literal(2n ** 64n, "u"), null, FE.OverflowError, "integer literal 18446744073709551616 does not fit 'unsigned long long'"],
  ];
  for (const [expression, variable, error, message] of refused) raises(error, () => ev(expression, { x: variable }), message);
  assert(ev(C.member(x, "name"), { x: board }) === "b");
  // an expression, its root written by a writer, C's holder naming the ABI its terms are read under
  const expression = store.transaction(() => store.Expressions.C.Expressions.Expression((e: any) => e.root(C.binary("+", x, 1n)).abi([AMD64_])));
  assert(CText.ToText(expression) === "x + 1" && CV.OfAny(expression, { x: 41n }) === 42n);
  assert(CV.OfAny(cc(C.call("max", x, 3n)), new CV.Scope({ x: 5n }, { functions: { max: (a: bigint, b: bigint) => (a > b ? a : b) } })) === 5n);
  // What C evaluated before it was complete keeps evaluating; any other kind, and what an evaluated kind holds beyond its
  // former shape, is refused by name.
  const star = (u: any) => u.castexpression((r: any) => r.type((t: any) => t.typeid((i: any) => i.specifiers([ // (char *)x
    (s: any) => s.primitivetypespecifier((p: any) => p.keyword("char"))]).declarator((d: any) => d.pointerdeclarator((p: any) => p)))).operand(x));
  for (const [expression, message] of [
    [(u: any) => u.sizeofexpression((r: any) => r.operand(x)), "C sizeofexpression is not evaluated"],
    [(u: any) => u.parenthesizedexpression((r: any) => r.expression(x)), "C parenthesizedexpression is not evaluated"],
    [C.unary("*", x), "C unaryexpression '*' is not evaluated"], [C.binary(",", x, 1n), "C binaryexpression ',' is not evaluated"],
    [C.literal(5n, "wb"), "C literal of _BitInt is not evaluated"], [star, "C castexpression to a type that is not arithmetic is not evaluated"],
    [C.cast(named_type("my_t"), 1n), "C castexpression to a type that is not arithmetic is not evaluated"], // a program's own type
    [C.subscript(x, 0n, 1n), "C subscriptexpression of several indices is not evaluated"],
    [C.call(C.member(x, "f"), 1n), "C callexpression of a callee that is not a function's name is not evaluated"],
    [(u: any) => u.conditionalexpression((r: any) => r.condition(x).alternative(C.literal(1n))), "C conditionalexpression without a consequence is not evaluated"],
  ] as [unknown, string][]) raises(NotImplementedError, () => ev(expression, { x: [1n] }), message);
  assert(same([...CV.EVALUATED], ["literal", "variable", "unaryexpression", "binaryexpression", "conditionalexpression", "castexpression",
    "memberexpression", "subscriptexpression", "callexpression"]));
  // Every type is a stored domain, read through the term's holder: the same term under each ABI its holder names
  // evaluates as that ABI says, where they differ: plain char signed or not, wchar_t an int or an unsigned int, long double
  // x87's 80-bit format or binary128
  for (const [expression, amd64, aapcs64] of [
    [C.cast(keywords("char"), 200n), -56n, 200n], [C.cast(named_type("wchar_t"), -1n), -1n, 4294967295n],
    [C.binary("/", C.literal(1.0, "L"), 3n), "0.33333333333333333334", "0.3333333333333333333333333333333333"],
    [C.binary("/", C.literal(1.0, "f64x"), 3n), "0.33333333333333333334", "0.3333333333333333333333333333333333"],
    [C.cast(keywords("long", "double"), 0.1), "0.10000000000000000555", "0.1000000000000000055511151231257827"],
    [C.binary("<", C.cast(keywords("char"), 200n), 0n), true, false],
  ] as [unknown, unknown, unknown][]) assert(ev(expression) === amd64 && ev(expression, {}, AAPCS64_) === aapcs64, String(amd64));
  assert(ev(C.binary("+", C.cast(keywords("int"), 1n), 2n), {}, AAPCS64_) === 3n && ev(C.binary("/", C.literal(1.0, "f32x"), 3n)) === 1 / 3); // alike
  // The types C's standard fixes are C's own: _FloatN and the decimal types evaluate in their formats, a decimal one with a
  // binary one refused, as C's usual arithmetic conversions do not apply
  assert(ev(C.binary("+", C.literal(0.1, "f16"), C.literal(0.2, "f16"))) === 0.2998046875 && ev(C.literal(0.1, "df")) === "0.1000000");
  assert(ev(C.binary("*", C.literal(0.1, "dd"), 3n)) === "0.3000000000000000" && ev(C.binary("/", C.literal(1.0, "f128"), 3n), {}, AAPCS64_) === "0.3333333333333333333333333333333333");
  assert(ev(C.binary("+", C.literal(1.5, "DL"), 1n)) === "2.5" && ev(C.binary("==", C.literal(0.5, "df"), C.literal(0.5, "dd"))) === true);
  assert(ev((u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(97n)).quote("'").prefix("L"))) === 97n); // L'a', a wchar_t
  for (const [expression, error, message] of [
    [C.binary("+", C.literal(0.1, "df"), 1.0), TypeError, "invalid operands to binary + ('_Decimal32' and 'double')"],
    [C.literal(1.0, "f128x"), ValueError, "'_Float128x' is not a type C or AMD64 defines"], // neither provides it
  ] as [unknown, Function, string][]) raises(error, () => ev(expression), message);
  // A holder naming no ABI defines none of the types C leaves to the implementation, and a lone term reaches no store:
  // each refused where it reads a type; what reads none evaluates
  assert(ev(C.literal(true), {}, null) === true && ev(C.literal(1.5, "f16"), {}, null) === 1.5 && ev(C.literal("hi"), {}, null) === "hi");
  for (const [expression, message] of [[C.literal(5n), "'int' is not a type C defines, and its expression names no ABI"],
    [C.literal(1.5), "'double' is not a type C defines, and its expression names no ABI"],
    [C.cast(keywords("long"), 1n), "'long' is not a type C defines, and its expression names no ABI"]] as [unknown, string][]) {
    raises(ValueError, () => ev(expression, {}, null), message);
  }
  const [lone, loneString] = store.transaction(() => [store.Expressions.Framework.Term(C.literal(5n)), store.Expressions.Framework.Term(C.literal("hi"))]);
  raises(TypeError, () => CV.OfAny(lone), "the value does not belong to a store");
  assert(CV.OfAny(loneString) === "hi");
}

<a id="DIA-13"></a>
## DIA-13 · SystemVerilog: expressions over 4-state vectors and reals, with IEEE 1800's precedence, sized literals and casts

In [ ]:
{
  const [x, y, v] = [SV.identifier("x"), SV.identifier("y"), SV.vector];
  const atom = (keyword: string) => (t: any) => t.integeratomtype((a: any) => a.keyword(keyword)); // a type, `int`, written through the builders
  const signing = (sign: string) => (t: any) => t.implicittype((i: any) => i.signing(sign)); // `signed`, a type of a signing alone
  const cases: [unknown, string][] = [
    [SV.binary("+", x, SV.binary("*", y, 2n)), "x + y * 2"], [SV.binary("*", SV.binary("+", x, y), 2n), "(x + y) * 2"],
    [SV.binary("-", x, SV.binary("-", y, 1n)), "x - (y - 1)"], [SV.binary("**", SV.binary("**", x, y), 2n), "x ** y ** 2"],
    [SV.binary("**", x, SV.binary("**", y, 2n)), "x ** (y ** 2)"], [SV.binary("<<", SV.binary("+", x, 1n), 2n), "x + 1 << 2"],
    [SV.binary("&&", SV.binary("||", x, y), x), "(x || y) && x"], [SV.binary("||", SV.binary("&&", x, y), x), "x && y || x"],
    [SV.binary("^~", SV.binary("|", x, y), SV.binary("&", x, y)), "(x | y) ^~ x & y"],
    [SV.binary("==", SV.binary("&", x, 1n), y), "(x & 1) == y"], [SV.binary("<", SV.binary(">>>", x, 1n), SV.binary("%", y, 2n)), "x >>> 1 < y % 2"],
    [SV.binary("->", SV.binary("->", x, y), x), "(x -> y) -> x"], [SV.binary("<->", x, SV.binary("->", y, x)), "x <-> y -> x"],
    [SV.binary("->", SV.conditional(x, y, 1n), y), "x ? y : 1 -> y"], [SV.conditional(SV.binary("->", x, y), 1n, 2n), "(x -> y) ? 1 : 2"],
    [SV.conditional(x, y, SV.conditional(y, 1n, 2n)), "x ? y : y ? 1 : 2"], [SV.conditional(SV.conditional(x, y, 1n), 2n, 3n), "(x ? y : 1) ? 2 : 3"],
    [SV.unary("-", SV.unary("-", x)), "-(-x)"], [SV.unary("&", SV.unary("&", x)), "&(&x)"], [SV.unary("~", SV.unary("!", x)), "~(!x)"],
    [SV.unary("-", -1n), "-(-1)"], [SV.binary("-", x, -1n), "x - -1"], [SV.unary("~&", SV.binary("+", x, 1n)), "~&(x + 1)"],
    [SV.inside(SV.binary("+", x, 1n), 1n, SV.span(2n, 5n)), "x + 1 inside {1, [2:5]}"], [SV.binary("<", SV.inside(x, 1n), 0n), "x inside {1} < 0"],
    [SV.binary("+", SV.inside(x, 1n), 1n), "(x inside {1}) + 1"], [SV.binary("==?", x, v("1x")), "x ==? 2'b1x"],
    [v("11111111", null, "h"), "8'hff"], [v("10x1", null, "h"), "4'b10x1"], [v("1xxxx", null, "h"), "5'h1x"], [v("1zzz", null, "h"), "4'b1zzz"],
    [v("00000101", true, "d"), "8'sd5"], [v("xxxx", null, "d"), "4'dx"], [v("0x", null, "d"), "2'b0x"], [v("111", null, "o"), "3'o7"],
    [v("10z1"), "4'b10z1"], [v("1", true), "1'sb1"],
    [SV.literal(1.5), "1.5"], [SV.literal(1.0), "1.0"], [SV.literal(1e300), "1e+300"], [SV.literal(NaN), "0.0 / 0.0"],
    [SV.literal(Infinity), "1.0 / 0.0"], [SV.literal(-Infinity), "-1.0 / 0.0"], [SV.binary("*", x, Infinity), "x * (1.0 / 0.0)"],
    [SV.unary("-", Infinity), "-(1.0 / 0.0)"], [SV.literal('a"\\\n\t'), '"a\\"\\\\\\n\\t"'], [SV.literal(-5n), "-5"],
    [SV.cast(signing("signed"), x), "signed'(x)"], [SV.cast(8n, SV.binary("+", x, 1n)), "8'(x + 1)"], [SV.cast(atom("byte"), -1n), "byte'(-1)"],
    [SV.member(SV.member(x, "a"), "b"), "x.a.b"], [SV.select(SV.binary("+", x, y), 0n), "(x + y)[0]"], [SV.range_(x, 7n, 4n), "x[7:4]"],
    [SV.concatenation(x, SV.replication(3n, y)), "{x, {3{y}}}"], [SV.call("$clog2", x), "$clog2(x)"], [SV.call("f", x, 1n), "f(x, 1)"],
  ];
  for (const [expression, source] of cases) assert(SVText.ToText(sv(expression)) === source, `${source}: ${SVText.ToText(sv(expression))}`);
  const sample = sv(SV.binary("->", SV.inside(SV.member(x, "a"), 1n, SV.span(2n, 5n)), SV.call("$clog2", SV.iterate(
    SV.method(x, "unique"), "sum", "q", SV.identifier("q")))));
  assert(roundTrips(sample) && SVText.ToText(sample) === "x.a inside {1, [2:5]} -> $clog2(x.unique().sum(q) with (q))");
  assert(SV.DIALECT.validate(sample, SY.free(sample, SV.DIALECT)).length === 0 && equal(F.form(sample).attributes, map({ operator: "->" })));
  // Validation: a constant's bits and base, and a cast's type or width.
  assert(same(SV.DIALECT.validate(sv(v("12"))), ["a constant's bits must be 0, 1, x and z, at least one, got '12'"]));
  assert(same(SV.DIALECT.validate(sv(v(""))), ["a constant's bits must be 0, 1, x and z, at least one, got ''"]));
  assert(same(SV.DIALECT.validate(sv(v("1", null, "q"))), ["a constant's base must be b, o, d or h, got 'q'"]));
  assert(same(SV.DIALECT.validate(sv(SV.cast((t: any) => t.namedtype((n: any) => n.name((i: any) => i.identifier((d: any) => d.spelling("word")))), x)),
    ["x"]), [])); // a type a program declares: no domain is read
  assert(same(SV.DIALECT.validate(sv(SV.cast(0n, x)), ["x"]), ["a cast's width must be positive, got 0"]));
  assert(same(SV.DIALECT.validate(sv((u: any) => u.castexpression((r: any) => r.value(x))), ["x"]), ["a castexpression needs a type"]));
  assert(same(SV.DIALECT.validate(sv(SV.binary("<<=", x, 1n)), ["x"], true), ["'<<=' is not a core operation"]));
  assert(SV.DIALECT.validate(sv(SV.call("f", x)), ["f", "x"], true).length === 0 && same(SV.DIALECT.validate(sv(SV.identifier("x"))), ["nameexpression 'x' is not bound"]));
  assert(same(SV.DIALECT.validate(sv(SV.literal(true))), ["a SystemVerilog literal cannot hold a bool"])
    && same(SV.DIALECT.validate(sv(SV.literal(new Uint8Array([0])))), ["a SystemVerilog literal cannot hold a bytes"]));
  assert(same(SV.DIALECT.validate(sv((u: any) => u.literal((r: any) => r))), ["a literal needs a value"]));
  assert(same(SV.DIALECT.validate(sv((u: any) => u.arraymethodwithexpression((r: any) => r.value(SV.method(x, "sum")))), ["x"]), [
    "an arraymethodwithexpression needs a body"])); // its iterator is `item`: bound within its body
  assert(same(SV.DIALECT.validate(sv(SV.iterate(x, "sum", "p", SV.identifier("q"))), ["x"]), ["body: nameexpression 'q' is not bound"]));
  raises(TypeError, () => SV.DIALECT.spec(true), "expected a term, a writer or a native value, got True"); // a bool is no SystemVerilog constant
  raises(ValueError, () => SVText.ToText(term(Patterns.Expressions.let_("a", 1n, Patterns.Expressions.variable("a")), Patterns.Expressions.DIALECT)), "a Patterns let is not a SystemVerilog term");
  // Types: integral vectors of a width, a signedness and 2 or 4 states, and reals, each named type read from the domain
  // SystemVerilog's define stores (its kind's properties), a sized vector's computed.
  const T = (name: string) => SVD.named(store, name);
  const widths = ["bit", "logic", "reg", "byte", "shortint", "int", "longint", "integer", "time"].map((n) => T(n).width);
  assert(same(widths, [1, 1, 1, 8, 16, 32, 64, 32, 64]));
  assert(T("int").signed && T("int").states === 2 && T("integer").states === 4 && !T("time").signed && T("logic").equals(T("reg")));
  assert(T("int").name() === "int" && String(T("integer")) === "integer" && SVD.vector(8).name() === "logic [7:0]");
  assert(SVD.vector(4, true, 2).name() === "bit signed [3:0]" && SVD.vector(32, true, 2).equals(T("int")) && T("shortreal").format === "binary32");
  assert(SVD.common([T("byte"), T("int")]).equals(T("int")) && SVD.common([T("byte"), SVD.vector(16)]).equals(SVD.vector(16)));
  assert(SVD.common([T("shortreal"), T("int")]).equals(T("real")) && SVD.common([T("shortreal"), T("shortreal")]).equals(T("shortreal")));
  assert(SVD.common([T("bit"), T("logic")]).equals(T("logic")));
  const ofs = [1n, -(2n ** 31n), 2n ** 31n, 2n ** 63n, 1.5, "a", true].map((value) => SVD.of(value));
  assert(same(ofs, ["integer", "integer", "longint", null, "real", null, null])); // a constant's type, by name
  assert(T("int").domain === store.Instances.Expressions.SystemVerilog.Expressions.domains({ name: "int" }) && SVD.vector(8).domain === null);
  // A named type is a stored domain: a store where SystemVerilog's define has not run defines none, and a type no stored
  // domain names is refused by name; a lone term reaches no store where it reads one.
  const bare: any = new Builders.Store();
  for (const install of [Framework.install, ISOIEC11404.install, IEEE754.install, SystemVerilog.install, SVTypes.install]) bare.transaction(install);
  const bareCast = F.term(bare.transaction(() => bare.Expressions.Framework.Expression((e: any) => e.root(SV.DIALECT.spec(SV.cast(atom("int"), 1n))))));
  assert(same(SV.DIALECT.validate(bareCast), ["'int' is not a type SystemVerilog defines"]));
  raises(ValueError, () => SVD.named(bare, "int"), "'int' is not a type SystemVerilog defines");
  const lone = store.transaction(() => store.Expressions.Framework.Term(SV.DIALECT.spec(SV.binary("+", 1n, SV.vector("10"))))); // held by no expression
  raises(TypeError, () => SV.DIALECT.infer(lone), "the value does not belong to a store");
  assert(same(SV.DIALECT.validate(lone), []) && SVD.vector(2).equals(SV.DIALECT.infer(F.form(lone).arguments[1]))); // it reads no type
  // Values: a Logic is a type and its bits, encoded as VPI does.
  const a = SVD.Logic.of("10xz");
  assert(a.aval === 0b1010n && a.bval === 0b0011n && a.bits === "10xz" && !a.known && String(a) === "4'b10xz");
  assert(SVD.Logic.of("10xz", true, 2).bits === "1000" && String(SVD.Logic.of("1", true)) === "1'sb1");
  assert(SVD.Logic.of("1110", true).integer() === -2n && SVD.Logic.of("1110").integer() === 14n && SVD.Logic.number(T("byte"), -1n).bits === "11111111");
  raises(ValueError, () => a.integer(), "4'b10xz has unknown bits");
  assert(T("int").contains(SVD.Logic.number(T("int"), 5n)) && !T("int").contains(SVD.Logic.number(T("integer"), 5n)) && !T("int").contains(5n));
  assert(T("real").contains(1.5) && !T("real").contains(1n) && T("int").includes(SVD.vector(32, true, 2)) && !T("int").includes(T("integer")));
  assert(T("int").includes(new D.OfUnion(T("int"))) && !T("int").includes(D.Anything));
  // Inference sizes an expression as IEEE 1800 does.
  const infer = (expression: unknown, environment?: Record<string, any>) => SV.DIALECT.infer(sv(expression), environment) as any;
  const is = (domain: unknown, expected: SystemVerilog.Domains.SvType) => expected.equals(domain);
  assert(is(infer(SV.binary("+", v("1010"), 1n)), SVD.vector(32)) && is(infer(SV.binary("+", 1n, 2n)), T("integer")));
  assert(is(infer(SV.binary("+", SV.cast(atom("byte"), 1n), SV.cast(atom("int"), 1n))), T("int")) && is(infer(SV.binary("*", 1n, 1.5)), T("real")));
  assert(is(infer(SV.binary("**", v("1010"), 2n)), SVD.vector(4)) && is(infer(SV.binary("**", 2n, 0.5)), T("real")));
  assert(is(infer(SV.binary("<<", v("1010"), 40n)), SVD.vector(4)) && is(infer(SV.binary("==", v("1010"), 1n)), SVD.vector(1)));
  assert(is(infer(SV.binary("===", v("1010"), 1n)), T("bit")) && is(infer(SV.binary("==?", v("1010"), 1n)), SVD.vector(1)));
  assert(is(infer(SV.binary("&&", x, 1n), { x: D.Anything }), SVD.vector(1)) && infer(SV.binary("+", x, 1n), { x: D.Anything }) === D.Anything);
  assert(is(infer(SV.binary("&", v("10"), v("1000"))), SVD.vector(4)) && is(infer(SV.unary("~", SV.cast(atom("byte"), 1n))), T("byte")));
  assert(is(infer(SV.unary("-", 1.5)), T("real")) && is(infer(SV.unary("!", 1.5)), SVD.vector(1)) && is(infer(SV.unary("^", v("10"))), SVD.vector(1)));
  assert(is(infer(SV.binary("%", 7n, 2n)), T("integer")) && is(infer(SV.cast(atom("int"), x), { x: D.Anything }), T("int")));
  assert(infer(SV.cast(8n, x), { x: D.Anything }) === D.Anything && infer(SV.cast(signing("signed"), 1n)) === D.Anything); // sized by the operand
  assert(is(infer(SV.inside(x, 1n), { x: D.Anything }), SVD.vector(1)) && infer(SV.conditional(x, 1n, 2n), { x: D.Anything }) === D.Anything);
  assert(SV.DIALECT.kinds().get("constant")!.typed!(F.kindOf(sv(v("12")))[1]) === null && is(infer(v("10x1", true)), SVD.vector(4, true)) && is(infer(SV.literal(1.5)), T("real")));
  assert(infer(SV.binary("+", "a", 1n)) === D.Anything); // a string has no type of its own
  raises(TypeError, () => infer(SV.binary("+", x, 1n), { x: new D.OfUnion(T("int"), T("real")) }),
    "+ cannot take (int | real, integer); it takes (T, T) -> their common type");
  assert(SVD.BINARY.get("+")?.describe() === "(T, T) -> their common type" && SVD.UNARY.get("!")?.arity() === 1);
  const refused: [unknown, string][] = [
    [SV.binary("%", 1.5, 1n), "% cannot take (real, integer); it takes (integral, integral) -> their common type"],
    [SV.binary("&", 1.5, 1n), "& cannot take (real, integer)"], [SV.binary("<<", 1.5, 1n), "<< cannot take (real, integer)"],
    [SV.binary("===", 1.5, 1n), "=== cannot take (real, integer)"], [SV.binary("==?", 1.5, 1n), "==? cannot take (real, integer)"],
    [SV.unary("~", 1.5), "~ cannot take (real); it takes (integral) -> its type"], [SV.unary("&", 1.5), "& cannot take (real)"],
  ];
  for (const [expression, message] of refused) raises(TypeError, () => infer(expression), message);
}

<a id="DIA-14"></a>
## DIA-14 · SystemVerilog evaluates by IEEE 1800's rules: sizing by context, 4-state logic, casts and system functions

In [ ]:
{
  const ev = (expression: unknown, variables: Record<string, unknown> = {}) => SVV.OfAny(sv(expression), variables) as any;
  const atom = (keyword: string) => (t: any) => t.integeratomtype((a: any) => a.keyword(keyword)); // a type, `int`, written through the builders
  const signing = (sign: string) => (t: any) => t.implicittype((i: any) => i.signing(sign)); // `signed`, a type of a signing alone
  /** A cast: to a width (a bigint, or any term), a signing, or a type SystemVerilog's keywords name, its type written
   * through the builders. */
  const c = (type: unknown, operand: unknown) => {
    if (typeof type !== "string") return SV.cast(type, operand);
    if (type === "signed" || type === "unsigned") return SV.cast(signing(type), operand);
    if (["real", "shortreal", "realtime"].includes(type)) return SV.cast((t: any) => t.nonintegertype((n: any) => n.keyword(type)), operand);
    if (["bit", "logic", "reg"].includes(type)) return SV.cast((t: any) => t.integervectortype((n: any) => n.keyword(type)), operand);
    return SV.cast(atom(type), operand);
  };
  const [v, b, u, call] = [SV.vector, SV.binary, SV.unary, SV.call];
  const x = SV.identifier("x");
  // Each expression and its value: a Logic by its text, or a real.
  const cases: [unknown, string | number][] = [
    [b("+", v("1010"), v("0110")), "4'b0000"], [b("+", v("1111"), 1n), "32'b" + "0".repeat(27) + "10000"],
    [b("/", -7n, 2n), "32'sb" + "1".repeat(30) + "01"], [b("%", -7n, 2n), "32'sb" + "1".repeat(32)],
    [b("/", 1n, 0n), "32'sb" + "x".repeat(32)], [b(">>>", v("11111110", true), 1n), "8'sb11111111"], [b(">>>", v("1000"), 1n), "4'b0100"],
    [b(">>", v("1000", true), 1n), "4'sb0100"], [b("&", v("10x1"), v("0011")), "4'b00x1"], [b("|", v("10z1"), v("1100")), "4'b11x1"],
    [b("^", v("10x1"), v("0011")), "4'b10x0"], [b("~^", v("1001"), v("0011")), "4'b0101"], [b("+", v("10x1"), 1n), "32'b" + "x".repeat(32)],
    [b("==", v("10x1"), v("10x1")), "1'bx"], [b("===", v("10x1"), v("10x1")), "1'b1"], [b("!==", v("10z1"), v("10x1")), "1'b1"],
    [b("==?", v("1001"), v("1xx1")), "1'b1"], [b("!=?", v("1001"), v("1xx0")), "1'b1"], [b("==?", v("1x01"), v("1001")), "1'bx"],
    [u("&", v("1111")), "1'b1"], [u("|", v("000x")), "1'bx"], [u("^", v("1011")), "1'b1"], [u("~&", v("1101")), "1'b1"],
    [u("~|", v("0000")), "1'b1"], [u("&", v("1x11")), "1'bx"], [u("|", v("0100")), "1'b1"], [u("~^", v("1x11")), "1'bx"], [u("~^", v("1011")), "1'b0"], [u("^", v("1x11")), "1'bx"], [u("&", v("0x11")), "1'b0"],
    [u("-", v("0001")), "4'b1111"], [u("~", v("10x1")), "4'b01x0"], [u("-", v("0x01")), "4'bxxxx"], [u("+", v("0x01")), "4'b0x01"],
    [b("**", 3n, -1n), "32'sb" + "0".repeat(32)], [b("**", 0n, -1n), "32'sb" + "x".repeat(32)],
    [b("**", -1n, -1n), "32'sb" + "1".repeat(32)], [b("**", -1n, -2n), "32'sb" + "0".repeat(31) + "1"],
    [b("**", 2n, 10n), "32'sb" + "0".repeat(21) + "10000000000"], [b("**", 1n, -3n), "32'sb" + "0".repeat(31) + "1"],
    [b("**", 2.0, 0.5), 2 ** 0.5], [b("/", 1.0, 0n), Infinity], [b("**", 2n, 0.5), 2 ** 0.5], [b("+", 1n, 0.5), 1.5],
    [SV.conditional(v("x"), v("1100"), v("1010")), "4'b1xx0"], [SV.conditional(v("1"), v("1100"), v("1010")), "4'b1100"],
    [SV.conditional(0n, v("1100"), v("1010")), "4'b1010"], [SV.conditional(v("z"), 1.5, 1.5), 1.5], [SV.conditional(v("z"), 1.5, 2.5), 0.0],
    [SV.inside(3n, 1n, SV.span(2n, 5n)), "1'b1"], [SV.inside(7n, 1n, SV.span(2n, 5n)), "1'b0"], [SV.inside(v("1x00"), v("1100")), "1'bx"],
    [SV.inside(v("1x00"), v("1100"), v("1x00")), "1'b1"], [SV.inside(v("1x00"), SV.span(0n, 3n)), "1'bx"],
    [SV.inside(v("1x00"), SV.span(8n, 15n)), "1'bx"], [SV.inside(v("1100"), v("1x00")), "1'b1"], [c("byte", 300n), "8'sb00101100"],
    [c("byte", 200n), "8'sb11001000"], [c("unsigned", -1n), "32'b" + "1".repeat(32)], [c(4n, v("11111111")), "4'b1111"],
    [c("int", 2.5), "32'sb" + "0".repeat(30) + "11"], [c("int", -2.5), "32'sb" + "1".repeat(30) + "01"], [c("real", 3n), 3.0],
    [c("shortreal", 0.1), 0.10000000149011612], [c("int", v("10x1")), "32'sb" + "0".repeat(28) + "1001"], [c("bit", v("x")), "1'b0"],
    [c("signed", v("1111")), "4'sb1111"], [c(8n, v("1111", true)), "8'sb11111111"], [c(8n, v("z1", true)), "8'sbzzzzzzz1"],
    [c(4n, v("x0", true)), "4'sbxxx0"], [call("$clog2", 8n), "32'sb" + "0".repeat(30) + "11"],
    [call("$clog2", 9n), "32'sb" + "0".repeat(29) + "100"], [call("$clog2", 0n), "32'sb" + "0".repeat(32)],
    [call("$clog2", v("1x")), "32'sb" + "x".repeat(32)], [call("$bits", v("00000000")), "32'sb" + "0".repeat(28) + "1000"],
    [call("$bits", 1.5), "32'sb" + "0".repeat(25) + "1000000"], [call("$countones", v("1011")), "32'sb" + "0".repeat(30) + "11"],
    [call("$onehot", v("0100")), "1'b1"], [call("$onehot", v("0000")), "1'b0"], [call("$onehot0", v("0000")), "1'b1"],
    [call("$isunknown", v("10x1")), "1'b1"], [call("$isunknown", v("1001")), "1'b0"], [call("$signed", v("1111")), "4'sb1111"],
    [call("$unsigned", v("1111", true)), "4'b1111"], [SV.concatenation(v("1010"), v("11")), "6'b101011"],
    [SV.replication(2n, v("10")), "4'b1010"], [SV.concatenation(v("x"), v("1")), "2'bx1"], [SV.select(v("1010"), 1n), "1'b1"],
    [SV.select(v("1010"), 4n), "1'bx"], [SV.select(v("1010"), v("x")), "1'bx"], [SV.range_(v("1010"), 3n, 2n), "2'b10"],
    [SV.range_(v("1010"), 5n, 2n), "4'bxx10"], [SV.select(c("int", v("1010")), 40n), "1'b0"],
    [b("+", v("1111", true), v("00000000", true)), "8'sb11111111"], [b("+", v("1111", true), v("00000000")), "8'b00001111"],
    [b("<", -1n, v("1")), "1'b0"], [b("<", -1n, 1n), "1'b1"], [b("==", b("+", v("1111"), v("0001")), v("10000")), "1'b1"],
    [b("==", b("<<", v("1000"), 1n), v("10000")), "1'b1"], [b("<<", v("0001"), v("10")), "4'b0100"],
    [b("<<", v("0001"), v("x")), "4'bxxxx"], [b("+", b("<", 2n, 1n), v("0001")), "4'b0001"], [b("+", b("<", 1n, 2n), v("0001")), "4'b0010"],
    [u("!", v("0000")), "1'b1"], [u("!", v("00x0")), "1'bx"], [u("!", 1.5), "1'b0"], [u("!", 0.0), "1'b1"], [b(">>>", v("x000", true), 1n), "4'sbxx00"], [b("&&", v("x"), 0n), "1'b0"],
    [b("||", v("x"), 1n), "1'b1"], [b("&&", v("x"), 1n), "1'bx"], [b("&&", 1n, 1n), "1'b1"], [b("||", 0n, 0n), "1'b0"], [b("&&", 0n, v("x")), "1'b0"],
    [b("||", 1n, v("x")), "1'b1"], [b("->", 0n, v("x")), "1'b1"], [b("->", 1n, v("x")), "1'bx"], [b("->", 1n, 0n), "1'b0"],
    [b("->", v("x"), 1n), "1'b1"], [b("<->", 1n, 1n), "1'b1"], [b("<->", 1n, 0n), "1'b0"], [b("<->", v("x"), 0n), "1'bx"],
    [b(">", 1.5, 1n), "1'b1"], [b("==", NaN, NaN), "1'b0"], [b("!=", NaN, 1.0), "1'b1"], [b("<", v("x"), 1n), "1'bx"], [u("-", 1.5), -1.5],
    [u("+", 1.5), 1.5], [b("-", c("shortreal", 1.5), c("shortreal", 0.25)), 1.25], [b("*", 3n, 4n), "32'sb" + "0".repeat(28) + "1100"],
    [b("-", 3n, 4n), "32'sb" + "1".repeat(32)], [b("%", 7n, -2n), "32'sb" + "0".repeat(31) + "1"],
  ];
  for (const [expression, value] of cases) {
    const result = ev(expression);
    assert(typeof value === "string" ? String(result) === value : result === value, `${SVText.ToText(sv(expression))}: ${result}`);
  }
  assert(Number.isNaN(ev(b("**", -8.0, 1 / 3))) && ev(b("**", 0.0, -1.0)) === Infinity && ev(b("**", 10.0, 400.0)) === Infinity);
  assert(ev(b("**", -10.0, 401.0)) === -Infinity && ev(b("**", -10.0, 400.0)) === Infinity && ev(b("**", c("shortreal", 2n), c("shortreal", 0.5))) === 1.4142135381698608);
  assert(ev(b("**", -0, -1.0)) === -Infinity && ev(b("**", -0, -2.0)) === Infinity && ev(b("**", 0.0, -3.0)) === Infinity);
  assert(ev(b("**", 1.0, NaN)) === 1.0 && ev(b("**", -1.0, Infinity)) === 1.0 && ev(b("**", NaN, 0.0)) === 1.0); // as C's pow
  assert(String(ev(b("/", c("int", 1n), c("int", 0n)))) === "32'sb" + "0".repeat(32) && String(ev(b("&", c("int", 5n), v("x")))) === "32'b" + "0".repeat(31) + "x");
  // Variables are typed by their values, Patterns' typed values among them.
  assert(String(ev(x, { x: true })) === "1'b1" && ev(x, { x: 5n }).type.equals(SVD.named(store, "integer")) && ev(x, { x: 2n ** 40n }).type.name() === "longint");
  assert(String(ev(b("<<", 1n, v("1".repeat(64))))) === "32'sb" + "0".repeat(32) && String(ev(b(">>>", -4n, v("1".repeat(64))))) === "32'sb" + "1".repeat(32));
  assert(String(ev(call("$bits", c("shortreal", 1.5)))) === "32'sb" + "0".repeat(26) + "100000" && String(ev(x, { x: "é" })) === "16'b1100001110101001");
  assert(String(ev(x, { x: "AB" })) === "16'b0100000101000010" && String(ev(x, { x: "" })) === "8'b00000000" && ev(x, { x: 1.5 }) === 1.5);
  const record = new Map([["a", 1n]]);
  assert(String(ev(x, { x: SVD.Logic.of("1x") })) === "2'b1x" && ev(x, { x: record }) === record); // a value carries no domain
  assert(String(ev(SV.member(x, "age"), { x: ann })) === "32'sb" + "0".repeat(26) + "011110" && String(ev(SV.member(x, "a"), { x: { a: true } })) === "1'b1");
  assert(String(ev(SV.select(x, 1n), { x: [1n, 2n] })) === "32'sb" + "0".repeat(30) + "10"
    && String(ev(SV.select(SV.member(x, "a"), 0n), { x: new Map([["a", [false]]]) })) === "1'b0");
  assert(String(ev(SV.select(SV.member(x, "ports"), 1n), { x: bus })) === "32'sb" + "0".repeat(29) + "101"); // data objects in the store
  const f = (a: any, b: any) => a.integer() + b.integer();
  assert((SVV.OfAny(sv(SV.call("f", x, 1n)), new SVV.Scope({ x: 2n }, { functions: { f } })) as any).type.name() === "integer");
  assert((SVV.OfAny(sv(b("+", 1n, 1n)), new SVV.Scope()) as any).integer() === 2n && (SVV.OfAny(sv(SV.binary("+", 1n, 2n))) as any).integer() === 3n);
  const expression = store.transaction(() => store.Expressions.Framework.Expression((e: any) => e.root(SV.binary("+", x, 1n))));
  assert((SVV.OfAny(expression, { x: 2n }) as any).integer() === 3n && SVText.ToText(expression) === "x + 1"); // evaluated from its root
  // Its types are the domains SystemVerilog defines in the store of the term's holder: a lone term is refused where it
  // needs one, and evaluates where it needs none.
  const [lone, sized] = store.transaction(() => [b("+", x, 1n), b("&", v("1010"), v("0110"))].map(
    (spec) => store.Expressions.Framework.Term(SV.DIALECT.spec(spec))));
  raises(TypeError, () => SVV.OfAny(lone, { x: 2n }), "the value does not belong to a store");
  assert(String(SVV.OfAny(sized)) === "4'b0010");
  const errors: [[unknown, Record<string, unknown>], Function, string][] = [
    [[x, {}], NameError, "identifier 'x' is not declared"], [[SV.call("f", 1n), {}], NameError, "function 'f' is not declared"],
    [[v("12"), {}], ValueError, "cannot evaluate an invalid expression: a constant's bits must be 0, 1, x and z, at least one, got '12'"],
    [[x, { x: 2n ** 70n }], OverflowError, "integer 1180591620717411303424 is too large for any type"],
    [[u("~", 1.5), {}], TypeError, "~ expects an integral operand, got real"],
    [[u("&", 1.5), {}], TypeError, "the reduction & expects an integral operand, got real"],
    [[b("%", 1.5, 1n), {}], TypeError, "% expects integral operands, got real"],
    [[b("&", 1.5, 1n), {}], TypeError, "& expects integral operands, got real"],
    [[b("<<", 1.5, 1n), {}], TypeError, "<< expects an integral operand, got real"],
    [[b("<<", 1n, 1.5), {}], TypeError, "<< expects integral operands, got real"],
    [[b("+", x, 1n), { x: new Map() }], TypeError, "+ expects a number, got dict"],
    [[c("int", x), { x: new Map() }], TypeError, "a cast expects a number, got dict"],
    [[SV.conditional(1n, x, 1n), { x: new Map() }], TypeError, "?: expects a number, got dict"],
    [[u("!", x), { x: new Map() }], TypeError, "a dict has no truth value"],
    [[c("signed", 1.5), {}], TypeError, "signed' expects an integral operand, got real"],
    [[c("int", NaN), {}], OverflowError, "nan has no integer value"],
    [[c("int", -Infinity), {}], OverflowError, "-inf has no integer value"],
    [[SV.member(x, "b"), { x: new Map([["a", 1n]]) }], KeyError, "no member named 'b'"],
    [[SV.member(x, "b"), { x: 1n }], TypeError, "a Logic has no members"],
    [[SV.select(x, 2n), { x: [1n, 2n] }], ValueError, "index 2 is out of bounds of an array of 2"],
    [[SV.select(x, v("x")), { x: [1n, 2n] }], ValueError, "an array's index must be known, got 1'bx"],
    [[SV.select(x, 0n), { x: new Map([["a", 1n]]) }], TypeError, "a dict cannot be selected from"],
    [[SV.range_(x, 1n, 0n), { x: new Map() }], TypeError, "a dict cannot be selected from"],
    [[SV.range_(v("1010"), 0n, 1n), {}], ValueError, "a range's msb must be at least its lsb, got [0:1]"],
    [[SV.range_(v("1010"), v("x"), 0n), {}], ValueError, "a range's msb must be a known integer, got 1'bx"],
    [[SV.range_(v("1010"), 1n, 0.5), {}], ValueError, "a range's lsb must be a known integer, got 0.5"],
    [[SV.replication(0n, v("1")), {}], ValueError, "a replication's count must be positive, got 0"],
    [[SV.concatenation(v("1"), 1.5), {}], TypeError, "a concatenation's parts must be integral, got float"],
    [[call("$clog2", 1n, 2n), {}], TypeError, "$clog2 takes 1 argument, got 2"],
    [[call("$signed", 1.5), {}], TypeError, "$signed expects an integral argument, got real"],
    [[call("$bits", x), { x: new Map() }], TypeError, "$bits expects a number, got dict"],
    [[call("$countones", 1.5), {}], TypeError, "$countones expects an integral argument, got real"],
  ];
  for (const [[expression, variables], error, message] of errors) raises(error, () => ev(expression, variables), message);
  // The complete dialect's literals: an unsized based integer is unsigned (unless written signed) of 32 bits, an unsized
  // vector extended to 32 bits by its leftmost bit where it is x or z, a fill fills its context's width; parentheses
  // written in the term pass their context through.
  const hexadecimal = (value: bigint, signed = false) => (w: any) => w.literal((r: any) => r.value((p: any) => p.as_scalar(value)).base("h").signed(signed));
  const unsized = (bits: string) => (w: any) => w.constant((r: any) => r.bits(bits).base("b")); // an unsized vector, 'bx01
  const fill = (bit: string) => (w: any) => w.constant((r: any) => r.fill(bit)); // '0, '1, 'x, 'z
  const parenthesized = (inner: unknown) => (w: any) => w.parenthesizedexpression((r: any) => r.expression(inner));
  assert(String(ev(hexadecimal(255n))) === "32'b" + "0".repeat(24) + "1".repeat(8) && String(ev(hexadecimal(2n ** 40n))) === "41'b1" + "0".repeat(40));
  assert(String(ev(hexadecimal(5n, true))) === "32'sb" + "0".repeat(29) + "101" && String(ev(b(">", hexadecimal(1n), -1n))) === "1'b0"); // unsigned
  assert(String(ev(unsized("x01"))) === "32'b" + "x".repeat(30) + "01" && String(ev(unsized("101"))) === "32'b" + "0".repeat(29) + "101");
  assert(String(ev(b("&", fill("1"), v("1010")))) === "4'b1010" && String(ev(fill("z"))) === "1'bz");
  assert(String(ev(b("+", fill("1"), v("00000001")))) === "8'b00000000" && ev(b("+", fill("1"), 0.5)) === 1.5); // a real context
  assert(String(ev(b("*", parenthesized(b("+", v("0001"), v("0001"))), v("0011")))) === "4'b0110");
  assert(String(ev(c("bit", 3n))) === "1'b1" && ev(c("real", 3n)) === 3.0); // a cast to a type its keyword names
  assert(String(ev(call("$bits", (t: any) => t.integeratomtype((a: any) => a.keyword("longint"))))) === "32'sb" + "0".repeat(25) + "1000000");
  assert(String(ev(c(b("+", 2n, 2n), v("11111111")))) === "4'b1111"); // a width, of any known value
  assert(String(ev(SV.method(x, "size"), { x: [1n, 2n] })) === "32'sb" + "0".repeat(30) + "10"); // an array's method, a call of its member
  // Every other kind is refused by name, and so is what an evaluated kind holds beyond what it evaluated before.
  const scoped = (w: any) => w.nameexpression((r: any) => r.name((n: any) => n.scopedname((s: any) => s.scope( // p::x
    (i: any) => i.identifier((d: any) => d.spelling("p"))).name((i: any) => i.identifier((d: any) => d.spelling("x"))))));
  const refusals: [unknown, string][] = [
    [(w: any) => w.incrementexpression((r: any) => r.operator("++").operand(x)), "SystemVerilog incrementexpression is not evaluated"],
    [(w: any) => w.assignmentpattern((r: any) => r.items([SV.literal(1n)])), "SystemVerilog assignmentpattern is not evaluated"],
    [(w: any) => w.macrousage((r: any) => r.name((i: any) => i.spelling("W"))), "SystemVerilog Preprocessor macrousage is not evaluated"],
    [(w: any) => w.valuerange((r: any) => r.left(SV.literal(1n)).right(SV.literal(2n))), "SystemVerilog valuerange is not evaluated"],
    [(w: any) => w.constant((r: any) => r.singleton("null")), "SystemVerilog constant null is not evaluated"],
    [(w: any) => w.literal((r: any) => r.value((p: any) => p.as_scalar(10.0)).unit("ns")), "SystemVerilog time literal is not evaluated"],
    [scoped, "SystemVerilog nameexpression of a name that is not an identifier is not evaluated"],
    [(w: any) => w.rangeselect((r: any) => r.value(v("1010")).left(SV.literal(0n)).operator("+:").right(SV.literal(2n))), "SystemVerilog rangeselect '+:' is not evaluated"],
    [SV.inside(1n, (w: any) => w.valuerange((r: any) => r.left(SV.literal(1n)).operator("+/-").right(SV.literal(2n)))), "SystemVerilog valuerange '+/-' is not evaluated"],
    [call("$display", 1n), "SystemVerilog systemcall '$display' is not evaluated"],
    [(w: any) => w.callexpression((r: any) => r.callee(scoped)), "SystemVerilog callexpression of a nameexpression is not evaluated"],
    [SV.method(x, "sum", 1n), "SystemVerilog callexpression of a memberexpression is not evaluated"],
    [(w: any) => w.arraymethodwithexpression((r: any) => r.value(SV.method(x, "unique")).body(SV.literal(1n))),
      "SystemVerilog arraymethodwithexpression of a call that is not a reduction's is not evaluated"],
    [(w: any) => w.arraymethodwithexpression((r: any) => r.value(x).body(SV.literal(1n))),
      "SystemVerilog arraymethodwithexpression of a call that is not a reduction's is not evaluated"],
    [SV.cast((t: any) => t.structtype((s: any) => s.keyword("struct")), 1n), "SystemVerilog Types structtype is not evaluated"],
    [c("realtime", 1n), "SystemVerilog Types nonintegertype is not evaluated"],
    [SV.cast((t: any) => t.integervectortype((n: any) => n.keyword("logic").dimensions([
      (d: any) => d.rangedimension((r: any) => r.left(SV.literal(7n)).right(SV.literal(0n)))])), 1n), "SystemVerilog Types integervectortype is not evaluated"],
  ];
  for (const [expression, message] of refusals) raises(NotImplementedError, () => ev(expression, { x: [1n, 2n] }), message);
  raises(ValueError, () => ev(c(b("-", 1n, 1n), 1n)), "a cast's width must be positive, got 0"); // a width known only as it is evaluated
}

<a id="DIA-15"></a>
## DIA-15 · SystemVerilog arrays: methods, reductions with a `with` clause, and `inside` over an array's elements

In [ ]:
{
  const [xs, p, q] = [SV.identifier("xs"), SV.identifier("p"), SV.identifier("q")];
  assert(SVText.ToText(sv(SV.iterate(SV.member(xs, "pins"), "and", "p", SV.binary(">", p, 0n)))) === "xs.pins.and(p) with (p > 0)");
  assert(SVText.ToText(sv(SV.select(SV.method(xs, "min"), 0n))) === "xs.min()[0]" && SVText.ToText(sv(SV.method(SV.binary("+", xs, 1n), "size"))) === "(xs + 1).size()");
  assert(SVText.ToText(sv(SV.inside(1n, xs, SV.span(2n, 3n)))) === "1 inside {xs, [2:3]}");
  const ev = (expression: unknown, variables: Record<string, unknown> = {}) => SVV.OfAny(sv(expression), variables) as any;
  const atom = (keyword: string) => (t: any) => t.integeratomtype((a: any) => a.keyword(keyword));
  assert(SV.DIALECT.validate(sv(SV.method(xs, "pop")), ["xs"]).length === 0 && SV.DIALECT.validate(sv(SV.iterate(xs, "min", "p", p)), ["xs"]).length === 0); // any method's
  raises(NotImplementedError, () => ev(SV.method(xs, "pop"), { xs: [1n] }), "SystemVerilog callexpression of a memberexpression is not evaluated"); // but these
  raises(NotImplementedError, () => ev(SV.iterate(xs, "min", "p", p), { xs: [1n] }),
    "SystemVerilog arraymethodwithexpression of a call that is not a reduction's is not evaluated");
  assert(SV.DIALECT.validate(sv(SV.iterate(xs, "and", "p", p)), ["xs"]).length === 0
    && same([...SY.free(sv(SV.iterate(xs, "and", "p", SV.binary("&", p, q))), SV.DIALECT)].sort(), ["q", "xs"]));
  assert(SV.DIALECT.infer(sv(SV.iterate(xs, "and", "p", p)), { xs: D.Anything }) === D.Anything);
  assert(SV.DIALECT.infer(sv(SV.method(xs, "size")), { xs: D.Anything }) === D.Anything);
  const data = [1n, 2n, 3n, 2n];
  const cases: [unknown, string][] = [ // each over [1, 2, 3, 2], and its value
    [SV.method(xs, "size"), "32'sb" + "0".repeat(29) + "100"], [SV.method(xs, "sum"), "32'sb" + "0".repeat(28) + "1000"],
    [SV.method(xs, "product"), "32'sb" + "0".repeat(28) + "1100"], [SV.method(xs, "and"), "32'sb" + "0".repeat(32)],
    [SV.method(xs, "or"), "32'sb" + "0".repeat(30) + "11"], [SV.method(xs, "xor"), "32'sb" + "0".repeat(30) + "10"],
    [SV.select(SV.method(xs, "min"), 0n), "32'sb" + "0".repeat(31) + "1"], [SV.select(SV.method(xs, "max"), 0n), "32'sb" + "0".repeat(30) + "11"],
    [SV.method(SV.method(xs, "unique"), "size"), "32'sb" + "0".repeat(30) + "11"],
    [SV.iterate(xs, "and", "p", SV.binary(">", p, 0n)), "1'b1"], [SV.iterate(xs, "and", "p", SV.binary(">", p, 1n)), "1'b0"],
    [SV.iterate(xs, "or", "p", SV.binary(">", p, 2n)), "1'b1"], [SV.iterate(xs, "xor", "p", SV.binary("==", p, 2n)), "1'b0"],
    [SV.iterate(xs, "sum", "p", SV.cast(atom("int"), SV.binary(">=", p, 2n))), "32'sb" + "0".repeat(30) + "11"],
    [SV.iterate(xs, "sum", "p", SV.binary("*", p, p)), "32'sb" + "0".repeat(27) + "10010"],
    [SV.iterate(xs, "product", "p", SV.binary("+", p, 1n)), "32'sb" + "0".repeat(25) + "1001000"],
    [SV.inside(3n, xs), "1'b1"], [SV.inside(5n, xs, 7n), "1'b0"], [SV.inside(7n, xs, 7n), "1'b1"], [SV.inside(2n, SV.span(0n, 1n), xs), "1'b1"],
  ];
  for (const [expression, value] of cases) assert(String(ev(expression, { xs: data })) === value, SVText.ToText(sv(expression)));
  // x and z reduce as the operators take them: and() is 0 if any value is, x if any is unknown and none 0.
  assert(String(ev(SV.iterate(xs, "and", "p", p), { xs: [SVD.Logic.of("1"), SVD.Logic.of("x")] })) === "1'bx");
  assert(String(ev(SV.iterate(xs, "and", "p", p), { xs: [SVD.Logic.of("0"), SVD.Logic.of("x")] })) === "1'b0");
  assert(String(ev(SV.inside(SV.vector("1x"), xs), { xs: [SVD.Logic.of("10")] })) === "1'bx");
  assert(String(ev(SV.method(SV.method(xs, "unique"), "size"), { xs: [SVD.Logic.of("1x"), SVD.Logic.of("1x"), 1.0, 1n, 1.0] })) === "32'sb" + "0".repeat(30) + "11");
  // Of no items, the reductions give their identities, and the locators no item.
  assert(same(["sum", "product", "and", "or", "xor"].map((m) => String(ev(SV.method(xs, m), { xs: [] }))), [
    "32'sb" + "0".repeat(32), "32'sb" + "0".repeat(31) + "1", "1'b1", "1'b0", "1'b0"]));
  assert(ev(SV.method(xs, "min"), { xs: [] }).length === 0 && String(ev(SV.iterate(xs, "and", "p", SV.binary(">", p, 0n)), { xs: [] })) === "1'b1");
  assert(ev(SV.method(xs, "sum"), { xs: [1.5, 2n] }) === 3.5 && String(ev(SV.select(SV.method(xs, "max"), 0n), { xs: [1.5, 2n] })) === "32'sb" + "0".repeat(30) + "10");
  const errors: [unknown, Record<string, unknown>, Function, string][] = [
    [SV.method(xs, "size"), { xs: 5n }, TypeError, "size() is a method of arrays, not of Logic"],
    [SV.iterate(xs, "and", "p", p), { xs: new Map([["a", 1n]]) }, TypeError, "and() is a method of arrays, not of dict"],
    [SV.method(xs, "sum"), { xs: [new Map([["a", 1n]])] }, TypeError, "an array's elements must be numbers, got dict"],
    [SV.inside(1n, xs), { xs: [new Map([["a", 1n]])] }, TypeError, "an array's elements must be numbers, got dict"],
    [SV.inside(1n, xs), { xs: new Map([["a", 1n]]) }, TypeError, "inside expects a number, got dict"],
    [SV.method(xs, "and"), { xs: [1.5, 2.5] }, TypeError, "& expects integral operands, got real"],
  ];
  for (const [expression, variables, error, message] of errors) raises(error, () => ev(expression, variables), message);
}

<a id="DIA-16"></a>
## DIA-16 · MATLAB arrays: arrayfun, indexing from 1, and the functions of arrays

In [ ]:
{
  const [xs, p] = [M.identifier("xs"), M.identifier("p")];
  assert(MText.ToText(mat(M.call("all", M.arrayfun("p", M.field(xs, "pins"), M.binary(">", p, 0n))))) === "all(arrayfun(@(p) p > 0, xs.pins))");
  assert(MText.ToText(mat(M.index(xs, M.binary("+", p, 1n)))) === "xs(p + 1)" && MText.ToText(mat(M.index(M.field(xs, "a"), 2n))) === "xs.a(2)");
  assert(same(M.DIALECT.validate(mat(M.arrayfun("_p", xs, 1n)), ["xs"]), ["an arrayfun's parameter must be an identifier, got '_p'"]));
  assert(M.DIALECT.validate(mat(M.arrayfun("p", xs, p)), ["xs"]).length === 0
    && same([...SY.free(mat(M.arrayfun("p", xs, M.binary("+", p, M.identifier("q")))), M.DIALECT)].sort(), ["q", "xs"]));
  assert(same(M.DIALECT.validate(mat(M.call("numel", xs, 1n)), ["xs"]), ["numel takes 1 arguments, got 2"]));
  assert(M.DIALECT.infer(mat(M.call("all", xs)), { xs: D.Anything }).name() === "logical");
  assert(M.DIALECT.infer(mat(M.arrayfun("p", xs, p)), { xs: D.Anything }) === D.Anything);
  assert(MText.ToText(mat(M.array(1n, xs, M.array()))) === "[1, xs, []]" && M.DIALECT.validate(mat(M.array(1n, xs)), ["xs"]).length === 0);
  assert(equal(MV.OfAny(mat(M.array(1n, xs)), { xs: [true, 2n] }), [1, 1, 2]) && equal(MV.OfAny(mat(M.array(true, false))), [true, false]));
  assert(equal(MV.OfAny(mat(M.array("a", xs)), { xs: ["b"] }), ["a", "b"])); // a string array
  raises(TypeError, () => MV.OfAny(mat(M.array(1n, "a"))), "Concatenation takes numbers and logicals, or strings.");
  const alike = (a: any, b: any): boolean => (Array.isArray(b)
    ? Array.isArray(a) && a.length === b.length && a.every((x, i) => alike(x, b[i]))
    : typeof a === typeof b && (a === b || (Number.isNaN(a) && Number.isNaN(b))));
  const VALUES: [any, unknown, unknown][] = [
    [M.call("all", M.arrayfun("p", xs, M.binary(">", p, 0n))), [1n, 2n, 3n, 2n], true],
    [M.call("any", M.arrayfun("p", xs, M.binary(">", p, 2n))), [1n, 2n, 3n, 2n], true],
    [M.call("nnz", M.arrayfun("p", xs, M.binary(">=", p, 2n))), [1n, 2n, 3n, 2n], 3.0], [M.call("numel", xs), [1n, 2n, 3n, 2n], 4.0],
    [M.index(xs, 2n), [1n, 2n, 3n, 2n], 2.0], [M.index(xs, true), [1n, 2n, 3n, 2n], 1.0], [M.index(xs, false), [1n, 2n, 3n, 2n], []],
    [M.call("ismember", 3n, xs), [1n, 2n, 3n, 2n], true], [M.call("ismember", "3", xs), [1n, 2n, 3n, 2n], true],
    [M.call("ismember", 5n, xs), [1n, 2n, 3n, 2n], false], [M.call("sum", xs), [1n, 2n, 3n, 2n], 8.0],
    [M.call("min", xs), [1n, 2n, 3n, 2n], 1.0], [M.call("max", xs), [1n, 2n, 3n, 2n], 3.0],
    [M.call("numel", M.call("unique", xs)), [1n, 2n, 3n, 2n], 3.0],
    [M.arrayfun("p", xs, M.binary(".*", p, 2n)), [1n, 2n, 3n, 2n], [2.0, 4.0, 6.0, 4.0]],
    [M.arrayfun("p", xs, M.binary(">", p, 1n)), [1n, 2n, 3n, 2n], [false, true, true, true]],
    [M.arrayfun("p", xs, p), [true, 2n], [1.0, 2.0]], [M.call("all", xs), [], true], [M.call("any", xs), [], false],
    [M.call("sum", xs), [], 0.0], [M.call("min", xs), [], []], [M.call("max", xs), [NaN, 2n], 2.0], [M.call("any", xs), [NaN], false],
    [M.call("all", xs), [NaN], true], [M.call("nnz", xs), [NaN, 0n], 1.0], [M.call("min", xs), [true, false], false],
    [M.call("sum", xs), [true, 2n], 3.0], [M.call("unique", xs), ["b", "a", "b"], ["a", "b"]], [M.call("unique", xs), [true, true], [true]],
    [M.call("unique", xs), [3n, 1n, 3n], [1.0, 3.0]], [M.call("numel", xs), 5n, 1.0], [M.index(xs, 1n), 5n, 5.0],
    [M.arrayfun("p", xs, M.binary("+", p, 1n)), 5n, [6.0]], [M.call("ismember", 1n, xs), [true], true], [M.call("numel", xs), "abc", 1.0],
    [M.call("numel", xs), new Map([["a", 1n]]), 1.0],
  ];
  const ERRORS: [any, unknown, Function, string][] = [
    [M.index(xs, 0n), [1n, 2n], ValueError, "Array indices must be positive integers or logical values."],
    [M.index(xs, 1.5), [1n, 2n], ValueError, "Array indices must be positive integers or logical values."],
    [M.index(xs, "a"), [1n, 2n], ValueError, "Array indices must be positive integers or logical values."],
    [M.index(xs, 5n), [1n, 2n, 3n, 2n], ValueError, "Index exceeds the number of array elements. Index must not exceed 4."],
    [M.arrayfun("p", xs, "a"), [1n, 2n], ValueError, "Non-scalar in Uniform output, at index 1, output 1. Set 'UniformOutput' to false."],
    [M.arrayfun("p", xs, M.binary("+", xs, 0n)), [1n, 2n], TypeError, "Operator '+' is not supported for operands of type 'array' and 'double'."],
    [M.arrayfun("p", xs, xs), [1n, 2n], ValueError, "Non-scalar in Uniform output, at index 1, output 1. Set 'UniformOutput' to false."],
    [M.call("sum", xs), ["a"], TypeError, "Invalid data type. First argument must be numeric or logical."],
    [M.call("all", xs), new Map([["a", 1n]]), TypeError, "Invalid data type. First argument must be numeric or logical."],
    [M.call("unique", xs), [new Map([["a", 1n]])], TypeError, "Invalid data type. First argument must be numeric or logical."],
    [M.call("ismember", xs, xs), [1n], TypeError, "ismember takes a scalar and an array of scalars."],
    [M.binary("==", xs, 1n), [1n], TypeError, "Operator '==' is not supported for operands of type 'array' and 'double'."],
    [M.binary("<", xs, 1n), new Map([["a", 1n]]), TypeError, "Operator '<' is not supported for operands of type 'struct' and 'double'."],
  ];
  for (const [expression, value, expected] of VALUES) {
    const result = MV.OfAny(mat(expression), { xs: value });
    assert(alike(result, expected), `${MText.ToText(mat(expression as any))}: ${String(result)}`);
  }
  for (const [expression, value, error, message] of ERRORS) raises(error, () => MV.OfAny(mat(expression), { xs: value }), message);
  assert(Number.isNaN(MV.OfAny(mat(M.call("max", xs)), { xs: [NaN] })) && alike(MV.OfAny(mat(M.call("unique", xs)), { xs: [NaN, 1n, NaN] }), [1.0, NaN, NaN]));
  assert(alike(MV.OfAny(mat(M.field(xs, "ranks")), { xs: squad }), [1.0, 2.0, 3.0])); // a list property is an array of doubles
  assert(MV.OfAny(mat(M.call("sum", M.field(xs, "ranks"))), { xs: squad }) === 6.0);
  raises(ValueError, () => MV.OfAny(mat(M.arrayfun("p", null, 1n))), "an arrayfun needs a value");
}

<a id="DIA-17"></a>
## DIA-17 · Excel arrays: MAP and LAMBDA, AND and OR of any number of arguments, and the functions of arrays

In [ ]:
{
  const [xs, p] = [X.name("xs"), X.name("p")];
  assert(XText.ToText(xl(X.function("AND", X.map_("p", X.field(xs, "pins"), X.infix(">", p, 0n))))) === "=AND(MAP(xs.pins, LAMBDA(p, p > 0)))");
  assert(XText.ToText(xl(X.function("INDEX", xs, X.infix("+", p, 1n)))) === "=INDEX(xs, p + 1)");
  assert(X.DIALECT.validate(xl(X.map_("p", xs, p)), ["xs"]).length === 0
    && same([...SY.free(xl(X.map_("p", xs, X.infix("+", p, X.name("q")))), X.DIALECT)].sort(), ["q", "xs"]));
  assert(X.DIALECT.validate(xl(X.function("AND", 1n, 2n, 3n))).length === 0 && same(X.DIALECT.validate(xl(X.function("ROWS", xs, 1n)), ["xs"]), ["ROWS takes 1 arguments, got 2"]));
  assert(X.DIALECT.infer(xl(X.function("AND", xs)), { xs: D.Anything }).name() === "logical");
  assert(X.DIALECT.infer(xl(X.map_("p", xs, p)), { xs: D.Anything }) === D.Anything);
  assert(XV.OfAny(xl(X.function("AND", true, true, false))) === false && XV.OfAny(xl(X.function("OR", false, false, true))) === true);
  // IF takes 2 or 3 arguments, and AND, OR, SUM, MIN and MAX 1 to 255, as Excel's do.
  assert(X.DIALECT.validate(xl(X.function("IF", true, 1n))).length === 0 && XV.OfAny(xl(X.function("IF", false, 1n))) === false && XV.OfAny(xl(X.function("IF", true, 1n))) === 1.0);
  assert(same(X.DIALECT.validate(xl(X.function("IF", true))), ["IF takes 2 or 3 arguments, got 1"]) && same(X.DIALECT.validate(xl(X.function("SUM"))), ["SUM takes 1 to 255 arguments, got 0"]));
  assert(XV.OfAny(xl(X.function("SUM", 1n, xs, true)), { xs: [2n, "a", true] }) === 4.0 && XV.OfAny(xl(X.function("MAX", xs, 7n)), { xs: [1n, 2n] }) === 7.0);
  assert(XV.OfAny(xl(X.function("MIN", 3n, X.name("e"))), { e: XD.Error.of("#N/A") }) === XD.Error.of("#N/A"));
  const arities: [unknown, string][] = [[X.function("IF", true, 1n, 2n, 3n), "IF takes 2 or 3 arguments, got 4"],
    [X.function("AND"), "AND takes 1 to 255 arguments, got 0"], [X.function("MIN"), "MIN takes 1 to 255 arguments, got 0"]];
  for (const [expression, message] of arities) raises(TypeError, () => XV.OfAny(xl(expression)), message);
  // VSTACK stacks its arguments into one array, an array's elements in turn, errors among them.
  assert(XText.ToText(xl(X.function("VSTACK", 1n, xs))) === "=VSTACK(1, xs)"
    && same(X.DIALECT.validate(xl(X.function("VSTACK")), ["xs"]), ["VSTACK takes 1 to 254 arguments, got 0"]));
  const stacked = XV.OfAny(xl(X.function("VSTACK", 1n, xs, X.name("e"))), { xs: [2n, "a"], e: XD.Error.of("#N/A") }) as unknown[];
  assert(equal(stacked.slice(0, 3), [1, 2, "a"]) && stacked[3] === XD.Error.of("#N/A"));
  raises(TypeError, () => XV.OfAny(xl(X.function("VSTACK"))), "VSTACK takes 1 to 254 arguments, got 0");
  const VALUES: [any, unknown, unknown][] = [
    [X.function("AND", X.map_("p", xs, X.infix(">", p, 0n))), [1n, 2n, 3n, 2n], true],
    [X.function("OR", X.map_("p", xs, X.infix(">", p, 2n))), [1n, 2n, 3n, 2n], true],
    [X.function("SUM", X.map_("p", xs, X.function("IF", X.infix(">=", p, 2n), 1n, 0n))), [1n, 2n, 3n, 2n], 3.0],
    [X.function("ROWS", xs), [1n, 2n, 3n, 2n], 4.0], [X.function("INDEX", xs, 2n), [1n, 2n, 3n, 2n], 2.0],
    [X.function("INDEX", xs, 2.9), [1n, 2n, 3n, 2n], 2.0], [X.function("INDEX", xs, 5n), [1n, 2n], XD.Error.of("#REF!")],
    [X.function("INDEX", xs, 0n), [1n, 2n], XD.Error.of("#REF!")], [X.function("INDEX", xs, "x"), [1n, 2n], XD.Error.of("#VALUE!")],
    [X.function("MATCH", 3n, xs, 0n), [1n, 2n, 3n, 2n], 3.0], [X.function("MATCH", "B", xs, 0n), ["a", "b"], 2.0],
    [X.function("MATCH", 5n, xs, 0n), [1n, 2n], XD.Error.of("#N/A")], [X.function("MATCH", "1", xs, 0n), [1n, 2n], XD.Error.of("#N/A")],
    [X.function("MATCH", 1n, xs, 1n), [1n, 2n], XD.Error.of("#VALUE!")], [X.function("MATCH", true, xs, 0n), [1n, true], 2.0],
    [X.function("ISNUMBER", X.function("MATCH", 2n, xs, 0n)), [1n, 2n], true],
    [X.function("ISNUMBER", X.function("MATCH", 7n, xs, 0n)), [1n, 2n], false], [X.function("ISNUMBER", "1"), [], false],
    [X.function("SUM", xs), [1n, 2n, 3n, 2n], 8.0], [X.function("SUM", xs), [1n, "a", true, 2n], 3.0],
    [X.function("SUM", xs), [1n, XD.Error.of("#N/A")], XD.Error.of("#N/A")], [X.function("SUM", true), [], 1.0],
    [X.function("SUM", "2"), [], 2.0], [X.function("SUM", "a"), [], XD.Error.of("#VALUE!")], [X.function("MIN", xs), [3n, 1n, 2n], 1.0],
    [X.function("MAX", xs), [3n, 1n, 2n], 3.0], [X.function("MAX", xs), ["a"], 0.0], [X.function("MIN", xs), [], 0.0],
    [X.function("ROWS", X.function("UNIQUE", xs)), [1n, 2n, 3n, 2n], 3.0],
    [X.function("ROWS", X.function("UNIQUE", xs)), ["a", "A", 1n, true, "1"], 4.0],
    [X.function("ROWS", X.function("UNIQUE", xs)), [XD.Error.of("#N/A"), XD.Error.of("#N/A"), new Map([["a", 1n]]), new Map([["a", 1n]])], 3.0],
    [X.function("ROWS", xs), 5n, 1.0], [X.function("INDEX", xs, 1n), 5n, 5.0],
    [X.function("ROWS", X.map_("p", xs, X.infix("*", p, 2n))), 5n, 1.0], [X.function("AND", xs), [true, "a", 1n], true],
    [X.function("AND", xs), ["a"], XD.Error.of("#VALUE!")], [X.function("OR", xs), [], XD.Error.of("#VALUE!")],
    [X.function("AND", xs, false), [true], false], [X.function("OR", xs), [0n, XD.Error.of("#DIV/0!")], XD.Error.of("#DIV/0!")],
    [X.function("INDEX", X.map_("p", xs, X.infix("-", p, "a")), 1n), [1n], XD.Error.of("#VALUE!")],
    [X.function("ROWS", X.map_("p", xs, p)), XD.Error.of("#REF!"), XD.Error.of("#REF!")], [X.infix("+", xs, 1n), [1n], XD.Error.of("#VALUE!")],
    [X.infix("=", xs, 1n), [1n], XD.Error.of("#VALUE!")],
    [X.function("MATCH", 1n, xs, "a"), [1n], XD.Error.of("#VALUE!")], [X.function("UNIQUE", xs), XD.Error.of("#REF!"), XD.Error.of("#REF!")],
  ];
  assert(XV.OfAny(xl(X.function("INDEX", xs, X.name("n"))), { xs: [1n], n: Infinity }) === XD.Error.of("#NUM!"));
  for (const [expression, value, expected] of VALUES) {
    const result = XV.OfAny(xl(expression), { xs: value });
    assert(result === expected && typeof result === typeof expected, `${XText.ToText(xl(expression as any))}: ${String(result)}`);
  }
  assert(same(XV.OfAny(xl(X.field(xs, "ranks")), { xs: squad }) as unknown[], [1.0, 2.0, 3.0])); // a list property is an array of numbers
  assert(XV.OfAny(xl(X.function("SUM", X.field(xs, "ranks"))), { xs: squad }) === 6.0);
  assert(same(XV.OfAny(xl(X.map_("p", xs, X.infix(">", p, 1n))), { xs: [1n, 2n] }) as unknown[], [false, true]));
}

<a id="DIA-18"></a>
## DIA-18 · A literal is a domain representation: printers write its spelling where it has one, else a default spelling; its value is what evaluates

In [ ]:
{
  /** A literal of `dialect` holding `value`, spelled `text`, as a parser would write it. */
  const spelled = (dialect: F.Dialect, value: unknown, text: string) => term((u: any) => u.literal(
    (r: any) => (value !== null ? r.value((v: any) => v.as_scalar(value)) : r).text(text)), dialect);
  for (const [dialect, printer, text, value, written] of [
    [P.DIALECT, PText.ToText, "0x10", 16n, "16"],
    [M.DIALECT, MText.ToText, "1e1", 10.0, "10.0"],
    [X.DIALECT, XText.ToText, "1E+1", 10.0, "=10.0"],
    [Latex.Expressions.DIALECT, LText.ToText, "10.00", 10.0, "10.0"],
    [SystemVerilog.Expressions.DIALECT, SVText.ToText, "'d16", 16n, "16"],
    [TS.DIALECT, TSText.ToText, "0x10", 16.0, "16"],
    [C.DIALECT, CText.ToText, "0x10", 16n, "16"],
    [CppDialect.Expressions.DIALECT, CppDialect.Text.ToText, "0x10", 16n, "16"],
  ] as [F.Dialect, (expression: unknown) => string, string, unknown, string][]) {
    const literal = spelled(dialect, value, text);
    assert(printer(literal) === (dialect === X.DIALECT ? "=" + text : text) && dialect.validate(literal).length === 0);
    assert(printer(term(value, dialect)) === written); // no spelling: the value's default one
  }
  assert(PText.ToText(term(P.binop("**", spelled(P.DIALECT, -2n, "-2"), 2n), P.DIALECT)) === "(-2) ** 2"); // a spelling's precedence
  assert(PText.ToText(term(P.unaryop("-", spelled(P.DIALECT, -1n, "-1")), P.DIALECT)) === "--1"); // as its value's would be
  assert(MText.ToText(term(M.unary("-", spelled(M.DIALECT, -1n, "-1")), M.DIALECT)) === "--1");
  const LX = Latex.Expressions;
  assert(LText.ToText(term(LX.binary("+", 1n, spelled(LX.DIALECT, -1n, "-1")), LX.DIALECT)) === "1 + -1");
  assert(PV.OfAny(spelled(P.DIALECT, 16n, "0x10")) === 16n && MV.OfAny(spelled(M.DIALECT, 10.0, "1e1")) === 10.0); // its value
  const invalid = spelled(P.DIALECT, null, "08");
  assert(PText.ToText(invalid) === "08" && same(P.DIALECT.validate(invalid), ["a literal needs a value"])); // kept as written
}

<a id="DIA-19"></a>
## DIA-19 · Python3's kinds beyond the narrower ones: literals keeping what their text encodes, f-strings as written, each kind's own checks and what it binds, stored as any term

In [ ]:
{
  const checked = (spec: unknown, bound: string[] = []): string[] => P.DIALECT.validate(py(spec), bound); // its problems
  const shown = (spec: unknown): string => PText.ToText(py(spec)); // its source
  const [x, y] = [P.name("x"), P.name("y")];
  // A Python literal keeps what its text encodes beside its text, its problems and its value: a string's prefix and
  // quotes, an int's base, digit separators; Python3's own constant holds None, ... and complex numbers, a complex number
  // its `complex`, a value of `Expressions.Python.Python3.Complex`, whose `real` and `imag` are IEEE 754's binary64s.
  // (Read from source by the Python implementation's `FromText`; written here through the builders.)
  const written = F.form(py(P.constant(new P.Complex(0, 10), "1_0j"))).attributes; // written with its spelling
  const complex = written.get("complex") as F.Part;
  assert(written.get("text") === "1_0j" && complex.kind === "complex" && complex.size === 0
    && equal(complex.attributes, new Map<string, unknown>([["real", 0], ["imag", 10]])));
  assert(["real", "imag"].every((part) => store.registered("Expressions.Python.Python3.Complex").properties.get(part).schema
    === store.registered("Expressions.IEEE754.Expressions.binary64")));
  assert(shown(P.constant(16n, "0x10")) === "0x10" && equal(F.form(py(P.constant(null, "None"))).attributes,
    new Map([["singleton", "None"], ["text", "None"]]))); // written with its spelling
  assert(shown(P.joinedstr(["a", P.formattedvalue(x, null, null, true), P.formattedvalue(P.set_(1n))])) === "f'a{x=}{ {1}}'"); // unspelled
  assert(shown(P.joinedstr([P.formattedvalue(x, null, null, false, "x = ")])) === "f'{x = }'"); // a field's expression as written
  assert(shown(P.templatestr([P.interpolation(x, "s", ["0", P.formattedvalue(y)])], "t", "\"")) === "t\"{x!s:0{y}}\""); // a format spec's fields are formatted values, as `ast` has them
  for (const [spec, problems] of [ // each new kind's own checks
    [(u: any) => u[P.CONSTANT]((r: any) => r), ["a constant needs a singleton or a complex number"]],
    [(u: any) => u[P.CONSTANT]((r: any) => r.singleton("None").complex((c: any) => c.imag(1.0))), ["a constant holds a singleton or a complex number, not both"]],
    [(u: any) => u[P.CONSTANT]((r: any) => r.singleton("Ellipsis")), ["a constant's singleton must be None or ..., got 'Ellipsis'"]],
    [(u: any) => u[P.CONSTANT]((r: any) => r.complex((c: any) => c.real(1.0))), ["a complex constant needs an imag"]],
    [(u: any) => u[P.CONSTANT]((r: any) => r.text("1__0j").problems(["invalid decimal literal"])), ["invalid decimal literal"]],
    [(u: any) => u[P.LITERAL]((r: any) => r.value((v: any) => v.as_scalar("x")).prefix("q").quote("`").base(3n)),
      ["a literal's prefix cannot be 'q'", "a literal's quote cannot be '`'", "a literal's base cannot be 3"]],
    [P.joinedstr(["x"], "b"), ["a joinedstr's prefix cannot be 'b'"]],
    [P.templatestr(["x"], "Rt", "`"), ["a templatestr's quote cannot be '`'"]],
    [P.joinedstr([P.formattedvalue(1n, "q", ["0", P.formattedvalue(2n, "z")])]),
      ["a formattedvalue's conversion cannot be 'q'", "a formattedvalue's conversion cannot be 'z'"]], // its spec's fields too
    [P.concatenatedstring("a"), ["a concatenatedstring needs two strings"]],
    [P.set_(), ["a set needs an element"]],
    [P.boolop("and", 1n), ["a boolop needs two values"]],
    [P.listcomp(1n), ["a listcomp needs a for clause"]],
    [P.call(P.name("f"), P.keyword("a b", 1n)), ["a keyword's arg must be an identifier, got 'a b'"]],
    [(u: any) => u[P.COMPARE]((r: any) => r.left(P.constant(1n)).comparisons([
      (a: any) => a.as_object((o: any) => o.comparator(P.constant(2n)))])), ["a comparison needs an op"]],
    [P.namedexpr(P.attribute(P.name("o"), "a"), 1n), ["a namedexpr's target must be a name"]],
    [P.lambda_(1n, [P.arg("a", null, P.name("int"))]), ["a lambda's parameters have no annotations"]],
    [P.lambda_(1n, ["a b"]), ["a lambda's parameter must be an identifier, got 'a b'"]],
    [(u: any) => u[P.LAMBDA]((r: any) => r.body(P.constant(1n)).args((a: any) => a.args([
      (e: any) => e.as_object((o: any) => o.default_value(P.constant(1n)))]))), ["a lambda's parameter needs an arg"]],
    [(u: any) => u[P.LISTCOMP]((r: any) => r.body(P.constant(1n)).generators([
      (a: any) => a.as_object((o: any) => o.iter(P.name("xs")))])), ["a listcomp needs a generators 0 target"]],
    [P.lambda_(1n), []], [P.yield_(), []], [P.dictcomp(1n, null, P.comprehension("x", P.name("xs"))), []], // optional slots
    [P.lambda_(P.binop("+", P.name("a"), P.name("k")), ["a"], { vararg: "v", kwonlyargs: ["k"], kwarg: "w" }), []],
  ] as [unknown, string[]][]) {
    assert(same(checked(spec, ["f", "xs", "o"]), problems), Repr.repr(checked(spec, ["f", "xs", "o"])));
  }
  // What each new kind binds: a comprehension its targets, a lambda its parameters, a named expression its target alone
  for (const [spec, free] of [
    [P.listcomp(P.name("q"), P.comprehension(P.tuple_(P.name("p"), P.starred(P.name("q"))), P.name("xs"), [P.name("p")])), ["xs"]],
    [P.listcomp(1n, P.comprehension(P.attribute(P.name("o"), "a"), P.name("xs"))), ["o", "xs"]], // an attribute binds nothing
    [P.dictcomp(P.name("k"), P.name("v"), P.comprehension(P.parenthesized(P.list_(P.name("k"), P.name("v"))), P.name("d"))), ["d"]],
    [P.setcomp(P.name("q"), P.comprehension("p", P.name("ps")), P.comprehension("q", P.name("p"))), ["ps"]], // later clauses see earlier
    [P.lambda_(P.binop("+", P.name("a"), P.name("b")), ["a", P.arg("b", P.name("a"))]), ["a"]], // a default: the scope's
    [P.boolop("and", P.namedexpr("y", 1n), P.name("y")), ["y"]], // beyond itself, the caller binds a named expression's target
    [(u: any) => u[P.LISTCOMP]((r: any) => r.body(P.name("e")).generators([
      (a: any) => a.as_object((o: any) => o.iter(P.name("xs")))])), ["e", "xs"]], // no target: nothing bound
  ] as [unknown, string[]][]) {
    assert(same([...SY.free(py(spec), P.DIALECT)].sort(), free), Repr.repr([...SY.free(py(spec), P.DIALECT)]));
  }
  // Printed as held, what the writers would not write: a complex number of no real part, a parameter of no name
  assert(shown((u: any) => u[P.CONSTANT]((r: any) => r.complex((c: any) => c.imag(2.0)))) === "2j");
  assert(shown((u: any) => u[P.LAMBDA]((r: any) => r.body(P.constant(1n)).args((a: any) => a.args([
    (e: any) => e.as_object((o: any) => o.default_value(P.constant(1n)))])))) === "lambda =1: 1");
  const spelled = py((u: any) => u[P.LITERAL]((r: any) => r.value((v: any) => v.as_scalar(255n)).text("0x_FF").base(16n).separators(true)));
  assert(equal(F.form(spelled).attributes, new Map<string, unknown>([["value", 255n], ["text", "0x_FF"], ["base", 16n], ["separators", true]]))
    && PText.ToText(spelled) === "0x_FF");
  // A union of helpers holding none (an f-string's value with no branch written) holds nothing, read or printed
  const empty = py((u: any) => u[P.JOINEDSTR]((r: any) => r.values([(a: any) => a.as_union((v: any) => v)])));
  assert(!F.form(empty).attributes.has("values") && PText.ToText(empty) === "f''");
  // Each new kind is stored as any term, its helpers nested in it: every kind of FRM's sample round-trips; here a few
  for (const spec of [P.lambda_(1n, [P.arg("a", 2n)], { posonlyargs: ["p"] }), P.joinedstr(["a", P.formattedvalue(x, "r", ["0"])]),
    P.call("f", P.keyword(null, x)), P.compare("<", 1n, 2n, ["in", x])]) {
    assert(roundTrips(py(spec)) && F.same(py(P.DIALECT.make(F.form(py(spec)))), py(spec)));
  }
  // Its patterns: a binding target, a comprehension's or a named expression's, holds `Expressions.Python.Python3.Pattern`, the
  // union of the terms a target may be, each a schema of a term too, and invalid syntax ("(ii)"); a form reads a target as
  // a part, its terms after the clause's own
  const Pattern = store.registered("Expressions.Python.Python3.Pattern");
  assert(same(Pattern.branches.map((b: any) => b.name), ["variable", "tuple", "list", "starred", "attribute", "subscript", "parenthesized", "error", "missing"]));
  assert(Pattern.branches.find((b: any) => b.name === "tuple").schema === store.registered("Expressions.Python.Python3.Tuple")); // a term's schema
  assert(["Comprehension", "NamedExpr"].every((name) => store.registered(`Expressions.Python.Python3.${name}`).properties.get("target").schema === Pattern));
  const generator = py(P.generatorexp(P.name("p"), P.comprehension(P.tuple_(P.name("p"), P.starred(P.name("q"))), P.name("ps"))));
  const clause = (F.form(generator).attributes.get("generators") as F.Part[])[0]!;
  const held = clause.attributes.get("target") as F.Part;
  assert(held.kind === "tuple" && held.size === 2 && clause.size === 3);
  assert(same([...P.targetNames(held, F.form(generator).arguments.slice(3))].sort(), ["p", "q"]) && P.targetNames(null).size === 0);
  assert(roundTrips(generator) && F.same(py(P.DIALECT.make(F.form(generator))), generator) && PText.ToText(generator) === "(p for p, *q in ps)");
  raises(Error, () => py(P.generatorexp(1n, P.comprehension(P.binop("+", P.name("a"), 1n), P.name("ps")))),
    "'binop' is not a member of a union value"); // a kind no target is
  for (const [spec, problems] of [ // a target's own checks, as its kind's
    [(u: any) => u[P.NAMEDEXPR]((r: any) => r.value(P.constant(1n))), ["a namedexpr needs a target"]],
    [P.generatorexp(1n, P.comprehension(P.attribute(P.name("o"), "a b"), P.name("ps"))),
      ["generators 0 target: an attribute must be an identifier, got 'a b'"]],
    [(u: any) => u[P.GENERATOREXP]((r: any) => r.body(P.constant(1n)).generators([(a: any) => a.as_object(
      (o: any) => o.target((t: any) => t.variable((v: any) => v)).iter(P.name("ps")))])), ["generators 0 target: a variable needs a name"]],
  ] as [unknown, string[]][]) {
    assert(same(checked(spec, ["ps", "o"]), problems), Repr.repr(checked(spec, ["ps", "o"])));
  }
  assert(shown((u: any) => u[P.LISTCOMP]((r: any) => r.body(P.constant(1n)).generators([
    (a: any) => a.as_object((o: any) => o.iter(P.name("xs")))]))) === "[1 for  in xs]"); // no target, as held
  assert(shown(P.listcomp(1n, P.comprehension(P.starred(P.name("q")), P.name("xs")))) === "[1 for (*q) in xs]"); // by its precedence
}

<a id="DIA-20"></a>
## DIA-20 · Python3 is complete, and its sublanguages' dialects: each node of `ast` an expression is or holds, 3.0 to 3.15, and each type parameter and pattern node, kept as data, is a kind, a helper or an operator of a kind, or is listed as not an expression with its reason

In [ ]:
import { readFileSync as readGrammar } from "node:fs";

{
  interface GrammarNode {
    node: string; class: string; since: string; until?: string; kinds?: string[]; helper?: string; op?: string;
    kind?: string; not?: string;
  }
  const grammar = globalThis.JSON.parse(readGrammar("../../grammars/python3-ast.json", "utf8")) as {
    nodes: GrammarNode[]; concrete: { kind: string }[] }; // `ast`'s nodes, as data
  const [PT, PP] = [PythonTypes.Expressions.DIALECT, PythonPatterns.Expressions.DIALECT];
  const dialectOf = new Map<string, F.Dialect>([["type_param", PT], ["pattern", PP]]); // a sublanguage's nodes, its dialect's kinds; every other node, Python3's
  const kinds = P.DIALECT.kinds();
  const mapped = new Map<F.Dialect, Set<string>>([[P.DIALECT, new Set()], [PT, new Set()], [PP, new Set()]]);
  for (const node of grammar.nodes) { // each a kind, a helper, an operator of a kind, or not an expression, with its reason
    const ways = (["kinds", "helper", "op", "not"] as const).filter((way) => way in node);
    assert(ways.length === 1, node.node);
    if (node.kinds !== undefined) {
      const dialect = dialectOf.get(node.class) ?? P.DIALECT;
      assert(node.kinds.length > 0 && node.kinds.every((kind) => dialect.kinds().has(kind)), node.node);
      node.kinds.forEach((kind) => mapped.get(dialect)!.add(kind));
    } else if (node.helper !== undefined) { // a value object a kind holds, not a term
      const helper = store.registered(`Expressions.Python.Python3.${node.helper}`);
      assert(helper !== null && helper !== undefined && !helper.ref && !kinds.has(node.helper.toLowerCase()), node.node);
    } else if (node.op !== undefined) {
      assert(kinds.get(node.kind as string)!.vocabulary!.has(node.op), node.node);
    } else {
      assert(typeof node.not === "string" && node.not !== "", node.node);
    }
  }
  const concrete = grammar.concrete.map((entry) => entry.kind); // what `ast` does not keep: parentheses
  assert(same([...kinds.keys()].sort(), [...new Set([...mapped.get(P.DIALECT)!, ...concrete, ...P.EXTENSIONS, "error", "missing"])].sort())); // every kind is `ast`'s, but these
  assert([PT, PP].every((dialect) => same([...dialect.kinds().keys()].sort(), [...mapped.get(dialect)!, "error", "missing"].sort()))); // each sublanguage's, exactly
  for (const kind of ["boolop", "binop", "unaryop", "compare"]) { // every operator of the vocabularies is `ast`'s, and the other way
    assert(same([...kinds.get(kind)!.vocabulary!.keys()].sort(),
      grammar.nodes.filter((node) => node.kind === kind).map((node) => node.op as string).sort()), kind);
  }
  // The running Python's `ast` against the data is the Python implementation's alone: `ast` is Python's.
}

<a id="DIA-21"></a>
## DIA-21 · TypeScript5: typescript-estree's expressions in mbse-programs' shapes, its types and patterns each a union of its own; literals keeping what their text encodes; each kind's checks and bindings; evaluation of literals alone

In [ ]:
{
  const ts = (spec: unknown): any => term(spec, TS.DIALECT); // TypeScript5 has no writers: its terms, through the builders
  const tsChecked = (spec: unknown, bound: string[] = []): string[] => TS.DIALECT.validate(ts(spec), bound); // its problems
  // Its union, and its sublanguages, each a dialect of its own, `TypeScript Types` and `TypeScript Patterns`, whose
  // `Term`s were its `Type` and `Pattern` unions, their branches unchanged; a schema is a branch of every union whose
  // positions hold it ("(b) as you recommended"), named alike in each, and TypeScript5's `Term` lists three patterns.
  const unions = new Map([["Term", "TypeScript5"], ["Pattern", "Patterns"], ["Type", "Types"]].map(([name, path]) =>
    [name as string, store.registered(`Expressions.TypeScript.${path}.Term`)]));
  const branches = new Map([...unions].map(([name, union]) =>
    [name, new Map<string, any>(union.branches.map((b: any) => [b.name, b.schema]))]));
  const tags = (name: string): string[] => [...branches.get(name)!.keys()];
  assert(same(tags("Term").filter((tag) => !TSP.TAGS.includes(tag)), [...TS.DIALECT.kinds().keys()]) && tags("Term").length === 46);
  assert(same(tags("Pattern"), ["arraypattern", "objectpattern", "assignmentpattern", "restelement", "identifier",
    "memberexpression", "parenthesizedexpression", "tsasexpression", "tssatisfiesexpression", "tstypeassertion",
    "tsnonnullexpression", "property", "elision", "tsparameterproperty", "error", "missing"]));
  assert(tags("Type").filter((tag) => tag.startsWith("ts")).length === 37); // mbse-programs' 37: typescript-estree's 36, and parentheses
  assert(same(tags("Type").slice(37), ["identifier", "thisexpression", "literal", "unaryexpression", "templateliteral",
    "objectexpression", "error", "missing"])); // the terms a type holds, and invalid syntax
  assert(branches.get("Pattern")!.get("identifier") === branches.get("Term")!.get("identifier")
    && branches.get("Type")!.get("identifier") === branches.get("Term")!.get("identifier"));
  assert(branches.get("Term")!.get("arraypattern") === branches.get("Pattern")!.get("arraypattern")
    && branches.get("Term")!.get("arraypattern") === store.registered("Expressions.TypeScript.Patterns.ArrayPattern")); // a moved
  // schema, named by its dialect
  assert(branches.get("Term")!.get("arrayexpression") === store.registered("Expressions.Framework.List")); // the shape matches exactly
  assert(branches.get("Term")!.get("literal").refines === store.registered("Expressions.Framework.Literal"));
  assert(store.registered("Expressions.TypeScript.TypeScript5.Expression").properties.get("root").schema === unions.get("Term"));
  assert(branches.get("Term")!.get("assignmentexpression").properties.get("left").schema === unions.get("Pattern")); // what holds a pattern
  assert(branches.get("Term")!.get("tsasexpression").properties.get("typeAnnotation").schema === unions.get("Type")); // and a type
  assert(store.registered("Expressions.TypeScript.Types.TSQualifiedName").properties.get("right").schema === branches.get("Term")!.get("identifier"));
  // A sublanguage's value is a term of its dialect, read as an argument, its patterns and types among an arrow's:
  // x = ([a, b = 1]: number[]) => a as string
  const arrow = ts((u: any) => u.assignmentexpression((r: any) => r.left((t: any) => t.identifier((i: any) => i.name("x"))).operator("=").right(
    (v: any) => v.arrowfunctionexpression((a: any) => a.params([(p: any) => p.arraypattern((ap: any) => ap.elements([
      (e: any) => e.identifier((i: any) => i.name("a")),
      (e: any) => e.assignmentpattern((s: any) => s.left((t: any) => t.identifier((i: any) => i.name("b"))).right(
        (t: any) => t.literal((li: any) => li.value((n: any) => n.as_scalar(1.0)))))]).typeAnnotation(
      (ta: any) => ta.typeAnnotation((t: any) => t.tsarraytype((at: any) => at.elementType((y: any) => y.tsnumberkeyword((k: any) => k))))))])
      .body((b: any) => b.tsasexpression((s: any) => s.expression((e: any) => e.identifier((i: any) => i.name("a"))).typeAnnotation(
        (t: any) => t.tsstringkeyword((k: any) => k))))))));
  const form = F.form(arrow);
  assert(form.kind === "assignmentexpression" && equal(form.attributes, map({ operator: "=" }))); // its target, a pattern, an argument
  assert(F.dialectOf(form.arguments[0]) === "Expressions.TypeScript.Patterns.Term" && equal(F.form(form.arguments[0]).attributes, map({ name: "x" })));
  const lambda = F.form(form.arguments[1]); // Let's `value`, unset, its body, and its parameter
  assert(lambda.kind === "arrowfunctionexpression" && lambda.arguments[0] === null && lambda.arguments.length === 3);
  const pattern = F.form(lambda.arguments[2]);
  assert(pattern.kind === "arraypattern" && same([...pattern.attributes].map(([name, part]) => // its annotation a helper;
    [name, (part as F.Part).kind, (part as F.Part).attributes.size, (part as F.Part).size].join(" ")), [ // its decorators and
    "typeAnnotation tstypeannotation 0 1", "decorators decorators 0 0", "elements elements 0 2"])); // elements containers of
  // terms, each its size
  assert(same(pattern.arguments.slice(0, 2).map((e) => F.form(e).kind), ["identifier", "assignmentpattern"]));
  assert(F.dialectOf(pattern.arguments[2]) === "Expressions.TypeScript.Types.Term" && F.form(pattern.arguments[2]).kind === "tsarraytype");
  const node = TS.tree(lambda.arguments[2]); // its helpers nested, as the printer and the checks read it
  assert(node.kind === "arraypattern" && same([...node.lists.keys()], ["decorators", "elements"]) && node.lists.get("decorators")!.length === 0
    && node.lists.get("elements")![0] === pattern.arguments[0] && node.lists.get("elements")![1] === pattern.arguments[1]);
  assert(TS.tree((node.helpers.get("typeAnnotation") as F.Node).slots.get("typeAnnotation")).kind === "tsarraytype");
  assert(same([...TS.boundNames(node)], ["a", "b"]) && TS.boundNames(null).size === 0 && TSP.boundNames === TS.boundNames);
  assert(same(TS.DIALECT.validate(arrow), ["left: identifier 'x' is not bound"])); // an assignment's target, a reference
  assert(TS.DIALECT.validate(arrow, ["x"]).length === 0 && roundTrips(arrow) && F.same(ts(TS.DIALECT.make(form)), arrow));
  assert(same([...SY.free(arrow, TS.DIALECT)], ["x"])); // its parameters bind its body
  // A literal keeps what its text encodes beside its text, its problems and its value: a number is a float, a bigint an
  // int; TypeScript5's own constant holds null and a regular expression, which no schema holds.
  for (const [spec, attributes, domain] of [
    [(u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(255.0)).text("0x_FF").base(16n).separators(true)),
      { value: 255.0, text: "0x_FF", base: 16n, separators: true }, "number"],
    [(u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(1000n)).text("1_000n").separators(true)),
      { value: 1000n, text: "1_000n", separators: true }, "bigint"],
    [(u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar("a")).text("'a'").quote("'")),
      { value: "a", text: "'a'", quote: "'" }, "string"],
    [(u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(true)).text("true")), { value: true, text: "true" }, "boolean"],
  ] as [unknown, Record<string, unknown>, string][]) {
    const spelled = ts(spec);
    assert(equal(F.form(spelled).attributes, map(attributes)) && TS.DIALECT.validate(spelled).length === 0);
    assert(TS.DIALECT.infer(spelled).name() === domain && TV.OfAny(spelled) === attributes.value); // its value
  }
  const nil = ts((u: any) => u.constant((r: any) => r.singleton("null").text("null")));
  const regex = ts((u: any) => u.constant((r: any) => r.pattern("a+").flags("gu")));
  assert(equal(F.form(regex).attributes, map({ pattern: "a+", flags: "gu" })) && TS.DIALECT.validate(regex).length === 0
    && TS.DIALECT.validate(nil).length === 0);
  assert(TS.DIALECT.infer(nil).name() === "null" && TS.DIALECT.infer(regex).name() === "RegExp");
  assert(same(TS.DIALECT.kinds().get("literal")!.spelling, ["text", "problems", "quote", "base", "separators"]));
  raises(TypeError, () => TS.DIALECT.spec(new Uint8Array([120])), "expected a term, a writer or a native value, got b'x'"); // no literal holds bytes
  assert(equal(F.form(ts(1n)).attributes, map({ value: 1n })) && equal(F.form(ts(1.5)).attributes, map({ value: 1.5 }))); // a bigint, a number
  // Evaluation is shallow: a literal's value; every other kind refused by name, a constant among them
  for (const [spec, message] of [[(u: any) => u.thisexpression((r: any) => r), "TypeScript5 thisexpression is not evaluated"],
    [(u: any) => u.constant((r: any) => r.singleton("null")), "TypeScript5 constant is not evaluated"]] as [unknown, string][]) {
    raises(NotImplementedError, () => TV.OfAny(ts(spec)), message);
  }
  raises(NotImplementedError, () => TV.OfAny(arrow), "TypeScript5 assignmentexpression is not evaluated");
  assert(same([...TV.EVALUATED], ["literal"]));
  // Each kind's own checks, mbse-programs', and those of the types, patterns and JSX it holds
  const x = (e: any) => e.identifier((i: any) => i.name("x")); // a spec of `x`, in any union that has identifiers
  for (const [spec, problems] of [
    [(u: any) => u.identifier((r: any) => r.name("a b")), ["an identifier's name cannot be 'a b'"]],
    [(u: any) => u.identifier((r: any) => r.name("$a")), []], // `$` is a letter
    [(u: any) => u.privateidentifier((r: any) => r.name("1")), ["a privateidentifier's name cannot be '1'"]],
    [(u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar("s")).quote("`").base(3n)),
      ["a literal's quote cannot be '`'", "a literal's base cannot be 3"]],
    [(u: any) => u.literal((r: any) => r.text("08n").problems(["a bigint has no leading zeros"])), ["a bigint has no leading zeros"]],
    [(u: any) => u.constant((r: any) => r), ["a constant needs a singleton or a regular expression"]],
    [(u: any) => u.constant((r: any) => r.text("/a").problems(["unterminated regular expression"])), ["unterminated regular expression"]],
    [(u: any) => u.constant((r: any) => r.singleton("null").pattern("a")), ["a constant holds a singleton or a regular expression, not both"]],
    [(u: any) => u.constant((r: any) => r.singleton("undefined")), ["a constant's singleton must be null, got 'undefined'"]],
    [(u: any) => u.constant((r: any) => r.flags("g")), ["a regular expression needs a pattern"]],
    [(u: any) => u.constant((r: any) => r.pattern("a")), []], // its flags, none
    [(u: any) => u.constant((r: any) => r.pattern("a").flags("gq")), ["a regular expression's flags cannot be 'gq'"]],
    [(u: any) => u.constant((r: any) => r.pattern("a").flags("gg")), ["a regular expression's flags cannot be 'gg'"]],
    [(u: any) => u.templateliteral((r: any) => r.quasis([(q: any) => q.raw("a")]).expressions([x])),
      ["a templateliteral needs one more quasi than expressions"]],
    [(u: any) => u.sequenceexpression((r: any) => r.expressions([x])), ["a sequenceexpression needs two expressions"]],
    [(u: any) => u.tsasexpression((r: any) => r.expression(x).typeAnnotation((t: any) => t.tsuniontype((y: any) => y.types([
      (z: any) => z.tsintersectiontype((w: any) => w.types([(v: any) => v.tsanykeyword((k: any) => k)]))])))),
      ["typeAnnotation: a tsuniontype needs two types", "typeAnnotation: argument 0: a tsintersectiontype needs two types"]], // a type's, a term
    [(u: any) => u.tsasexpression((r: any) => r.expression(x).typeAnnotation((t: any) => t.tstemplateliteraltype((y: any) => y.quasis([
      (q: any) => q.raw("a")]).types([(v: any) => v.tsstringkeyword((k: any) => k)])))),
      ["typeAnnotation: a tstemplateliteraltype needs one more quasi than types"]],
    [(u: any) => u.arrowfunctionexpression((r: any) => r.typeParameters((d: any) => d.params([]))), ["a tstypeparameterdeclaration needs a type parameter"]],
    [(u: any) => u.arrowfunctionexpression((r: any) => r.params([(p: any) => p.identifier((i: any) => i.name("a b"))])),
      ["argument 0: an identifier's name cannot be 'a b'"]], // a pattern's, a term
    [(u: any) => u.binaryexpression((r: any) => r.left(x).operator("<>").right(x)), []], // an operator outside its vocabulary
    [(u: any) => u.binaryexpression((r: any) => r.left(x).right(x)), ["a binaryexpression needs an operator"]],
    [(u: any) => u.tsasexpression((r: any) => r.expression(x)), ["a tsasexpression needs a typeAnnotation"]], // a type it needs
    [(u: any) => u.assignmentexpression((r: any) => r.right(x)), ["an assignmentexpression needs an operator", "an assignmentexpression needs a left"]],
    [(u: any) => u.unaryexpression((r: any) => r.operator("-")), ["an unaryexpression needs an argument"]],
    [(u: any) => u.yieldexpression((r: any) => r), []], [(u: any) => u.importexpression((r: any) => r.source(x)), []], // optional slots
    [(u: any) => u.arrowfunctionexpression((r: any) => r), []], // a body of statements, not part of the term
    [(u: any) => u.classexpression((r: any) => r), []],
    [(u: any) => u.assignmentexpression((r: any) => r.left((t: any) => t.assignmentpattern((a: any) => a.left(x))).operator("=").right(x)),
      ["left: an assignmentpattern needs a right"]], // a pattern's term, labelled by its slot
  ] as [unknown, string[]][]) {
    const found = tsChecked(spec, ["x", "$a"]);
    assert(same(found, problems), `${Repr.repr(found)} != ${Repr.repr(problems)}`);
  }
  assert(same(TS.DIALECT.validate(ts((u: any) => u.binaryexpression((r: any) => r.left(x).operator("<>").right(x))), ["x"], true),
    ["'<>' is not a core operation"]));
  // What each kind binds: a function's and an arrow's parameters; a member's property and a key, not computed, are names
  // rather than references
  for (const [spec, free] of [
    [(u: any) => u.memberexpression((r: any) => r.object(x).property((p: any) => p.identifier((i: any) => i.name("y")))), ["x"]],
    [(u: any) => u.memberexpression((r: any) => r.object(x).computed(true).property((p: any) => p.identifier((i: any) => i.name("y")))), ["x", "y"]],
    [(u: any) => u.objectexpression((r: any) => r.properties([(p: any) => p.property((q: any) => q.key((k: any) => k.identifier((i: any) => i.name("k")))
      .value((v: any) => v.literal((li: any) => li.value((n: any) => n.as_scalar("s"))))), (p: any) => p.property((q: any) => q.computed(true)
      .key(x).value(x)), (p: any) => p.property((q: any) => q.key((k: any) => k.literal((li: any) => li.value((n: any) => n.as_scalar("s")))).value(x)),
      (p: any) => p.property((q: any) => q.value(x))])), ["x"]], // a key that is a literal, or none, names nothing
    [(u: any) => u.tsasexpression((r: any) => r.expression(x).typeAnnotation((t: any) => t.tstypeliteral((y: any) => y.members([
      (m: any) => m.tspropertysignature((s: any) => s.key((k: any) => k.identifier((i: any) => i.name("k")))),
      (m: any) => m.tsmethodsignature((s: any) => s.computed(true).key((k: any) => k.identifier((i: any) => i.name("z")))),
      (m: any) => m.comment((c: any) => c.text(" c"))])))), ["x", "z"]], // a type literal's keys, within a type
    [(u: any) => u.functionexpression((r: any) => r.params([(p: any) => p.objectpattern((o: any) => o.properties([
      (q: any) => q.property((s: any) => s.key((k: any) => k.identifier((i: any) => i.name("a"))).value((v: any) => v.identifier((i: any) => i.name("b")))),
      (q: any) => q.restelement((s: any) => s.argument((a: any) => a.identifier((i: any) => i.name("c"))))])),
    (p: any) => p.tsparameterproperty((s: any) => s.parameter((a: any) => a.assignmentpattern((t: any) => t.left((i: any) => i.identifier((j: any) => j.name("d"))).right(
      (e: any) => e.identifier((i: any) => i.name("b")))))),
    (p: any) => p.memberexpression((m: any) => m.object(x).property((y: any) => y.identifier((i: any) => i.name("y"))))])), ["x"]],
    [(u: any) => u.arrowfunctionexpression((r: any) => r.params([(p: any) => p.identifier((i: any) => i)]).body(x)), ["x"]], // a name not written binds none
  ] as [unknown, string[]][]) {
    assert(same([...SY.free(ts(spec), TS.DIALECT)].sort(), free));
  }
  // Empty value objects are held and read back present (mbse-schemas 0.26.9): a fragment's opening and closing, an empty
  // template, an empty quasi, each printed as written
  for (const [spec, text, held] of [
    [(u: any) => u.jsxfragment((f: any) => f.openingFragment((h: any) => h).closingFragment((h: any) => h)), "<></>", ["openingFragment", "closingFragment"]],
    [(u: any) => u.taggedtemplateexpression((r: any) => r.tag(x).quasi((q: any) => q.quasis([(e: any) => e]))), "x``", ["quasi"]],
    [(u: any) => u.templateliteral((r: any) => r.quasis([(e: any) => e])), "``", ["quasis"]],
  ] as [unknown, string, string[]][]) {
    const value = ts(spec);
    assert(roundTrips(value) && TSText.ToText(value) === text && same([...F.form(value).attributes.keys()], held), text);
  }
}

<a id="DIA-22"></a>
## DIA-22 · TypeScript5 printed as TypeScript writes it: every kind, its types, patterns and JSX, by precedence, as an expression statement holds it, a literal in its spelling or its value's

In [ ]:
{
  const ts = (spec: unknown): any => term(spec, TS.DIALECT);
  const shown = (spec: unknown): string => TSText.ToText(ts(spec)); // a TypeScript5 term's source
  // Specs of terms, types and patterns, written through the builders, each in any union whose branches it names
  const [x, y, z] = ["x", "y", "z"].map((n) => (e: any) => e.identifier((i: any) => i.name(n)));
  const one = (e: any) => e.literal((r: any) => r.value((n: any) => n.as_scalar(1.0)));
  const number = (t: any) => t.tsnumberkeyword((k: any) => k);
  const string = (t: any) => t.tsstringkeyword((k: any) => k);
  const [T_, U_] = ["T", "U"].map((n) => (t: any) => t.tstypereference((r: any) => r.typeName((m: any) => m.identifier((i: any) => i.name(n)))));
  const empty = (e: any) => e.objectexpression((r: any) => r); // `{}`
  // A number as JavaScript's `String(number)` writes it, a string in its quote with its escapes
  assert(same([NaN, Infinity, -Infinity, 0.0, -0.0, 100.0, 1.5, 0.001, 1e21, 1.5e-7, 5e-7, -2.5, 123456789012345680000.0, 1e-6]
    .map(TSText.number), ["NaN", "Infinity", "-Infinity", "0", "0", "100", "1.5", "0.001", "1e+21", "1.5e-7", "5e-7", "-2.5",
    "123456789012345680000", "0.000001"]));
  assert(TSText.string("it's \"x\"\\\n\x01\x7f\u2028é", "\"") === "\"it's \\\"x\\\"\\\\\\n\\x01\\x7f\\u2028é\"");
  assert(TSText.string("it's", "'") === "'it\\'s'");
  for (const [spec, text] of [
    // literals: a value's default spelling, else its text
    [(u: any) => u.literal((r: any) => r.value((n: any) => n.as_scalar(true))), "true"],
    [(u: any) => u.literal((r: any) => r.value((n: any) => n.as_scalar(false))), "false"],
    [(u: any) => u.literal((r: any) => r.value((n: any) => n.as_scalar(5n))), "5n"],
    [(u: any) => u.literal((r: any) => r.value((n: any) => n.as_scalar(1.5))), "1.5"],
    [(u: any) => u.literal((r: any) => r.value((n: any) => n.as_scalar("a\n"))), '"a\\n"'],
    [(u: any) => u.literal((r: any) => r.value((n: any) => n.as_scalar("a")).quote("'")), "'a'"],
    [(u: any) => u.literal((r: any) => r.value((n: any) => n.as_scalar(16.0)).text("0x10")), "0x10"],
    [(u: any) => u.binaryexpression((r: any) => r.left((e: any) => e.literal((li: any) => li.value((n: any) => n.as_scalar(-5n))))
        .operator("**").right(one)), "(-5n) ** 1"],  // a negative value's precedence, a unary operator's
    [(u: any) => u.constant((r: any) => r.singleton("null").text("null")), "null"],
    [(u: any) => u.constant((r: any) => r.singleton("null")), "null"],
    [(u: any) => u.constant((r: any) => r.pattern("a+").flags("g")), "/a+/g"],
    [(u: any) => u.constant((r: any) => r.pattern("a")), "/a/"],
    // names and templates
    [(u: any) => u.arrowfunctionexpression((r: any) => r.params([(p: any) => p.identifier((i: any) => i.decorators([(d: any) => d.expression(x)])
        .name("y").optional(true).typeAnnotation((a: any) => a.typeAnnotation(number))), (p: any) => p.identifier((i: any) => i)])),
     "(@x y?: number, ) => {}"],  // a parameter's decorators, `?` and type; one of no name, as held
    [(u: any) => u.memberexpression((r: any) => r.object(x).property((p: any) => p.privateidentifier((i: any) => i.name("p")))), "x.#p"],
    [(u: any) => u.templateliteral((r: any) => r), "``"],
    [(u: any) => u.templateliteral((r: any) => r.quasis([(q: any) => q.tail(true)])), "``"],
    [(u: any) => u.templateliteral((r: any) => r.quasis([(q: any) => q.raw("a"), (q: any) => q.tail(true)]).expressions([x])), "`a${x}`"],
    [(u: any) => u.taggedtemplateexpression((r: any) => r.tag(x).typeArguments((a: any) => a.params([number])).quasi(
        (q: any) => q.quasis([(e: any) => e.raw("a").tail(true)]))), "x<number>`a`"],
    [(u: any) => u.parenthesizedexpression((r: any) => r.expression((e: any) => e.sequenceexpression((s: any) => s.expressions([x, y])))), "(x, y)"],
    // displays
    [(u: any) => u.arrayexpression((r: any) => r), "[]"],
    [(u: any) => u.arrayexpression((r: any) => r.elements([(e: any) => e.typescript5((t: any) => t.elision((h: any) => h)),
                                                       (e: any) => e.typescript5(x), (e: any) => e.typescript5((t: any) => t.elision((h: any) => h))])),
     "[, x,,]"],  // a hole as nothing, a last hole kept
    [empty, "({})"],  // an object, as a statement holds it
    [(u: any) => u.objectexpression((r: any) => r.properties([
        (p: any) => p.property((q: any) => q.key(x).shorthand(true).value(x)),
        (p: any) => p.property((q: any) => q.key(y).shorthand(true).value((v: any) => v.assignmentpattern((a: any) => a.left(y).right(one)))),
        (p: any) => p.property((q: any) => q.propertyKind("get").key(z).value((v: any) => v.functionexpression((f: any) => f))),
        (p: any) => p.property((q: any) => q.propertyKind("set").key(z).value((v: any) => v.functionexpression((f: any) => f.params([x])))),
        (p: any) => p.property((q: any) => q.method(true).key(x).value((v: any) => v.functionexpression((f: any) => f.isAsync(true).generator(true)))),
        (p: any) => p.property((q: any) => q.method(true).optional(true).key(x).value((v: any) => v.functionexpression((f: any) => f))),
        (p: any) => p.property((q: any) => q.method(true).key(x).value((v: any) => v.tsemptybodyfunctionexpression(
            (f: any) => f.returnType((a: any) => a.typeAnnotation(number))))),
        (p: any) => p.property((q: any) => q.computed(true).key(x).value((v: any) => v.sequenceexpression((s: any) => s.expressions([x, y])))),
        (p: any) => p.spreadelement((s: any) => s.argument(x))])),
     "({ x, y = 1, get z() {}, set z(x) {}, async *x() {}, x?() {}, x(): number, [x]: (x, y), ...x })"],
    // functions and classes: a body of statements is written `{}`
    [(u: any) => u.functionexpression((r: any) => r), "(function() {})"],
    [(u: any) => u.sequenceexpression((s: any) => s.expressions([x, (e: any) => e.functionexpression((r: any) => r.isAsync(true).generator(true)
        .id((i: any) => i.name("f")).typeParameters((d: any) => d.params([(p: any) => p.name((i: any) => i.name("T"))])).params([
            (p: any) => p.identifier((i: any) => i.name("a").typeAnnotation((a: any) => a.typeAnnotation(T_)))]).returnType(
            (a: any) => a.typeAnnotation(T_)))])), "x, async function* f<T>(a: T): T {}"],
    [(u: any) => u.tsemptybodyfunctionexpression((r: any) => r.isAsync(true).generator(true)), "async *()"],
    [(u: any) => u.tsemptybodyfunctionexpression((r: any) => r), "()"],
    [(u: any) => u.arrowfunctionexpression((r: any) => r.typeParameters((d: any) => d.params([(p: any) => p.name((i: any) => i.name("T"))]))
        .params([x])), "<T,>(x) => {}"],  // a lone type parameter, as a .tsx file reads it
    [(u: any) => u.arrowfunctionexpression((r: any) => r.isAsync(true).typeParameters((d: any) => d.params([
        (p: any) => p.name((i: any) => i.name("T")).constraint(U_)])).body(x)), "async <T extends U>() => x"],
    [(u: any) => u.arrowfunctionexpression((r: any) => r.typeParameters((d: any) => d.params([
        (p: any) => p.name((i: any) => i.name("T")), (p: any) => p.name((i: any) => i.name("U"))])).body(x)), "<T, U>() => x"],
    [(u: any) => u.arrowfunctionexpression((r: any) => r.body(empty)), "() => ({})"],  // an object body, parenthesized
    [(u: any) => u.arrowfunctionexpression((r: any) => r.body((b: any) => b.memberexpression((m: any) => m.object(empty).property(x)))),
     "() => ({}.x)"],
    [(u: any) => u.arrowfunctionexpression((r: any) => r.body((b: any) => b.sequenceexpression((s: any) => s.expressions([x, y])))), "() => (x, y)"],
    [(u: any) => u.classexpression((r: any) => r), "(class {})"],
    [(u: any) => u.sequenceexpression((s: any) => s.expressions([x, (e: any) => e.classexpression((r: any) => r.decorators([(d: any) => d.expression(x)])
        .id((i: any) => i.name("K")).typeParameters((d: any) => d.params([(p: any) => p.name((i: any) => i.name("T"))])).superClass(x)
        .superTypeArguments((a: any) => a.params([number])).implements([(c: any) => c.expression(y).typeArguments((a: any) => a.params([T_])),
                                                                       (c: any) => c.expression(z)]))])),
     "x, @x class K<T> extends x<number> implements y<T>, z {}"],
    [(u: any) => u.sequenceexpression((s: any) => s.expressions([x, (e: any) => e.classexpression((r: any) => r.superClass((c: any) => c.callexpression(
        (f: any) => f.callee(x))))])), "x, class extends x() {}"],
    [(u: any) => u.metaproperty((r: any) => r.meta((i: any) => i.name("new")).property((i: any) => i.name("target"))), "new.target"],
    // operations
    [(u: any) => u.memberexpression((r: any) => r.object(one).property(x)), "(1).x"],  // `1.x` would be a number
    [(u: any) => u.memberexpression((r: any) => r.object((e: any) => e.literal((li: any) => li.value((n: any) => n.as_scalar(1.5)))).property(x)), "1.5.x"],
    [(u: any) => u.memberexpression((r: any) => r.object((e: any) => e.literal((li: any) => li.value((n: any) => n.as_scalar(1000.0)).text("1_000"))).property(x)),
     "(1_000).x"],
    [(u: any) => u.memberexpression((r: any) => r.object((e: any) => e.thisexpression((h: any) => h)).computed(true).optional(true).property(y)), "this?.[y]"],
    [(u: any) => u.memberexpression((r: any) => r.object(x).computed(true).property((p: any) => p.sequenceexpression((s: any) => s.expressions([x, y])))),
     "x[x, y]"],
    [(u: any) => u.chainexpression((r: any) => r.expression((e: any) => e.memberexpression((m: any) => m.object(x).optional(true).property(y)))), "x?.y"],
    [(u: any) => u.callexpression((r: any) => r.callee(x).optional(true).typeArguments((a: any) => a.params([number])).arguments([
        y, (e: any) => e.sequenceexpression((s: any) => s.expressions([x, y]))])), "x?.<number>(y, (x, y))"],
    [(u: any) => u.newexpression((r: any) => r.callee((c: any) => c.callexpression((f: any) => f.callee(x)))), "new (x())()"],
    [(u: any) => u.newexpression((r: any) => r.callee((c: any) => c.memberexpression((m: any) => m.object((o: any) => o.tsnonnullexpression(
        (n: any) => n.expression(x))).property(y)))), "new (x!.y)()"],  // a callee that calls, in parentheses
    [(u: any) => u.newexpression((r: any) => r.callee((c: any) => c.memberexpression((m: any) => m.object(x).property(y))).typeArguments(
        (a: any) => a.params([T_])).arguments([z])), "new x.y<T>(z)"],
    [(u: any) => u.importexpression((r: any) => r.phase("defer").source(x)), "import.defer(x)"],
    [(u: any) => u.importexpression((r: any) => r.source(x).options(y)), "import(x, y)"],
    [(u: any) => u.updateexpression((r: any) => r.operator("++").prefix(true).argument(x)), "++x"],
    [(u: any) => u.updateexpression((r: any) => r.operator("--").argument((a: any) => a.binaryexpression((b: any) => b.left(x).operator("+").right(y)))),
     "(x + y)--"],
    [(u: any) => u.unaryexpression((r: any) => r.operator("void").argument(x)), "void x"],
    [(u: any) => u.unaryexpression((r: any) => r.operator("-").argument((a: any) => a.unaryexpression((b: any) => b.operator("-").argument(x)))), "- -x"],
    [(u: any) => u.unaryexpression((r: any) => r.operator("+").argument((a: any) => a.literal((li: any) => li.value((n: any) => n.as_scalar(1.0)).text("+1")))), "+ +1"],
    [(u: any) => u.unaryexpression((r: any) => r.operator("!").argument((a: any) => a.binaryexpression((b: any) => b.left(x).operator("in").right(y)))), "!(x in y)"],
    [(u: any) => u.awaitexpression((r: any) => r.argument(x)), "await x"],
    // binary and logical operators by precedence: `**`'s left, `??` against `||` and `&&`
    [(u: any) => u.binaryexpression((r: any) => r.left((a: any) => a.binaryexpression((b: any) => b.left(x).operator("+").right(y))).operator("*").right(
        (a: any) => a.binaryexpression((b: any) => b.left(x).operator("-").right(y)))), "(x + y) * (x - y)"],
    [(u: any) => u.binaryexpression((r: any) => r.left((a: any) => a.binaryexpression((b: any) => b.left(x).operator("**").right(y))).operator("**").right(
        (a: any) => a.binaryexpression((b: any) => b.left(x).operator("**").right(y)))), "(x ** y) ** x ** y"],
    [(u: any) => u.binaryexpression((r: any) => r.left((a: any) => a.awaitexpression((b: any) => b.argument(x))).operator("**").right(y)), "(await x) ** y"],
    [(u: any) => u.binaryexpression((r: any) => r.left((a: any) => a.tstypeassertion((b: any) => b.typeAnnotation(number).expression(x))).operator("**").right(y)),
     "(<number>x) ** y"],
    [(u: any) => u.logicalexpression((r: any) => r.left((a: any) => a.logicalexpression((b: any) => b.left(x).operator("??").right(y))).operator("||").right(
        (a: any) => a.logicalexpression((b: any) => b.left(x).operator("&&").right(y)))), "(x ?? y) || x && y"],
    [(u: any) => u.logicalexpression((r: any) => r.left(x).operator("??").right((a: any) => a.logicalexpression((b: any) => b.left(y).operator("||").right(z)))),
     "x ?? (y || z)"],
    [(u: any) => u.logicalexpression((r: any) => r.left((a: any) => a.logicalexpression((b: any) => b.left(x).operator("??").right(y))).operator("??").right(z)),
     "x ?? y ?? z"],
    [(u: any) => u.conditionalexpression((r: any) => r.test((t: any) => t.conditionalexpression((c: any) => c.test(x).consequent(y).alternate(z)))
        .consequent((c: any) => c.assignmentexpression((a: any) => a.left(x).operator("=").right(y))).alternate(z)), "(x ? y : z) ? x = y : z"],
    // an assignment's target, a pattern: a term it may be, or a destructuring
    [(u: any) => u.assignmentexpression((r: any) => r.left((t: any) => t.memberexpression((m: any) => m.object(x).property(y))).operator("+=").right(one)),
     "x.y += 1"],
    [(u: any) => u.assignmentexpression((r: any) => r.left((t: any) => t.arraypattern((a: any) => a.elements([x, (e: any) => e.elision((h: any) => h),
        (e: any) => e.restelement((s: any) => s.argument(y))]))).operator("=").right(z)), "[x,, ...y] = z"],
    [(u: any) => u.assignmentexpression((r: any) => r.left((t: any) => t.objectpattern((o: any) => o)).operator("=").right(z)), "({} = z)"],
    [(u: any) => u.assignmentexpression((r: any) => r.left((t: any) => t.tsasexpression((a: any) => a.expression(x).typeAnnotation(
        (k: any) => k.tsanykeyword((h: any) => h)))).operator("=").right(z)), "(x as any) = z"],
    [(u: any) => u.assignmentexpression((r: any) => r.left((t: any) => t.parenthesizedexpression((a: any) => a.expression(x))).operator("??=").right(z)),
     "(x) ??= z"],
    [(u: any) => u.yieldexpression((r: any) => r), "yield"],
    [(u: any) => u.yieldexpression((r: any) => r.argument((a: any) => a.sequenceexpression((s: any) => s.expressions([x, y])))), "yield (x, y)"],
    [(u: any) => u.tsasexpression((r: any) => r.expression((e: any) => e.tsasexpression((a: any) => a.expression(x).typeAnnotation(T_))).typeAnnotation(U_)),
     "x as T as U"],
    [(u: any) => u.tssatisfiesexpression((r: any) => r.expression((e: any) => e.binaryexpression((b: any) => b.left(x).operator("==").right(y)))
        .typeAnnotation((t: any) => t.tsuniontype((n: any) => n.types([number, string])))), "(x == y) satisfies number | string"],
    [(u: any) => u.tsinstantiationexpression((r: any) => r.expression((e: any) => e.tsnonnullexpression((n: any) => n.expression(x))).typeArguments(
        (a: any) => a.params([T_]))), "x!<T>"],
    // patterns, as parameters
    [(u: any) => u.arrowfunctionexpression((r: any) => r.params([
        (p: any) => p.arraypattern((a: any) => a.decorators([(d: any) => d.expression(x)]).elements([x, (e: any) => e.elision((h: any) => h)]).optional(true)
            .typeAnnotation((t: any) => t.typeAnnotation((k: any) => k.tsarraytype((n: any) => n.elementType(number))))),
        (p: any) => p.objectpattern((o: any) => o.properties([(q: any) => q.property((s: any) => s.key(x).shorthand(true).value(x)),
            (q: any) => q.property((s: any) => s.key(y).value((v: any) => v.assignmentpattern((a: any) => a.left(z).right(one)))),
            (q: any) => q.restelement((s: any) => s.argument(z))])),
        (p: any) => p.objectpattern((o: any) => o.optional(true)),
        (p: any) => p.assignmentpattern((a: any) => a.left((t: any) => t.identifier((i: any) => i.name("x").typeAnnotation((b: any) => b.typeAnnotation(number))))
            .right(one)),
        (p: any) => p.restelement((s: any) => s.argument(y).typeAnnotation((t: any) => t.typeAnnotation((k: any) => k.tsarraytype((n: any) => n.elementType(T_))))),
        (p: any) => p.tsparameterproperty((s: any) => s.decorators([(d: any) => d.expression(x)]).accessibility("private").static(true).override(true)
            .readonly(true).parameter(z))])),
     "(@x [x,,]?: number[], { x, y: z = 1, ...z }, {}?, x: number = 1, ...y: T[], @x private static override readonly z) => {}"],
    // types, as `x as <type>`
    [(u: any) => u.tsasexpression((r: any) => r.expression(x).typeAnnotation((t: any) => t.tstupletype((n: any) => n.elementTypes([
        ...["tsanykeyword", "tsunknownkeyword", "tsnumberkeyword", "tsbigintkeyword", "tsbooleankeyword", "tsstringkeyword",
          "tssymbolkeyword", "tsobjectkeyword", "tsneverkeyword", "tsvoidkeyword", "tsundefinedkeyword", "tsnullkeyword",
          "tsintrinsickeyword", "tsthistype"].map((keyword) => (k: any) => k[keyword]((h: any) => h))])))),
     "x as [any, unknown, number, bigint, boolean, string, symbol, object, never, void, undefined, null, intrinsic, this]"],
    [(u: any) => u.tsasexpression((r: any) => r.expression(x).typeAnnotation((t: any) => t.tsuniontype((n: any) => n.types([
        (k: any) => k.tstypereference((q: any) => q.typeName((m: any) => m.tsqualifiedname((a: any) => a.left((l: any) => l.tsqualifiedname(
            (b: any) => b.left(x).right((i: any) => i.name("y")))).right((i: any) => i.name("z")))).typeArguments((a: any) => a.params([T_]))),
        (k: any) => k.tstypereference((q: any) => q.typeName((m: any) => m.thisexpression((h: any) => h))),
        (k: any) => k.tsparenthesizedtype((q: any) => q.typeAnnotation(number)),
        (k: any) => k.tsliteraltype((q: any) => q.literal((li: any) => li.literal((v: any) => v.value((n: any) => n.as_scalar("s"))))),
        (k: any) => k.tsliteraltype((q: any) => q.literal((li: any) => li.unaryexpression((v: any) => v.operator("-").argument(one)))),
        (k: any) => k.tsliteraltype((q: any) => q.literal((li: any) => li.templateliteral((v: any) => v.quasis([(e: any) => e.raw("a").tail(true)])))),
        (k: any) => k.tstemplateliteraltype((q: any) => q.quasis([(e: any) => e.raw("a"), (e: any) => e.tail(true)]).types([number])),
        (k: any) => k.tstemplateliteraltype((q: any) => q), (k: any) => k.tstemplateliteraltype((q: any) => q.quasis([(e: any) => e.tail(true)]))])))),
     'x as x.y.z<T> | this | (number) | "s" | -1 | `a` | `a${number}` | `` | ``'],
    [(u: any) => u.tsasexpression((r: any) => r.expression(x).typeAnnotation((t: any) => t.tsintersectiontype((n: any) => n.types([
        (k: any) => k.tsarraytype((a: any) => a.elementType((e: any) => e.tsuniontype((q: any) => q.types([number, string])))),
        (k: any) => k.tsuniontype((q: any) => q.types([number, (e: any) => e.tsfunctiontype((f: any) => f.returnType((a: any) => a.typeAnnotation(
            (v: any) => v.tsvoidkeyword((h: any) => h))))])),
        (k: any) => k.tstupletype((q: any) => q.elementTypes([
            (e: any) => e.tsnamedtuplemember((m: any) => m.label((i: any) => i.name("a")).elementType(number)),
            (e: any) => e.tsnamedtuplemember((m: any) => m.label((i: any) => i.name("b")).optional(true).elementType(string)),
            (e: any) => e.tsoptionaltype((o: any) => o.typeAnnotation(number)),
            (e: any) => e.tsresttype((o: any) => o.typeAnnotation((a: any) => a.tsarraytype((b: any) => b.elementType(T_))))])),
        (k: any) => k.tsfunctiontype((f: any) => f.typeParameters((d: any) => d.params([(p: any) => p.name((i: any) => i.name("T"))])).params([
            (p: any) => p.identifier((i: any) => i.name("a").typeAnnotation((a: any) => a.typeAnnotation(T_)))]).returnType(
            (a: any) => a.typeAnnotation(T_))),
        (k: any) => k.tsconstructortype((f: any) => f.abstract(true).returnType((a: any) => a.typeAnnotation(T_))),
        (k: any) => k.tsconstructortype((f: any) => f)])))),
     "x as (number | string)[] & (number | (() => void)) & [a: number, b?: string, number?, ...T[]] & (<T>(a: T) => T) & (abstract new () => T) & (new ())"],
    [(u: any) => u.tsasexpression((r: any) => r.expression(x).typeAnnotation((t: any) => t.tsuniontype((n: any) => n.types([
        (k: any) => k.tstypeliteral((q: any) => q),
        (k: any) => k.tstypeliteral((q: any) => q.members([
            (m: any) => m.tspropertysignature((s: any) => s.readonly(true).key(x).optional(true).typeAnnotation((a: any) => a.typeAnnotation(T_))),
            (m: any) => m.tsmethodsignature((s: any) => s.computed(true).key(y).optional(true).returnType((a: any) => a.typeAnnotation(
                (v: any) => v.tsvoidkeyword((h: any) => h)))),
            (m: any) => m.tsmethodsignature((s: any) => s.methodKind("get").key(z).returnType((a: any) => a.typeAnnotation(T_))),
            (m: any) => m.tsmethodsignature((s: any) => s.methodKind("method").key(z)),
            (m: any) => m.tscallsignaturedeclaration((s: any) => s.returnType((a: any) => a.typeAnnotation(T_))),
            (m: any) => m.tsconstructsignaturedeclaration((s: any) => s.params([x])),
            (m: any) => m.tsindexsignature((s: any) => s.accessibility("public").static(true).readonly(true).parameters([
                (p: any) => p.identifier((i: any) => i.name("k").typeAnnotation((a: any) => a.typeAnnotation(string)))]).typeAnnotation(
                (a: any) => a.typeAnnotation(T_))),
            (m: any) => m.tspropertysignature((s: any) => s.key((k: any) => k.literal((li: any) => li.value((n: any) => n.as_scalar("k"))))),
            (m: any) => m.tsindexsignature((s: any) => s.parameters([x]))]))])))),
     'x as {} | { readonly x?: T; [y]?(): void; get z(): T; z(); (): T; new (x); public static readonly [k: string]: T; "k"; [x]; }'],
    [(u: any) => u.tsasexpression((r: any) => r.expression(x).typeAnnotation((t: any) => t.tstypeliteral((q: any) => q.members([
        (m: any) => m.comment((c: any) => c.text(" first").trailing(true)),
        (m: any) => m.tspropertysignature((s: any) => s.key(x)),
        (m: any) => m.comment((c: any) => c.text(" of x").trailing(true)),
        (m: any) => m.comment((c: any) => c.block(true).text(" y ")),
        (m: any) => m.comment((c: any) => c)])))),
     "x as {\n    // first\n    x; // of x\n    /* y */\n    //\n}"],  // with comments, its members' lines
    [(u: any) => u.tsasexpression((r: any) => r.expression(x).typeAnnotation((t: any) => t.tsuniontype((n: any) => n.types([
        (k: any) => k.tsmappedtype((m: any) => m.readonly("+readonly").key((i: any) => i.name("K")).value(T_).nameType(U_).optional("?").body(T_)),
        (k: any) => k.tsmappedtype((m: any) => m.key((i: any) => i.name("K")).value(T_)),
        (k: any) => k.tsindexedaccesstype((a: any) => a.objectType((o: any) => o.tsuniontype((q: any) => q.types([T_, U_]))).indexType(
            (i: any) => i.tstypeoperator((o: any) => o.operator("keyof").typeAnnotation(T_)))),
        (k: any) => k.tstypeoperator((o: any) => o.operator("unique").typeAnnotation((s: any) => s.tssymbolkeyword((h: any) => h))),
        (k: any) => k.tstypeoperator((o: any) => o.operator("readonly").typeAnnotation((a: any) => a.tsuniontype((q: any) => q.types([T_, U_])))),
        (k: any) => k.tstypeoperator((o: any) => o.operator("keyof")),
        (k: any) => k.tstypequery((q: any) => q.exprName(x).typeArguments((a: any) => a.params([T_]))),
        (k: any) => k.tstypequery((q: any) => q.exprName((e: any) => e.tsimporttype((i: any) => i.source((s: any) => s.literal(
            (li: any) => li.value((n: any) => n.as_scalar("m"))))))),
        (k: any) => k.tsimporttype((i: any) => i.source((s: any) => s.literal((li: any) => li.value((n: any) => n.as_scalar("m")).quote("'"))).options(
            (o: any) => o.objectexpression((e: any) => e.properties([(p: any) => p.property((q: any) => q.key((w: any) => w.identifier((j: any) => j.name("with")))
                .value(empty))]))).qualifier((q: any) => q.identifier((j: any) => j.name("A"))).typeArguments((a: any) => a.params([T_])))])))),
     "x as { +readonly [K in T as U]?: T } | { [K in T] } | (T | U)[keyof T] | unique symbol | readonly (T | U) | keyof | typeof x<T> | "
     + "typeof import(\"m\") | import('m', { with: {} }).A<T>"],
    [(u: any) => u.tsasexpression((r: any) => r.expression(x).typeAnnotation((t: any) => t.tsconditionaltype((c: any) => c
        .checkType((k: any) => k.tsuniontype((q: any) => q.types([T_, U_])))
        .extendsType((k: any) => k.tsarraytype((a: any) => a.elementType((e: any) => e.tsinfertype((i: any) => i.typeParameter(
            (p: any) => p.name((n: any) => n.name("V")).constraint(string)))))).trueType((k: any) => k.tsconditionaltype((d: any) => d
                .checkType(T_).extendsType((e: any) => e.tsconditionaltype((f: any) => f.checkType(T_).extendsType(U_).trueType(T_).falseType(U_)))
                .trueType(T_).falseType(U_))).falseType((k: any) => k.tsconditionaltype((d: any) => d.checkType(T_)
                    .extendsType((e: any) => e.tsfunctiontype((f: any) => f.returnType((a: any) => a.typeAnnotation(
                        (g: any) => g.tsconditionaltype((h: any) => h.checkType(T_).extendsType(U_).trueType(T_).falseType(U_))))))
                    .trueType((e: any) => e.tsfunctiontype((f: any) => f.returnType((a: any) => a.typeAnnotation(T_)))).falseType(
                    (e: any) => e.tsconditionaltype((h: any) => h.checkType(T_).extendsType((g: any) => g.tsfunctiontype((f: any) => f)).trueType(T_)
                        .falseType(U_)))))))),
     "x as T | U extends (infer V extends string)[] ? T extends (T extends U ? T : U) ? T : U : "
     + "T extends (() => T extends U ? T : U) ? () => T : T extends () ? T : U"],
    [(u: any) => u.tsasexpression((r: any) => r.expression(x).typeAnnotation((t: any) => t.tsuniontype((n: any) => n.types([
        (k: any) => k.tsfunctiontype((f: any) => f.params([x]).returnType((a: any) => a.typeAnnotation((p: any) => p.tstypepredicate((q: any) => q
            .asserts(true).parameterName(x).typeAnnotation((b: any) => b.typeAnnotation(T_)))))),
        (k: any) => k.tsfunctiontype((f: any) => f.returnType((a: any) => a.typeAnnotation((p: any) => p.tstypepredicate((q: any) => q.parameterName(
            (m: any) => m.tsthistype((h: any) => h)))))),
        (k: any) => k.tsfunctiontype((f: any) => f.typeParameters((d: any) => d.params([(p: any) => p.const(true).isIn(true).isOut(true).name(
            (i: any) => i.name("T")).default(U_)])))])))),
     "x as ((x) => asserts x is T) | (() => this) | (<const in out T = U>())"],
    // JSX, as written
    [(u: any) => u.jsxelement((r: any) => r.openingElement((o: any) => o.name((n: any) => n.jsxmemberexpression((m: any) => m.object(
        (b: any) => b.jsxidentifier((i: any) => i.name("a"))).property((i: any) => i.name("b")))).typeArguments((a: any) => a.params([T_])).attributes([
            (a: any) => a.jsxattribute((t: any) => t.name((n: any) => n.jsxnamespacedname((s: any) => s.namespace((i: any) => i.name("x")).name(
                (i: any) => i.name("y"))))),
            (a: any) => a.jsxattribute((t: any) => t.name((n: any) => n.jsxidentifier((i: any) => i.name("z"))).value((v: any) => v.jsxexpressioncontainer(
                (c: any) => c.expression(one)))),
            (a: any) => a.jsxattribute((t: any) => t.name((n: any) => n.jsxidentifier((i: any) => i.name("e"))).value((v: any) => v.jsxelement(
                (c: any) => c.openingElement((o: any) => o.name((n: any) => n.jsxidentifier((i: any) => i.name("b"))).selfClosing(true))))),
            (a: any) => a.jsxattribute((t: any) => t.name((n: any) => n.jsxidentifier((i: any) => i.name("f"))).value((v: any) => v.jsxfragment((c: any) => c))),
            (a: any) => a.jsxspreadattribute((s: any) => s.argument((e: any) => e.sequenceexpression((q: any) => q.expressions([x, y]))))]))
        .children([(c: any) => c.jsxtext((t: any) => t.raw("hi ")), (c: any) => c.jsxtext((t: any) => t), (c: any) => c.jsxspreadchild((s: any) => s.expression(x)),
                   (c: any) => c.jsxelement((e: any) => e.openingElement((o: any) => o.name((n: any) => n.jsxidentifier((i: any) => i)))),
                   (c: any) => c.jsxexpressioncontainer((e: any) => e.expression((v: any) => v.jsxemptyexpression((h: any) => h)))])
        .closingElement((c: any) => c.name((n: any) => n.jsxmemberexpression((m: any) => m.object((b: any) => b.jsxidentifier((i: any) => i.name("a")))
            .property((i: any) => i.name("b")))))),
     "<a.b<T> x:y z={1} e=<b /> f=<></> {...(x, y)}>hi {...x}<></>{}</a.b>"],  // an element without its closing one closes its name
    // as an expression statement holds it: parenthesized where it would start with `{`, `function`, `class` or `let [`
    [(u: any) => u.memberexpression((r: any) => r.object(empty).property(x)), "({}.x)"],
    [(u: any) => u.binaryexpression((r: any) => r.left(empty).operator("+").right(x)), "({} + x)"],
    [(u: any) => u.callexpression((r: any) => r.callee((c: any) => c.functionexpression((f: any) => f))), "(function() {}())"],
    [(u: any) => u.taggedtemplateexpression((r: any) => r.tag((c: any) => c.classexpression((f: any) => f)).quasi(
      (q: any) => q.quasis([(e: any) => e.tail(true)]))), "(class {}``)"],
    [(u: any) => u.conditionalexpression((r: any) => r.test(empty).consequent(x).alternate(y)), "({} ? x : y)"],
    [(u: any) => u.tssatisfiesexpression((r: any) => r.expression(empty).typeAnnotation(T_)), "({} satisfies T)"],
    [(u: any) => u.updateexpression((r: any) => r.operator("++").argument(empty)), "({}++)"],
    [(u: any) => u.updateexpression((r: any) => r.operator("++").prefix(true).argument(empty)), "++{}"],
    [(u: any) => u.memberexpression((r: any) => r.object((e: any) => e.identifier((i: any) => i.name("let"))).computed(true).property(one)), "(let[1])"],
    [(u: any) => u.memberexpression((r: any) => r.object((e: any) => e.identifier((i: any) => i.name("let"))).property(x)), "let.x"],
  ] as [unknown, string][]) {
    const found = shown(spec);
    assert(found === text, `${Repr.repr(found)} != ${Repr.repr(text)}`);
  }
  raises(ValueError, () => shown((u: any) => u.binaryexpression((r: any) => r.left(x).operator("<>").right(y))),
    "TypeScript5 binaryexpression has no operator '<>'"); // an operator outside its vocabulary
}

<a id="DIA-23"></a>
## DIA-23 · TypeScript5 is complete: each node type of typescript-estree, TypeScript 5.0 to 5.9, kept as data, is a kind, a type or a pattern, or held by one, or not an expression

In [ ]:
import { readFileSync as readEstree } from "node:fs";

{
  interface EstreeNode {
    node: string; class?: string; kinds?: string[]; types?: string[]; patterns?: string[]; helper?: string; not?: string;
  }
  const estree = globalThis.JSON.parse(readEstree("../../grammars/typescript5-estree.json", "utf8")) as {
    nodes: EstreeNode[]; concrete: { union: string; kind: string }[] }; // typescript-estree's nodes, as data
  const unions = new Map([["Term", "TypeScript5.Term"], ["Type", "Types.Term"], ["Pattern", "Patterns.Term"],
    ["TypeElement", "Types.TypeElement"]].map(([name, path]) =>
    [name as string, new Set<string>(store.registered(`Expressions.TypeScript.${path}`).branches.map((b: any) => b.name))]));
  const mapped = new Map([...unions.keys()].map((name) => [name, new Set<string>()]));
  for (const node of estree.nodes) { // each a kind, a type, a pattern, any of them at once, a helper, or not an expression, with its reason
    const ways = (["kinds", "types", "patterns", "helper", "not"] as const).filter((way) => way in node);
    assert(ways.length > 0 && (ways.every((way) => ["kinds", "types", "patterns"].includes(way)) || ways.length === 1), node.node);
    for (const [way, union] of [["kinds", "Term"], ["types", "Type"], ["patterns", "Pattern"]] as const) {
      const held = node[way];
      if (held !== undefined) {
        assert(held.length > 0 && held.every((tag) => unions.get(union)!.has(tag)), node.node);
        held.forEach((tag) => mapped.get(union)!.add(tag));
      }
    }
    if (node.helper !== undefined) { // a value object a kind holds, not a term
      const owner = TST.HELPERS.includes(node.helper) ? "Types" : "TypeScript5"; // an object type's members, `TypeScript Types`'
      const helper = store.registered(`Expressions.TypeScript.${owner}.${node.helper}`);
      assert(helper !== null && helper !== undefined && !helper.ref && !unions.get("Term")!.has(node.helper.toLowerCase()), node.node);
      if (unions.get("TypeElement")!.has(node.helper.toLowerCase())) mapped.get("TypeElement")!.add(node.helper.toLowerCase());
    }
    if (node.not !== undefined) assert(typeof node.not === "string" && node.not !== "", node.node);
  }
  assert(estree.nodes.length === 168 && new Set(estree.nodes.map((node) => node.node)).size === 168); // every node type, once
  assert(same(estree.nodes.filter((node) => node.kinds !== undefined && node.class !== "Expression").map((node) => node.node), [
    "AssignmentPattern", "JSXEmptyExpression", "PrivateIdentifier", "Property", "SpreadElement",
    "TSEmptyBodyFunctionExpression"])); // an expression's kinds, and what an expression's place holds beside them
  for (const [name, branches] of unions) { // every branch of every union is typescript-estree's, but these
    const expected = new Set([...mapped.get(name)!, ...estree.concrete.filter((entry) => entry.union === name).map((entry) => entry.kind),
      ...(["Term", "Type", "Pattern"].includes(name) ? ["error", "missing"] : [])]);
    assert(branches.size === expected.size && [...branches].every((tag) => expected.has(tag)), name);
  }
  assert(same([...TS.DIALECT.kinds().keys(), ...TSP.TAGS.slice(0, 3)].sort(), [...unions.get("Term")!].sort())); // the
  // patterns, `TypeScript Patterns`' kinds; JSX, in typescript-estree, an extension here
  assert([...TS.EXTENSIONS].every((tag) => mapped.get("Term")!.has(tag)));
  // Each dialect's own kinds are those of its `Term` that are its schemas, the rest its family's terms, each dialect's
  // `types` or `patterns` entries; its helpers, too
  for (const [dialect, union] of [[TST, "Type"], [TSP, "Pattern"]] as const) {
    const path = `Expressions.TypeScript.${dialect.DIALECT.path.split(".")[1]}.`;
    const own = store.registered(path + "Term").branches.filter((b: any) => b.schema.name.startsWith(path)).map((b: any) => b.name);
    const concrete = estree.concrete.filter((entry) => entry.union === union).map((entry) => entry.kind);
    assert(same(own, dialect.TAGS) && own.every((tag: string) => mapped.get(union)!.has(tag) || concrete.includes(tag)), union);
    assert(dialect.HELPERS.every((helper) => store.registered(path + helper) !== null));
  }
  assert(same([...TST.DIALECT.kinds().keys()].sort(), [...TST.TAGS, "identifier", "error", "missing"].sort())); // an
  // identifier read as a type's name
  assert(same([...TSP.DIALECT.kinds().keys()].sort(), [...TSP.TAGS, "error", "missing"].sort()));
}

<a id="DIA-24"></a>
## DIA-24 · C and C++: mbse-programs' kinds each language has, in its shapes, its type names a dialect of their own and its binding targets a union of its own; literals keeping what their text encodes, names, what each kind binds, the slots that may be empty, and evaluation refusing new kinds by name

In [ ]:
{
  const CPPE = CppDialect.Expressions;
  const CPP = CPPE.DIALECT;
  const CPV = CppDialect.Evaluators;
  const cpp = (spec: unknown): any => term(spec, CPP);
  /** A union's branches, by name. */
  const branches = (name: string): string[] => store.registered(name).branches.map((b: any) => b.name);
  const schemas = (name: string): Map<string, any> => new Map(store.registered(name).branches.map((b: any) => [b.name, b.schema]));
  // Each dialect's terms, its sublanguages, a type name and binding targets (C++'s an init capture's name, an identifier;
  // none in a C expression, so invalid syntax alone), and the helpers a type name and a lambda hold, each union named
  // after mbse-programs' category
  // its type names a dialect of its own (C Types, Cpp Types: suite CTY and XTY), whose `Term` is a type name, an
  // enumerator, invalid syntax, and the directives and comment an enumeration's items, which C Preprocessor's install adds
  const DIRECTIVES = ["includedirective", "definedirective", "ifdirective", "ifdefdirective", "elifdirective", "elifdefdirective",
    "elsedirective", "otherdirective", "comment"];
  assert(same(branches("Expressions.C.Types.Term"), ["typeid", "enumerator", "error", "missing", ...DIRECTIVES])
    && same(branches("Expressions.Cpp.Types.Term"), ["typeid", "enumerator", "error", "missing", ...DIRECTIVES]));
  assert(same(branches("Expressions.C.Expressions.Pattern"), ["error", "missing"]) && same(branches("Expressions.Cpp.Expressions.Pattern"), ["identifier", "error", "missing"]));
  assert(same(branches("Expressions.C.Types.Specifier"), [
    "declspecifier", "cvqualifier", "primitivetypespecifier", "namedtypespecifier", "classspecifier", "enumspecifier",
    "placeholdertypespecifier", "typeofspecifier", "atomictypespecifier", "bitintspecifier", "standardattributespecifier",
    "alignasspecifier", "gnuattributespecifier", "declspecspecifier"]));
  assert(same(branches("Expressions.Cpp.Expressions.Specifier"), [
    "declspecifier", "cvqualifier", "primitivetypespecifier", "namedtypespecifier", "typenamespecifier", "classspecifier",
    "enumspecifier", "decltypespecifier", "placeholdertypespecifier", "standardattributespecifier", "alignasspecifier",
    "gnuattributespecifier", "declspecspecifier"]));
  assert(same(branches("Expressions.C.Types.Declarator"), ["iddeclarator", "pointerdeclarator", "arraydeclarator", "functiondeclarator",
    "parenthesizeddeclarator"]));
  assert(same(branches("Expressions.Cpp.Expressions.Declarator"), ["iddeclarator", "packdeclarator", "pointerdeclarator", "referencedeclarator",
    "arraydeclarator", "functiondeclarator", "parenthesizeddeclarator"]));
  for (const dialect of ["C.Types", "Cpp.Expressions"]) { // C's, which no expression holds, C Types'; C++'s, C++'s
    assert(same(branches(`Expressions.${dialect}.Parameter`), ["parameterdeclaration", "ellipsisparameter"]));
    assert(same(branches(`Expressions.${dialect}.AttributeSpecifier`), [
      "standardattributespecifier", "alignasspecifier", "gnuattributespecifier", "declspecspecifier"]));
  }
  assert(same(branches("Expressions.C.Expressions.Designator"), ["fielddesignator", "indexdesignator"])); // C++20's designators name fields alone
  assert(same(branches("Expressions.Cpp.Expressions.Designator"), ["fielddesignator"]));
  assert(same(branches("Expressions.C.Expressions.Initializer"), ["equalinitializer", "initializerlist"]));
  assert(same(branches("Expressions.Cpp.Expressions.Initializer"), ["equalinitializer", "parenthesizedinitializer", "initializerlist"]));
  assert(same(branches("Expressions.C.Expressions.Name"), ["identifier"])); // C's one name, held wherever mbse-programs holds a name
  assert(same(branches("Expressions.Cpp.Expressions.Name"), ["identifier", "operatorname", "conversionname", "literaloperatorname",
    "destructorname", "templateid", "qualifiedname", "decltypespecifier"]));
  assert(same(branches("Expressions.Cpp.Expressions.Capture"), ["defaultcapture", "simplecapture", "thiscapture", "initcapture"]));
  assert(same(branches("Expressions.Cpp.Expressions.Requirement"), ["simplerequirement", "typerequirement", "compoundrequirement", "nestedrequirement"]));
  assert(same(branches("Expressions.Cpp.Expressions.TemplateParameter"), ["typeparameter", "templatetemplateparameter", "parameterdeclaration"]));
  assert(same(branches("Expressions.Cpp.Expressions.ExceptionSpecification"), ["noexceptspecifier", "throwspecifier"]));
  // As the user decided ("(b)"), one schema is a branch of every union whose positions hold it
  for (const [dialect, types, helpers] of [["C.Expressions", "C.Types", "C.Types"], ["Cpp.Expressions", "Cpp.Types", "Cpp.Expressions"]]) {
    const [term_, initializer] = ["Term", "Initializer"].map((union) => schemas(`Expressions.${dialect}.${union}`));
    const type_ = schemas(`Expressions.${types}.Term`);
    const [specifier, attribute_] = ["Specifier", "AttributeSpecifier"].map((union) => schemas(`Expressions.${helpers}.${union}`));
    assert(term_!.get("typeid") === type_!.get("typeid") && term_!.get("initializerlist") === initializer!.get("initializerlist"));
    assert(type_!.get("typeid") === store.registered(`Expressions.${types}.TypeId`)); // the type name is its dialect's kind
    assert([...attribute_!].every(([name, schema]) => specifier!.get(name) === schema)); // a type name's specifiers hold them
    assert(term_!.get("error") === type_!.get("error") && type_!.get("error") === schemas(`Expressions.${dialect}.Pattern`).get("error")); // the framework's
  }
  assert(schemas("Expressions.C.Expressions.Name").get("identifier") === store.registered("Expressions.C.Expressions.Identifier"));
  assert(store.registered("Expressions.C.Expressions.FieldDesignator").properties.get("name").schema === store.registered("Expressions.C.Expressions.Identifier"));
  assert(store.registered("Expressions.C.Expressions.MemberExpression").properties.get("member").schema === store.registered("Expressions.C.Expressions.Name"));
  assert(store.registered("Expressions.Cpp.Expressions.SimpleCapture").properties.get("name").schema === store.registered("Expressions.Cpp.Expressions.Identifier"));
  // C++'s init capture holds its name as a Pattern, whose identifier is C++'s; C's Pattern is held by nothing, as C
  // expressions bind no names
  assert(store.registered("Expressions.Cpp.Expressions.InitCapture").properties.get("name").schema === store.registered("Expressions.Cpp.Expressions.Pattern"));
  assert(schemas("Expressions.Cpp.Expressions.Pattern").get("identifier") === store.registered("Expressions.Cpp.Expressions.Identifier"));
  const C_PATTERN = store.registered("Expressions.C.Expressions.Pattern");
  assert(!store.names().filter((name: string) => name.startsWith("Expressions.C.Expressions.")).some((name: string) =>
    [...(store.registered(name).properties ?? new Map()).values()].some((prop: any) => prop.schema === C_PATTERN || prop.schema.element === C_PATTERN)));
  assert(schemas("Expressions.Cpp.Expressions.Name").get("decltypespecifier") === schemas("Expressions.Cpp.Expressions.Specifier").get("decltypespecifier"));
  assert(schemas("Expressions.Cpp.Expressions.TemplateParameter").get("parameterdeclaration") === schemas("Expressions.Cpp.Expressions.Parameter").get("parameterdeclaration"));
  // Literals: C's and C++'s refine the framework's, adding what their text encodes
  for (const [dialect, encoded] of [["C.Expressions", ["prefix", "quote", "base", "separators", "suffix"]],
    ["Cpp.Expressions", ["prefix", "quote", "raw", "delimiter", "base", "separators", "suffix"]]] as [string, string[]][]) {
    const Literal = store.registered(`Expressions.${dialect}.Literal`);
    assert(Literal.refines === store.registered("Expressions.Framework.Literal") && same([...Literal.properties.keys()].slice(-encoded.length), encoded));
  }
  const encoded = cc((u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(1n)).prefix("Q").quote("`").base(3n)));
  assert(same(C.DIALECT.validate(encoded), ["a literal's prefix cannot be 'Q'", "a literal's quote cannot be '`'", "a literal's base cannot be 3"]));
  assert(same(C.DIALECT.validate(cc(C.literal(5n, "z"))), ["a literal's suffix cannot be 'z'"]) && CPP.validate(cpp(CPPE.literal(5n, "zu"))).length === 0);
  assert(same(CPP.validate(cpp(CPPE.literal(5n, "wb"))), ["a literal's suffix cannot be 'wb'"]) && CPP.validate(cpp(CPPE.literal(1.5, "bf16"))).length === 0);
  assert(same(C.DIALECT.validate(cc(C.literal(true, "u"))), ["a literal's suffix cannot be 'u'"])); // a bool has none
  assert(same(CPP.validate(cpp(CPPE.literal(2n ** 70n, "z"))), ["integer literal 1180591620717411303424 does not fit 'long'"]));
  // A character's quote is C's spelling, `'a'` an int as 97 is, and not C++'s, where `'a'` is a char
  assert(C.DIALECT.kinds().get("literal")!.spelling.includes("quote") && !CPP.kinds().get("literal")!.spelling.includes("quote"));
  const problem = cc((u: any) => u.literal((r: any) => r.text("0x").problems(["a hexadecimal literal needs digits"])));
  assert(same(C.DIALECT.validate(problem), ["a hexadecimal literal needs digits"]) && CText.ToText(problem) === "0x"); // kept as written
  // Constants: C23's and C++'s nullptr
  assert(same(C.DIALECT.validate(cc((u: any) => u.constant((r: any) => r.singleton("null")))), ["a constant's singleton must be nullptr, got 'null'"]));
  assert(same(C.DIALECT.validate(cc((u: any) => u.constant((r: any) => r))), ["a constant needs a singleton"]));
  assert(same(C.DIALECT.validate(cc((u: any) => u.constant((r: any) => r.text("nullpt").problems(["no such keyword"])))), ["no such keyword"]));
  assert(C.DIALECT.infer(cc((u: any) => u.constant((r: any) => r.singleton("nullptr")))).name() === "nullptr_t");
  // Names: an identifier, as mbse-programs checks it, wherever one is held
  assert(same(C.DIALECT.validate(cc(C.identifier("1x")), ["1x"]), ["'1x' is not an identifier"]));
  assert(C.DIALECT.validate(cc(C.identifier("été")), ["été"]).length === 0 && C.DIALECT.validate(cc(C.identifier("$a")), ["$a"]).length === 0);
  assert(same(C.DIALECT.validate(cc(C.member(C.identifier("x"), "a b")), ["x"]), ["'a b' is not an identifier"]));
  assert(same(C.DIALECT.validate(cc((u: any) => u.designatedinitializer((r: any) => r.designators([(d: any) => d.fielddesignator(
    (f: any) => f.name((i: any) => i.spelling("1a")))]).initializer((i: any) => i.equalinitializer((e: any) => e.value(C.literal(1n))))))), [
    "'1a' is not an identifier"]));
  assert(same(CPP.validate(cpp((u: any) => u.idexpression((r: any) => r.name((m: any) => m.qualifiedname((q: any) => q.qualifiers([
    (n: any) => n.identifier((i: any) => i.spelling("a-b"))]).name((n: any) => n.identifier((i: any) => i.spelling("c")))))))), ["'a-b' is not an identifier"]));
  assert(same(C.DIALECT.validate(cc((u: any) => u.castexpression((r: any) => r.type((t: any) => t.typeid((i: any) => i.specifiers([
    (s: any) => s.namedtypespecifier((n: any) => n.name((m: any) => m.identifier((i: any) => i.spelling("2t"))))]))).operand(C.literal(1n))))), ["type: '2t' is not an identifier"])); // within its type, a C Types term
  // What each kind binds: a function declarator's, a lambda's or a requires expression's parameters, and a lambda's init
  // captures, within it, and not beyond
  const [x, w] = [CPPE.identifier("x"), CPPE.identifier("w")];
  const lambda_ = (u: any) => u.lambdaexpression((r: any) => r.captures([ // [w = 1, x](int a, ...) noexcept(a + w) {}
    (c: any) => c.initcapture((i: any) => i.name((n: any) => n.identifier((j: any) => j.spelling("w"))).initializer((e: any) => e.equalinitializer((q: any) => q.value(CPPE.literal(1n))))),
    (c: any) => c.simplecapture((s: any) => s.name((n: any) => n.spelling("x")))]).declarator((d: any) => d.parameters([
    (p: any) => p.parameterdeclaration((q: any) => q.specifiers([(s: any) => s.primitivetypespecifier((k: any) => k.keyword("int"))]).declarator(
      (e: any) => e.iddeclarator((i: any) => i.name((n: any) => n.identifier((j: any) => j.spelling("a")))))),
    (p: any) => p.ellipsisparameter((e: any) => e)]).exception((e: any) => e.noexceptspecifier((n: any) => n.condition(
    CPPE.binary("+", CPPE.identifier("a"), w))))));
  assert(CPP.validate(cpp(lambda_)).length === 0 && CppDialect.Text.ToText(cpp(lambda_)) === "[w = 1, x](int a, ...) noexcept(a + w) {}");
  assert(same(CPP.validate(cpp(CPPE.binary(",", lambda_, w))), ["right: idexpression 'w' is not bound"]));
  const abstract = (u: any) => u.requiresexpression((r: any) => r.parameters([(p: any) => p.parameterdeclaration( // requires (int) { w; }
    (q: any) => q.specifiers([(s: any) => s.primitivetypespecifier((k: any) => k.keyword("int"))]))]).requirements([
    (q: any) => q.simplerequirement((s: any) => s.expression(w))]));
  assert(same(CPP.validate(cpp(abstract)), ["requirements 0 expression: idexpression 'w' is not bound"])); // an abstract parameter binds none
  // Slots that may be empty, wherever a type name's helpers nest them; the others are needed
  for (const [dialect, spec] of [
    [C.DIALECT, (u: any) => u.sizeofexpression((r: any) => r.operand((t: any) => t.typeid((i: any) => i.specifiers([
      (s: any) => s.primitivetypespecifier((k: any) => k.keyword("int"))]).declarator((d: any) => d.arraydeclarator((a: any) => a)))))], // int[]
    [C.DIALECT, (u: any) => u.conditionalexpression((r: any) => r.condition(C.literal(1n)).alternative(C.literal(2n)))], // GNU's ?:
    [CPP, (u: any) => u.foldexpression((r: any) => r.left(CPPE.literal(1n)).operator("+"))], [CPP, (u: any) => u.throwexpression((r: any) => r)],
  ] as [F.Dialect, unknown][]) assert(dialect.validate(term(spec, dialect)).length === 0);
  for (const [dialect, spec, problems] of [
    [C.DIALECT, (u: any) => u.designatedinitializer((r: any) => r), ["a designatedinitializer needs an initializer"]],
    [C.DIALECT, (u: any) => u.compoundliteralexpression((r: any) => r), ["a compoundliteralexpression needs an initializer",
      "a compoundliteralexpression needs a type"]],
    [C.DIALECT, (u: any) => u.memberexpression((r: any) => r.object(C.literal(1n))), ["a memberexpression needs an operator", "a memberexpression needs a member"]],
    [C.DIALECT, (u: any) => u.alignofexpression((r: any) => r.operand(C.literal(1n))), ["an alignofexpression needs a keyword"]],
    [C.DIALECT, (u: any) => u.genericselection((r: any) => r.controlling(C.literal(1n)).associations([(a: any) => a.type((t: any) => t.typeid((i: any) => i.specifiers([
      (s: any) => s.primitivetypespecifier((k: any) => k.keyword("int"))])))])), [
      "a genericselection needs a value 0"]], // an association, a pair of terms, labelled by its member
    [CPP, (u: any) => u.userdefinedliteral((r: any) => r), ["an userdefinedliteral needs a literal", "an userdefinedliteral needs a suffix"]], // the article by its letter
    [CPP, (u: any) => u.idexpression((r: any) => r), ["an idexpression needs a name"]],
    [CPP, (u: any) => u.namedcastexpression((r: any) => r), ["a namedcastexpression needs an operator", "a namedcastexpression needs a type",
      "a namedcastexpression needs an operand"]],
    [CPP, (u: any) => u.sizeofpackexpression((r: any) => r), ["a sizeofpackexpression needs a pack"]],
    [CPP, (u: any) => u.newexpression((r: any) => r), ["a newexpression needs a type"]],
    [CPP, (u: any) => u.namedcastexpression((r: any) => r.operator("bit_cast").type((t: any) => t.typeid((i: any) => i)).operand(CPPE.literal(1n))), []],
  ] as [F.Dialect, unknown, string[]][]) assert(same(dialect.validate(term(spec, dialect)), problems), problems.join("; "));
  assert(same(CPP.validate(cpp((u: any) => u.namedcastexpression((r: any) => r.operator("bit_cast").type((t: any) => t.typeid((i: any) => i)).operand(CPPE.literal(1n)))), [], true), [
    "'bit_cast' is not a core operation"]));
  // C++ writes a name only as an idexpression, holding its Name, as C++'s grammar has it (C keeps the framework's
  // variable): its writers write one, a lexical reference whose name is its identifier's, and C++'s union has no variable
  assert(!CPP.kinds().has("variable") && (CPP.kinds().get("idexpression") as F.Kind).role === F.REFERENCE && C.DIALECT.kinds().has("variable"));
  assert(F.form(cpp(x)).kind === "idexpression" && Repr.repr(F.form(cpp(x)).attributes) === "{'name': Part(kind='identifier', attributes={'spelling': 'x'}, size=0)}"
    && F.form(cpp(x)).arguments.length === 0 && F.nameOf(cpp(x)) === "x");
  assert(same(CPP.validate(cpp(x)), ["idexpression 'x' is not bound"]) && same([...SY.free(cpp(CPPE.binary("+", x, w)), CPP)].sort(), ["w", "x"]));
  raises(Error, () => cpp((u: any) => u.variable((r: any) => r.name("x"))), "'variable' is not a member of a union value");
  const cout = (u: any) => u.idexpression((r: any) => r.name((n: any) => n.qualifiedname((q: any) => q.qualifiers([ // std::cout
    (m: any) => m.identifier((i: any) => i.spelling("std"))]).name((m: any) => m.identifier((i: any) => i.spelling("cout"))))));
  const fW = (u: any) => u.idexpression((r: any) => r.name((n: any) => n.templateid((t: any) => t.name( // f<w>
    (m: any) => m.identifier((i: any) => i.spelling("f"))).arguments([w]))));
  assert(F.nameOf(cpp(cout)) === null && CPP.validate(cpp(cout)).length === 0 && SY.free(cpp(cout), CPP).size === 0); // the scope's alone
  assert(same(CPP.validate(cpp(fW)), ["name argument 0: idexpression 'w' is not bound"]) && same([...SY.free(cpp(fW), CPP)], ["w"])); // its terms
  // An init capture's name holds C++'s Pattern: an identifier, or invalid syntax, and no other kind
  const capture = (name: unknown) => (u: any) => u.lambdaexpression((r: any) => r.captures([(c: any) => c.initcapture( // [name = 1] {}
    (i: any) => i.name(name).initializer((e: any) => e.equalinitializer((q: any) => q.value(CPPE.literal(1n)))))]));
  const invalidCapture = cpp(capture((n: any) => n.error((e: any) => e.text("1w"))));
  assert(same(CPP.validate(invalidCapture), ["invalid syntax '1w'"]) && CppDialect.Text.ToText(invalidCapture) === "[1w = 1] {}");
  assert(C.boundNames(F.node("lambdaexpression", F.kindOf(invalidCapture)[1])).size === 0); // invalid syntax binds none
  raises(Error, () => cpp(capture(w)), "'idexpression' is not a member of a union value"); // a term, not a pattern
  // A dialect's own kinds hold its terms alone: a C term is not a C++ term, nor a C++ term a C term
  assert(same(CPP.validate(term(C.binary("+", 1n, 2n), C.DIALECT)), ["a C binaryexpression is not a Cpp term"]));
  assert(same(C.DIALECT.validate(cpp(x), ["x"]), ["a Cpp idexpression is not a C term"]));
  // Evaluation stays shallow: C++'s by C's rules, for the kinds C evaluated, a character a char; its own kinds refused by name
  assert(CPV.OfAny(cpp(CPPE.binary("+", CPPE.literal(1n, "u"), 1n))) === 2n && CPV.OfAny(cpp(CPPE.member(x, "a")), { x: { a: 1n } }) === 1n);
  assert(CPV.OfAny(cpp((u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(97n)).quote("'")))) === 97n);
  assert(CPP.infer(cpp((u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(97n)).quote("'")))).name() === "char");
  for (const [spec, message] of [
    [lambda_, "Cpp lambdaexpression is not evaluated"],
    [(u: any) => u.memberexpression((r: any) => r.object(x).operator(".").member((m: any) => m.operatorname((o: any) => o.operator("+")))),
      "Cpp memberexpression of a member that is not an identifier is not evaluated"],
    [(u: any) => u.memberexpression((r: any) => r.object(x).operator(".").template_keyword(true).member((m: any) => m.identifier((i: any) => i.spelling("a")))),
      "Cpp memberexpression of a member that is not an identifier is not evaluated"],
  ] as [unknown, string][]) raises(NotImplementedError, () => CPV.OfAny(cpp(spec), { x: { a: 1n } }), message);
  assert(same([...CPV.EVALUATED].sort(), [...CV.EVALUATED].filter((tag) => tag !== "variable").concat(["idexpression"]).sort()));
  for (const held of [cout, (u: any) => u.idexpression((r: any) => r.name((n: any) => n.identifier((i: any) => i)))]) { // an identifier with no spelling
    raises(NotImplementedError, () => CPV.OfAny(cpp(held)), "Cpp idexpression of a name that is not an identifier is not evaluated");
  }
  raises(NotImplementedError, () => CPV.OfAny(cpp(CPPE.call(cout, 1n))), "Cpp callexpression of a callee that is not a function's name is not evaluated");
  // C++'s types: its own (`bool`, the exact-width typedefs, `std::float16_t` to `std::float128_t`, `std::bfloat16_t`), then
  // its holder's ABI's, as C's are; a C++ literal typed by C++'s names
  assert(CPP.infer(cpp(CPPE.literal(1.5, "f16"))).name() === "std::float16_t" && CPP.infer(cpp(CPPE.literal(1.5, "BF16"))).name() === "std::bfloat16_t");
  assert(CPV.OfAny(cpp(CPPE.binary("+", CPPE.literal(1.5, "bf16"), 1n))) === "2.5" && CPV.OfAny(cpp(CPPE.binary("/", CPPE.literal(1.0, "bf16"), 3n))) === "0.334");
  assert(CPV.OfAny(cpp(CPPE.literal(0.1, "f16"))) === 0.0999755859375 && CPP.infer(cpp(CPPE.cast(named_type("int32_t"), 1n))).name() === "int32_t");
  assert(CPV.OfAny(cpp(CPPE.cast(keywords("char"), 200n))) === -56n && CPV.OfAny(term(CPPE.cast(keywords("char"), 200n), CPP, AAPCS64_)) === 200n);
  const CPT = new CD.Types(term(0n, CPP, AAPCS64_), "Cpp.Expressions");
  assert(CPT.named("std::float128_t").equals(CPT.named("long double")) && equal(CPT.named("std::bfloat16_t").format, [2n, 8, 127])); // binary128, bfloat16
  assert(!CPT.named("std::bfloat16_t").equals(CPT.named("std::float16_t")) && !CPT.named("std::float16_t").equals(CPT.named("std::bfloat16_t"))); // 16 bits each
  raises(ValueError, () => CPT.named("_Float16"), "'_Float16' is not a type Cpp or AAPCS64 defines"); // C's own, not C++'s
  raises(ValueError, () => CPV.OfAny(term(CPPE.literal(5n), CPP, null)), "'int' is not a type Cpp defines, and its expression names no ABI");
  // Inference: sizeof and alignof give a size_t, noexcept a bool, a cast to a type that is not arithmetic what is not known
  assert(C.DIALECT.infer(cc((u: any) => u.sizeofexpression((r: any) => r.operand(C.literal(1n))))).name() === "size_t");
  assert(CPP.infer(cpp((u: any) => u.noexceptexpression((r: any) => r.operand(CPPE.literal(1n))))).name() === "bool");
  assert(CPP.infer(cpp((u: any) => u.sizeofpackexpression((r: any) => r.pack((i: any) => i.spelling("p"))))).name() === "size_t");
  assert(C.DIALECT.infer(cc((u: any) => u.castexpression((r: any) => r.type((t: any) => t.typeid((i: any) => i.specifiers([
    (s: any) => s.primitivetypespecifier((k: any) => k.keyword("void"))]))).operand(C.literal(1n))))) === D.Anything);
  assert(C.DIALECT.infer(cc((u: any) => u.castexpression((r: any) => r.type((t: any) => t.typeid((i: any) => i.specifiers([
    (s: any) => s.classspecifier((k: any) => k.key("struct").name((n: any) => n.identifier((i: any) => i.spelling("S"))))]))).operand(C.literal(1n))))) === D.Anything);
  for (const parts of [(i: any) => i, (i: any) => i.specifiers([(s: any) => s.namedtypespecifier((n: any) => n)]),
    (i: any) => i.specifiers([(s: any) => s.primitivetypespecifier((k: any) => k)])]) { // no specifier, a named type or a keyword with no name
    assert(C.DIALECT.infer(cc((u: any) => u.castexpression((r: any) => r.type((t: any) => t.typeid(parts)).operand(C.literal(1n))))) === D.Anything);
  }
  const invalid = cc((u: any) => u.castexpression((r: any) => r.type((t: any) => t.error((e: any) => e.text("int int"))).operand(C.literal(1n))));
  assert(same(C.DIALECT.validate(invalid), ["type: invalid syntax 'int int'"])); // invalid syntax in a type name, a C Types term
  raises(ValueError, () => CV.OfAny(invalid), "C Types error is not evaluated"); // refused by its owner's name, as every dialect refuses it
  raises(NotImplementedError, () => CPV.OfAny(cpp((u: any) => u.memberexpression((r: any) => r.object(x).operator(".").member((m: any) => m.identifier((i: any) => i)))), { x: { a: 1n } }),
    "Cpp memberexpression of a member that is not an identifier is not evaluated"); // no spelling
  const unnamed = (u: any) => u.requiresexpression((r: any) => r.parameters([(p: any) => p.parameterdeclaration( // requires (int <no name>) { w; }
    (q: any) => q.declarator((d: any) => d.iddeclarator((i: any) => i.attributes([(a: any) => a.alignasspecifier((s: any) => s.keyword("alignas").operand(CPPE.literal(1n)))]))))]).requirements([
    (q: any) => q.simplerequirement((s: any) => s.expression(w))]));
  assert(same(CPP.validate(cpp(unnamed)), ["requirements 0 expression: idexpression 'w' is not bound"]));
  // The writers: C's and C++'s write a name as an identifier, and a cast's type is a spec of its family's Types' `Term`
  assert(Repr.repr(F.form(cpp(CPPE.member(x, "a"))).attributes.get("member")) === "Part(kind='identifier', attributes={'spelling': 'a'}, size=0)");
  assert(Repr.repr(F.form(cc(C.member(C.identifier("x"), "a"))).attributes.get("member")) === "Part(kind='identifier', attributes={'spelling': 'a'}, size=0)");
  assert(CppDialect.Text.ToText(cpp(CPPE.cast((t: any) => t.typeid((i: any) => i.specifiers([
    (s: any) => s.primitivetypespecifier((p: any) => p.keyword("unsigned")),
    (s: any) => s.namedtypespecifier((n: any) => n.name((m: any) => m.identifier((d: any) => d.spelling("size_t"))))])), x))) === "(unsigned size_t)x");
  assert(F.form(cpp(CPPE.call(CPPE.identifier("f"), 1n))).kind === "callexpression"); // a callee that is a term
  // An empty initializer list is held and read back present (mbse-schemas 0.26.9): `(int){}`, printed as written
  const emptyList = cc((u: any) => u.compoundliteralexpression((r: any) => r.type((t: any) => t.typeid((i: any) => i.specifiers([
    (s: any) => s.primitivetypespecifier((k: any) => k.keyword("int"))]))).initializer((li: any) => li)));
  assert(C.DIALECT.validate(emptyList).length === 0 && roundTrips(emptyList) && CText.ToText(emptyList) === "(int){}");
  assert(Repr.repr(F.form(emptyList).attributes.get("initializer")) === "Part(kind='initializerlist', attributes={}, size=0)");
}

<a id="DIA-25"></a>
## DIA-25 · C and C++ printed as they write them: every kind, its names, type names, lambdas and attributes, by precedence, a literal's default spelling with what it encodes

In [ ]:
{
  const CPPE = CppDialect.Expressions;
    /** A C term's source. */
  const cText = (spec: unknown): string => CText.ToText(term(spec, C.DIALECT));
    /** A C++ term's source. */
  const cppText = (spec: unknown): string => CppDialect.Text.ToText(term(spec, CPPE.DIALECT));
    // Specs of terms and their helpers, written through the builders, each in any union whose branches it names
  let [x, y, z] = ["x", "y", "z"].map((n) => (e: any) => e.variable((v: any) => v.name(n)));
  const [one, two] = [1n, 2n].map((n) => (e: any) => e.literal((r: any) => r.value((v: any) => v.as_scalar(n))));
  const true_ = (e: any) => e.literal((r: any) => r.value((v: any) => v.as_scalar(true)));
  const keyword = (word: string) => (s: any) => s.primitivetypespecifier((p: any) => p.keyword(word));
  const [int_, char] = [keyword("int"), keyword("char")];
  const const_ = (s: any) => s.cvqualifier((p: any) => p.keyword("const"));
  const intType = (t: any) => t.typeid((i: any) => i.specifiers([int_]));
  const named = (n: string) => (m: any) => m.identifier((i: any) => i.spelling(n)); // a name that is an identifier
  const [T_, U_] = ["T", "U"].map((n) => (t: any) => t.typeid((i: any) => i.specifiers([(s: any) => s.namedtypespecifier((m: any) => m.name(named(n)))])));
  const parameter = (specifier: unknown, name: unknown) => (p: any) => p.parameterdeclaration((d: any) => d.specifiers([specifier]).declarator(
    (e: any) => e.iddeclarator((i: any) => i.name(name))));
  const attribute = (name: string) => (a: any) => a.standardattributespecifier((s: any) => s.attributes([(t: any) => t.name(name)]));
  // C: literals in their default spellings, with what their text encodes
  for (const [spec, text] of [
      [(u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(255n)).base(16n).suffix("u")), "0xffu"],
      [(u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(255n)).base(8n)), "0377"],
      [(u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(0n)).base(8n)), "0"],
      [(u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(5n)).base(2n)), "0b101"],
      [(u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(255n)).base(10n)), "255"],
      [(u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(-1n)).base(16n)), "-0x1"],
      [(u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(97n)).quote("'")), "'a'"],
      [(u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(39n)).quote("'").prefix("L")), "L'\\''"],
      [(u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(10n)).quote("'")), "'\\n'"],
      [(u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar('x"\0')).prefix("u8")), 'u8"x\\"\\0"'],
      [(u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(1n)).text("0b1")), "0b1"],
      [(u: any) => u.constant((r: any) => r.singleton("nullptr")), "nullptr"],
      [(u: any) => u.constant((r: any) => r.singleton("nullptr").text("nullptr")), "nullptr"],
      [(u: any) => u.concatenatedstring((r: any) => r.parts([C.literal("a"), (p: any) => p.variable((v: any) => v.name("PRId64"))])), '"a" PRId64'],
      // precedence: assignments, the comma, the conditional, unary operators' spacing, postfix, sizeof
      [C.DIALECT.spec((u: any) => u.assignmentexpression((r: any) => r.left(x).operator("=").right(
          (e: any) => e.assignmentexpression((a: any) => a.left(y).operator("*=").right(one))))), "x = y *= 1"],
      [(u: any) => u.assignmentexpression((r: any) => r.left((e: any) => e.assignmentexpression((a: any) => a.left(x).operator("=").right(y)))
          .operator("=").right(one)), "(x = y) = 1"],
      [(u: any) => u.binaryexpression((r: any) => r.left((e: any) => e.assignmentexpression((a: any) => a.left(x).operator("=").right(one)))
          .operator("+").right(y)), "(x = 1) + y"],
      [C.binary(",", x, y), "x, y"], [C.call("f", C.binary(",", x, y)), "f((x, y))"],
      [(u: any) => u.conditionalexpression((r: any) => r.condition(x).alternative(y)), "x ?: y"],
      [C.conditional(x, C.binary(",", y, z), 1n), "x ? y, z : 1"],
      [C.unary("&", C.unary("*", x)), "&*x"], [C.unary("*", C.unary("*", x)), "* *x"], [C.unary("&", C.unary("&", x)), "& &x"],
      [C.unary("-", C.unary("--", x)), "- --x"], [C.unary("++", x), "++x"], [C.unary("!", C.unary("-", x)), "!-x"],
      [(u: any) => u.postfixexpression((r: any) => r.operand(C.binary("+", x, 1n)).operator("++")), "(x + 1)++"],
      [(u: any) => u.sizeofexpression((r: any) => r.operand(x)), "sizeof x"],
      [(u: any) => u.sizeofexpression((r: any) => r.operand((e: any) => e.parenthesizedexpression((p: any) => p.expression(x)))), "sizeof(x)"],
      [(u: any) => u.sizeofexpression((r: any) => r.operand(C.binary("+", x, 1n))), "sizeof(x + 1)"],
      [(u: any) => u.alignofexpression((r: any) => r.keyword("_Alignof").operand(intType)), "_Alignof(int)"],
      [C.unary("-", (e: any) => e.sizeofexpression((r: any) => r.operand(x))), "-sizeof x"],
      // type names: specifiers and abstract declarators
      [(u: any) => u.castexpression((r: any) => r.type((t: any) => t.typeid((i: any) => i.specifiers([char]).declarator(
          (d: any) => d.pointerdeclarator((p: any) => p)))).operand(x)), "(char*)x"],
      [(u: any) => u.castexpression((r: any) => r.type((t: any) => t.typeid((i: any) => i.specifiers([const_, char]).declarator(
          (d: any) => d.pointerdeclarator((p: any) => p.qualifiers([(q: any) => q.keyword("const")]).declarator(
              (e: any) => e.pointerdeclarator((q: any) => q)))))).operand(x)), "(const char*const *)x"],
      [(u: any) => u.castexpression((r: any) => r.type((t: any) => t.typeid((i: any) => i.specifiers([int_]).declarator(
          (d: any) => d.functiondeclarator((f: any) => f.declarator((e: any) => e.pointerdeclarator((p: any) => p)).parameters([
              parameter(int_, named("n")), (p: any) => p.parameterdeclaration((q: any) => q.specifiers([int_]).declarator(
                  (e: any) => e.arraydeclarator((a: any) => a.declarator((g: any) => g.iddeclarator((i: any) => i.name((n: any) => n.identifier((j: any) => j.spelling("a"))))).size(
                      (s: any) => s.variable((v: any) => v.name("n"))))))]))))).operand(x)), "(int(*)(int n, int a[n]))x"],
      [(u: any) => u.castexpression((r: any) => r.type((t: any) => t.typeid((i: any) => i.specifiers([int_]).declarator(
          (d: any) => d.functiondeclarator((f: any) => f.declarator((e: any) => e.parenthesizeddeclarator((p: any) => p.declarator(
              (g: any) => g.pointerdeclarator((q: any) => q)))).parameters([(p: any) => p.parameterdeclaration((q: any) => q.attributes([
                  attribute("maybe_unused")]).specifiers([int_])), (p: any) => p.ellipsisparameter((e: any) => e)]).attributes([
              attribute("a")]))))).operand(x)), "(int(*)([[maybe_unused]] int, ...) [[a]])x"],
      [(u: any) => u.sizeofexpression((r: any) => r.operand((t: any) => t.typeid((i: any) => i.specifiers([int_]).declarator(
          (d: any) => d.arraydeclarator((a: any) => a.declarator((e: any) => e.pointerdeclarator((p: any) => p.attributes([attribute("a")])))
                                      .size(two).attributes([attribute("b")])))))), "sizeof(int(*[[a]])[2] [[b]])"],
      [(u: any) => u.sizeofexpression((r: any) => r.operand((t: any) => t.typeid((i: any) => i.specifiers([int_]).declarator(
          (d: any) => d.arraydeclarator((a: any) => a.static(true).qualifiers([(q: any) => q.keyword("const")]).size(two)))))),
       "sizeof(int[static const 2])"],
      [(u: any) => u.sizeofexpression((r: any) => r.operand((t: any) => t.typeid((i: any) => i.specifiers([int_]).declarator(
          (d: any) => d.arraydeclarator((a: any) => a.star(true).static(true)))))), "sizeof(int[static *])"],
      [(u: any) => u.sizeofexpression((r: any) => r.operand((t: any) => t.typeid((i: any) => i.specifiers([int_]).declarator(
          (d: any) => d.functiondeclarator((f: any) => f.declarator((e: any) => e.iddeclarator((g: any) => g.name((n: any) => n.identifier((j: any) => j.spelling("f"))).attributes([attribute("c")])))))))),
       "sizeof(int f [[c]]())"],
      [(u: any) => u.sizeofexpression((r: any) => r.operand((t: any) => t.typeid((i: any) => i.declarator((d: any) => d.pointerdeclarator((p: any) => p))))),
       "sizeof(*)"],  // a declarator without specifiers
      [(u: any) => u.compoundliteralexpression((r: any) => r.type((t: any) => t.typeid((i: any) => i.specifiers([
          (s: any) => s.classspecifier((c: any) => c.key("struct").attributes([attribute("deprecated")]).name((n: any) => n.identifier((j: any) => j.spelling("S"))).body(true))]))).initializer(
          (li: any) => li.items([one]).trailing_comma(true))), "(struct [[deprecated]] S {}){1,}"],
      [(u: any) => u.compoundliteralexpression((r: any) => r.type((t: any) => t.typeid((i: any) => i.specifiers([
          (s: any) => s.declspecifier((d: any) => d.keyword("static")), (s: any) => s.classspecifier((c: any) => c.key("union").body(true))]))).initializer(
          (li: any) => li)), "(static union {}){}"],
      [(u: any) => u.sizeofexpression((r: any) => r.operand((t: any) => t.typeid((i: any) => i.specifiers([
          (s: any) => s.enumspecifier((c: any) => c.key("enum").attributes([attribute("a")]).name((n: any) => n.identifier((j: any) => j.spelling("E"))).base(intType).body(true))])))),
       "sizeof(enum [[a]] E : int {})"],
      [(u: any) => u.sizeofexpression((r: any) => r.operand((t: any) => t.typeid((i: any) => i.specifiers([
          (s: any) => s.classspecifier((c: any) => c.key("struct").name((n: any) => n.identifier((j: any) => j.spelling("S")))), (s: any) => s.enumspecifier((c: any) => c.key("enum"))])))),
       "sizeof(struct S enum)"],  // elaborated, without a body
      [(u: any) => u.sizeofexpression((r: any) => r.operand((t: any) => t.typeid((i: any) => i.specifiers([
          (s: any) => s.typeofspecifier((c: any) => c.keyword("typeof_unqual").operand(intType)), (s: any) => s.typeofspecifier((c: any) => c.keyword("typeof").operand(x)),
          (s: any) => s.atomictypespecifier((c: any) => c.type(intType)), (s: any) => s.bitintspecifier((c: any) => c.width(two)),
          (s: any) => s.placeholdertypespecifier((c: any) => c), (s: any) => s.namedtypespecifier((c: any) => c.name((n: any) => n.identifier((j: any) => j.spelling("size_t"))))])))),
       "sizeof(typeof_unqual(int) typeof(x) _Atomic(int) _BitInt(2) auto size_t)"],
      [(u: any) => u.sizeofexpression((r: any) => r.operand((t: any) => t.typeid((i: any) => i.specifiers([
          (s: any) => s.standardattributespecifier((a: any) => a.attributes([(t2: any) => t2.namespace("gnu").name("cold"),
                                                                         (t2: any) => t2.name("deprecated").arguments([C.literal("x")])])),
          (s: any) => s.alignasspecifier((a: any) => a.keyword("_Alignas").operand(intType)), (s: any) => s.alignasspecifier((a: any) => a.keyword("alignas").operand(two)),
          (s: any) => s.gnuattributespecifier((a: any) => a.keyword("__attribute__").attributes([(t2: any) => t2.name("packed"), (t2: any) => t2.name("aligned").arguments([two])])),
          (s: any) => s.declspecspecifier((a: any) => a.attributes([(t2: any) => t2.name("dllexport"), (t2: any) => t2.name("noinline")])), int_])))),
       'sizeof([[gnu::cold, deprecated("x")]] _Alignas(int) alignas(2) __attribute__((packed, aligned(2))) __declspec(dllexport noinline) int)'],
      // selections, designators, compound literals and GNU's extensions
      [(u: any) => u.genericselection((r: any) => r.controlling(intType).associations([(a: any) => a.value(two)])), "_Generic(int, default: 2)"],
      [(u: any) => u.initializerlist((r: any) => r.items([(i: any) => i.designatedinitializer((d: any) => d.designators([
          (g: any) => g.indexdesignator((e: any) => e.index(one).last(two)), (g: any) => g.fielddesignator((f: any) => f.name((j: any) => j.spelling("a")))]).initializer(
          (n: any) => n.initializerlist((li: any) => li.items([one, two]).trailing_comma(true)))),
          (i: any) => i.designatedinitializer((d: any) => d.designators([(g: any) => g.indexdesignator((e: any) => e.index(x))]).initializer(
              (n: any) => n.equalinitializer((e: any) => e.value(C.binary(",", x, y)))))])), "{[1 ... 2].a{1, 2,}, [x] = (x, y)}"],
      [(u: any) => u.extensionexpression((r: any) => r.operand(C.binary("+", x, 1n))), "__extension__ (x + 1)"],
  ] as [unknown, string][]) assert(cText(spec) === text, `${text}: ${cText(spec)}`);
  assert(C.DIALECT.validate(term((u: any) => u.castexpression((r: any) => r.type((t: any) => t.typeid((i: any) => i.specifiers([int_]).declarator(
      (d: any) => d.functiondeclarator((f: any) => f.declarator((e: any) => e.pointerdeclarator((p: any) => p)).parameters([
          parameter(int_, named("n")), (p: any) => p.parameterdeclaration((q: any) => q.specifiers([int_]).declarator(
              (e: any) => e.arraydeclarator((a: any) => a.declarator((g: any) => g.iddeclarator((i: any) => i.name((n: any) => n.identifier((j: any) => j.spelling("a"))))).size(
                  (s: any) => s.variable((v: any) => v.name("n"))))))]))))).operand(x)), C.DIALECT), ["x"]).length === 0); // n, its parameter's
  // C++: its own kinds, its names, lambdas, requires expressions and its declarators; a name used as an expression is an
  // idexpression, holding its Name
  const name = (spelling: string) => (u: any) => u.idexpression((r: any) => r.name(named(spelling)));
  [x, y, z] = ["x", "y", "z"].map(name);
  const qualified = (...parts: string[]) => (m: any) => m.qualifiedname((q: any) => q.qualifiers(parts.slice(0, -1).map(named)).name(named(parts[parts.length - 1] as string)));
  const template = (spelling: string, ...args: unknown[]) => (m: any) => m.templateid((t: any) => t.name(named(spelling)).arguments(args));
  const typename = (t: any) => t.typeparameter((p: any) => p.key("typename").name((j: any) => j.spelling("T")));
  for (const [spec, text] of [
      [(u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar("a")).raw(true).delimiter("d").prefix("u8")), 'u8R"d(a)d"'],
      [(u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(97n)).quote("'")), "'a'"],
      [(u: any) => u.userdefinedliteral((r: any) => r.literal((li: any) => li.value((v: any) => v.as_scalar("s"))).suffix("_s")), '"s"_s'],
      [(u: any) => u.userdefinedliteral((r: any) => r.literal((li: any) => li.value((v: any) => v.as_scalar(1.5)).text("1.50")).suffix("_m")), "1.50_m"],
      // names
      [name("x"), "x"], [(u: any) => u.idexpression((r: any) => r.name(qualified("std", "cout"))), "std::cout"],
      [(u: any) => u.idexpression((r: any) => r.name((m: any) => m.qualifiedname((q: any) => q.global_scope(true).name(named("x"))))), "::x"],
      [(u: any) => u.idexpression((r: any) => r.name((m: any) => m.operatorname((o: any) => o.operator("+")))), "operator+"],
      [(u: any) => u.idexpression((r: any) => r.name((m: any) => m.operatorname((o: any) => o.operator("new[]")))), "operator new[]"],
      [(u: any) => u.idexpression((r: any) => r.name((m: any) => m.conversionname((o: any) => o.type(intType)))), "operator int"],
      [(u: any) => u.idexpression((r: any) => r.name((m: any) => m.literaloperatorname((o: any) => o.suffix("_km")))), 'operator""_km'],
      [(u: any) => u.memberexpression((r: any) => r.object(x).operator("->").member((m: any) => m.destructorname((d: any) => d.type(named("T"))))), "x->~T"],
      [(u: any) => u.idexpression((r: any) => r.name(template("f", intType, one, CPPE.binary(">", x, y), CPPE.binary(">>", x, one), CPPE.binary("<", x, y)))),
       "f<int, 1, (x > y), x >> 1, x < y>"],  // an argument holding `>`, parenthesized where it would end the list
      [(u: any) => u.memberexpression((r: any) => r.object(x).operator(".").template_keyword(true).member(template("f", intType))), "x.template f<int>"],
      [(u: any) => u.idexpression((r: any) => r.name((m: any) => m.qualifiedname((q: any) => q.qualifiers([
          (n: any) => n.decltypespecifier((d: any) => d.expression(x)), (n: any) => n.templateid((t: any) => t.template_keyword(true).name(named("V")).arguments([T_]))]).name(
          named("type"))))), "decltype(x)::template V<T>::type"],
      [(u: any) => u.thisexpression((r: any) => r), "this"],
      // lambdas: their bodies of statements written `{}`
      [(u: any) => u.lambdaexpression((r: any) => r), "[] {}"],
      [(u: any) => u.lambdaexpression((r: any) => r.captures([(c: any) => c.defaultcapture((d: any) => d.mode("&")), (c: any) => c.thiscapture((t: any) => t),
                                                          (c: any) => c.thiscapture((t: any) => t.copy(true))]).declarator((d: any) => d)), "[&, this, *this]() {}"],  // an empty declarator, held: its parameter list
      [(u: any) => u.lambdaexpression((r: any) => r.captures([
          (c: any) => c.simplecapture((s: any) => s.by_reference(true).name((j: any) => j.spelling("x")).pack(true)), (c: any) => c.simplecapture((s: any) => s.name((j: any) => j.spelling("y"))),
          (c: any) => c.initcapture((s: any) => s.pack(true).name(named("z")).initializer((i: any) => i.equalinitializer((e: any) => e.value(one)))),
          (c: any) => c.initcapture((s: any) => s.by_reference(true).name(named("w")).initializer((i: any) => i.initializerlist((li: any) => li.items([x]))))]).declarator(
          (d: any) => d.parameters([(p: any) => p.parameterdeclaration((q: any) => q.specifiers([(s: any) => s.placeholdertypespecifier((h: any) => h)]).declarator(
              (e: any) => e.packdeclarator((k: any) => k.declarator((g: any) => g.iddeclarator((i: any) => i.name(named("a")))))))]).specifiers([
              (s: any) => s.declspecifier((k: any) => k.keyword("mutable")), (s: any) => s.declspecifier((k: any) => k.keyword("constexpr"))]).exception(
              (e: any) => e.noexceptspecifier((n: any) => n)).attributes([attribute("a")]).trailing_return(intType).requires(true_))),
       "[&x..., y, ...z = 1, &w{x}](auto... a) mutable constexpr noexcept [[a]] -> int requires true {}"],
      [(u: any) => u.lambdaexpression((r: any) => r.template_parameters([typename, (t: any) => t.typeparameter((p: any) => p.key("class").pack(true).name((j: any) => j.spelling("Us"))),
          (t: any) => t.typeparameter((p: any) => p.constraint(qualified("std", "integral")).name((j: any) => j.spelling("V")).default(intType)),
          (t: any) => t.typeparameter((p: any) => p),
          (t: any) => t.templatetemplateparameter((p: any) => p.parameters([(q: any) => q.typeparameter((h: any) => h.key("typename"))]).requires(true_).key("class").name((j: any) => j.spelling("TT"))
                                              .default(qualified("std", "vector"))),
          (t: any) => t.parameterdeclaration((q: any) => q.specifiers([int_]).declarator((e: any) => e.iddeclarator((i: any) => i.name(named("N")))).default(one))])
          .template_requires(CPPE.binary("&&", CPPE.binary("||", x, y), z)).attributes([attribute("nodiscard")]).declarator(
              (d: any) => d.parameters([parameter((s: any) => s.namedtypespecifier((n: any) => n.name(named("T"))), named("t"))]))),
       "[]<typename T, class... Us, std::integral V = int, typename, template <typename> requires true class TT = std::vector, int N = 1>"
       + " requires (x || y) && z [[nodiscard]] (T t) {}"],
      [(u: any) => u.lambdaexpression((r: any) => r.template_parameters([(t: any) => t.templatetemplateparameter(
          (p: any) => p.key("typename").pack(true).name((j: any) => j.spelling("TT")))])), "[]<template <> typename... TT> {}"],
      [(u: any) => u.lambdaexpression((r: any) => r.declarator((d: any) => d.parameters([(p: any) => p.parameterdeclaration((q: any) => q.specifiers([
        (s: any) => s.placeholdertypespecifier((h: any) => h)]).declarator((e: any) => e.packdeclarator((k: any) => k)))]))), "[](auto...) {}"],
      // folds and requires expressions
      [(u: any) => u.foldexpression((r: any) => r.operator("+").right(x)), "(... + x)"],
      [(u: any) => u.foldexpression((r: any) => r.left(x).operator("&&").right(CPPE.binary("+", y, one))), "(x && ... && (y + 1))"],
      [(u: any) => u.requiresexpression((r: any) => r), "requires {}"],
      [(u: any) => u.requiresexpression((r: any) => r.parameters([parameter((s: any) => s.namedtypespecifier((n: any) => n.name(named("T"))), named("t"))]).requirements([
          (q: any) => q.simplerequirement((s: any) => s.expression(CPPE.binary("+", CPPE.identifier("t"), 1n))),
          (q: any) => q.typerequirement((s: any) => s.name(qualified("T", "type"))),
          (q: any) => q.compoundrequirement((s: any) => s.expression(CPPE.identifier("t")).noexcept(true).return_type((m: any) => m.templateid(
              (t: any) => t.name(qualified("std", "same_as")).arguments([intType])))),
          (q: any) => q.compoundrequirement((s: any) => s.expression(CPPE.identifier("t"))),
          (q: any) => q.nestedrequirement((s: any) => s.constraint(CPPE.binary("||", CPPE.binary("&&", true_, x), y)))])),
       "requires (T t) { t + 1; typename T::type; { t } noexcept -> std::same_as<int>; { t }; requires true && x || y; }"],
      // casts, typeid, sizeof..., alignof, noexcept, new and delete, throw, co_await and co_yield, packs
      [(u: any) => u.functionalcastexpression((r: any) => r.type((s: any) => s.namedtypespecifier((n: any) => n.name(named("T")))).initializer(
          (i: any) => i.initializerlist((li: any) => li.items([one])))), "T{1}"],
      [(u: any) => u.namedcastexpression((r: any) => r.operator("reinterpret_cast").type((t: any) => t.typeid((i: any) => i.specifiers([char]).declarator(
          (d: any) => d.pointerdeclarator((p: any) => p)))).operand(CPPE.binary(",", x, y))), "reinterpret_cast<char*>(x, y)"],  // its operand an expression, a comma among them
      [(u: any) => u.typeidexpression((r: any) => r.operand(intType)), "typeid(int)"],
      [(u: any) => u.sizeofpackexpression((r: any) => r.pack((j: any) => j.spelling("Ts"))), "sizeof...(Ts)"],
      [(u: any) => u.alignofexpression((r: any) => r.keyword("alignof").operand(x)), "alignof(x)"],
      [(u: any) => u.newexpression((r: any) => r.global_scope(true).placement([x, one]).parenthesized_type(true).type((t: any) => t.typeid((i: any) => i.specifiers([int_]).declarator(
          (d: any) => d.arraydeclarator((a: any) => a.size(two)))))), "::new (x, 1) (int[2])"],
      [(u: any) => u.newexpression((r: any) => r.type(T_).initializer((i: any) => i.parenthesizedinitializer((p: any) => p.arguments([one, two])))), "new T(1, 2)"],
      [(u: any) => u.newexpression((r: any) => r.type(intType)), "new int"],
      [(u: any) => u.deleteexpression((r: any) => r.global_scope(true).array(true).operand(x)), "::delete[] x"],
      [(u: any) => u.deleteexpression((r: any) => r.operand(x)), "delete x"],
      [(u: any) => u.throwexpression((r: any) => r.operand(CPPE.binary("+", x, 1n))), "throw x + 1"],
      [(u: any) => u.yieldexpression((r: any) => r.operand((e: any) => e.assignmentexpression((a: any) => a.left(x).operator("=").right(y)))), "co_yield x = y"],
      [CPPE.binary("+", (e: any) => e.awaitexpression((a: any) => a.operand(x)), (e: any) => e.throwexpression((t: any) => t)), "co_await x + (throw)"],
      [(u: any) => u.packexpansion((r: any) => r.pattern(CPPE.binary("+", x, y))), "(x + y)..."],
      [(u: any) => u.idexpression((r: any) => r.name(template("f", (e: any) => e.packexpansion((p: any) => p.pattern(T_))))), "f<T...>"],
      // C++'s binary operators, several indices, designated initializers
      [CPPE.binary("<=>", x, y), "x <=> y"], [CPPE.binary("->*", CPPE.binary(".*", x, y), z), "x.*y->*z"],
      [CPPE.subscript(x, 1n, 2n), "x[1, 2]"],
      [(u: any) => u.initializerlist((r: any) => r.items([(i: any) => i.designatedinitializer((d: any) => d.designators([(g: any) => g.fielddesignator((f: any) => f.name((j: any) => j.spelling("y")))]).initializer(
          (n: any) => n.initializerlist((li: any) => li.items([two]))))])), "{.y{2}}"],
      // C++'s specifiers and declarators
      [(u: any) => u.sizeofexpression((r: any) => r.operand((t: any) => t.typeid((i: any) => i.specifiers([
          (s: any) => s.typenamespecifier((n: any) => n.name(qualified("T", "type"))), (s: any) => s.placeholdertypespecifier((p: any) => p.decltype(true)),
          (s: any) => s.placeholdertypespecifier((p: any) => p.constraint(qualified("std", "integral"))), (s: any) => s.decltypespecifier((d: any) => d.expression(x))])))),
       "sizeof(typename T::type decltype(auto) std::integral auto decltype(x))"],
      [(u: any) => u.sizeofexpression((r: any) => r.operand((t: any) => t.typeid((i: any) => i.specifiers([(s: any) => s.classspecifier((c: any) => c.key("struct").name(named("S")).final(true).bases([
          (b: any) => b.access("public").virtual(true).type(named("B")), (b: any) => b.attributes([attribute("a")]).type(named("C")).pack(true)]).body(true)),
          (s: any) => s.enumspecifier((e: any) => e.key("enum class").name(named("E")).base(intType).body(true))])))),
       "sizeof(struct S final : virtual public B, [[a]] C... {} enum class E : int {})"],
      [(u: any) => u.sizeofexpression((r: any) => r.operand((t: any) => t.typeid((i: any) => i.specifiers([int_]).declarator((d: any) => d.referencedeclarator((e: any) => e.rvalue(true).declarator(
          (g: any) => g.pointerdeclarator((p: any) => p.scope(named("S")).qualifiers([(q: any) => q.keyword("const")])))))))), "sizeof(int&&S::*const)"],
      [(u: any) => u.sizeofexpression((r: any) => r.operand((t: any) => t.typeid((i: any) => i.specifiers([(s: any) => s.placeholdertypespecifier((h: any) => h)]).declarator(
          (d: any) => d.functiondeclarator((f: any) => f.declarator((e: any) => e.referencedeclarator((g: any) => g.attributes([attribute("r")]))).parameters([
              (p: any) => p.parameterdeclaration((q: any) => q.attributes([attribute("a")]).this_keyword(true).specifiers([(s: any) => s.placeholdertypespecifier((h: any) => h)]).declarator(
                  (e: any) => e.iddeclarator((g: any) => g.name(named("self"))))),
              (p: any) => p.parameterdeclaration((q: any) => q.specifiers([int_]).declarator((e: any) => e.iddeclarator((g: any) => g.name(named("n")))).default(one))]).qualifiers([
              (q: any) => q.keyword("const")]).ref_qualifier("&").exception((e: any) => e.throwspecifier((h: any) => h.types([intType]))).trailing_return(intType)))))),
       "sizeof(auto(&[[r]])([[a]] this auto self, int n = 1) const & throw(int) -> int)"],
      [(u: any) => u.sizeofexpression((r: any) => r.operand((t: any) => t.typeid((i: any) => i.specifiers([
          (s: any) => s.standardattributespecifier((a: any) => a.using_namespace("gnu").attributes([(t2: any) => t2.name("cold"), (t2: any) => t2.name("hot").pack(true)])),
          (s: any) => s.alignasspecifier((a: any) => a.keyword("alignas").operand(T_).pack(true)), int_])))),
       "sizeof([[using gnu: cold, hot...]] alignas(T...) int)"],
  ] as [unknown, string][]) assert(cppText(spec) === text, `${text}: ${cppText(spec)}`);
  // Without a declarator, a lambda has no parameter list (`[] {}`); an empty one is held and read back (mbse-schemas 0.26.9)
  const emptyDeclarator = term((u: any) => u.lambdaexpression((r: any) => r.declarator((d: any) => d)), CPPE.DIALECT);
  assert(roundTrips(emptyDeclarator) && CppDialect.Text.ToText(emptyDeclarator) === "[]() {}");
  assert((F.form(emptyDeclarator).attributes.get("declarator") as F.Part).kind === "lambdadeclarator");
}

<a id="DIA-26"></a>
## DIA-26 · C and C++ are complete: each named node type of tree-sitter-c and tree-sitter-cpp, kept as data, is a kind or a helper of its family's dialects (its expressions, its types, C's preprocessor), or listed as not an expression with its reason

In [ ]:
import { readFileSync as readTreeSitter } from "node:fs";

{
  const CPP = CppDialect.Expressions.DIALECT;
  const [CT, XT, CPRE] = [CTypes.Expressions.DIALECT, CppTypes.Expressions.DIALECT, CPreprocessor.Expressions.DIALECT];
  const branches = (name: string): string[] => store.registered(name).branches.map((b: any) => b.name);
  for (const [file, count, outside, dialects] of [
    ["c-tree-sitter.json", 125, ["gnu_asm_expression"], { Expressions: C.DIALECT, Types: CT, Preprocessor: CPRE }],
    ["cpp-tree-sitter.json", 206, ["generic_expression", "gnu_asm_expression", "requires_clause"],
      { Expressions: CPP, Types: XT, Preprocessor: CPRE }], // C++ reads C's preprocessor
  ] as [string, number, string[], Record<string, F.Dialect>][]) {
    const grammar = globalThis.JSON.parse(readTreeSitter(`../../grammars/${file}`, "utf8")); // the grammar's node types, as data
    const mapped: Record<string, Set<string>> = Object.fromEntries(Object.keys(dialects).map((name) => [name, new Set<string>()]));
    for (const node of grammar.nodes) { // each a kind or a helper of one of its family's dialects, or both, or not an expression
      const ways = ["kinds", "helpers", "not"].filter((way) => way in node);
      assert(ways.length > 0 && (!ways.includes("not") || ways.length === 1), node.node);
      assert(Object.keys(node).every((key) => key === "node" || key === "supertypes" || ways.includes(key)), node.node);
      assert(!("not" in node) || (typeof node.not === "string" && node.not !== ""), node.node);
      for (const way of ["kinds", "helpers"]) {
        for (const item of node[way] ?? []) {
          const [name, tag] = item.split(".");
          const dialect = dialects[name] as F.Dialect;
          assert(dialect !== undefined && tag !== undefined && tag !== "", node.node);
          if (way === "kinds") { // a kind of its dialect's own, a branch of its `Term`
            assert(dialect.kinds().has(tag) && branches(dialect.union).includes(tag), node.node);
            mapped[name]!.add(tag);
          } else { // a value object schema its dialect registers, not a term
            const schema = store.registered(`Expressions.${dialect.path}.${tag}`);
            assert(!schema.ref && !branches(dialect.union).includes(tag.toLowerCase()), node.node);
          }
        }
      }
    }
    assert(grammar.nodes.length === new Set(grammar.nodes.map((node: any) => node.node)).size && grammar.nodes.length === count); // every named node type, once
    for (const [name, dialect] of Object.entries(dialects)) { // each of its dialects' kinds a node type's, but invalid syntax, which none is
      assert(same([...new Set([...mapped[name]!, "error", "missing"])].sort(), [...dialect.kinds().keys()].sort()), `${file} ${name}`);
    }
    assert(same(grammar.nodes.filter((node: any) => (node.supertypes ?? []).includes("expression") && "not" in node).map((node: any) => node.node), outside));
  }
}


<a id="DIA-27"></a>
## DIA-27 · SystemVerilog's Expressions and Types: their kinds and unions, literals and constants, checks, names, binders, holders, and each dialect of the family reading the others' terms

In [ ]:
{
  const [SVP, SVA, SVC, SVPP] = [SVPatterns.Expressions, SVAssertions.Expressions, SVConstraints.Expressions, SVPreprocessor.Expressions];
  const FAMILY: F.Dialect[] = [SV.DIALECT, SVT.DIALECT, SVP.DIALECT, SVA.DIALECT, SVC.DIALECT, SVPP.DIALECT];
  const validate = (spec: unknown, bound: string[] = []): string[] => SV.DIALECT.validate(sv(spec), bound);
  const union = (dialect: F.Dialect): string[] => store.registered(dialect.union).branches.map((b: any) => b.name); // its Term's branches
  const spelled = (text: string) => (i: any) => i.spelling(text); // an identifier, of its spelling
  const name = (text: string) => (u: any) => u.nameexpression((n: any) => n.name((i: any) => i.identifier(spelled(text))));
  const literal = (value: unknown, encoded: Record<string, unknown> = {}) => (u: any) => u.literal((r: any) => // a literal, with what its text encodes
    Object.entries(encoded).reduce((b: any, [key, part]) => b[key](part), r.value((p: any) => p.as_scalar(value))));
  const constant = (parts: Record<string, unknown> = {}) => (u: any) => u.constant((r: any) => // a constant, of its parts
    Object.entries(parts).reduce((b: any, [key, part]) => b[key](part), r));
  const names = (set: ReadonlySet<string>): string[] => [...set].sort();
  // Its kinds: mbse-programs' Verilog expressions, named and shaped as it has them, 41 branches with the Preprocessor's
  // macro usage and invalid syntax; each sublanguage's Term lists them beside its own, but the Preprocessor's.
  assert(same(union(SV.DIALECT), [...SV.TAGS, "error", "missing", "macrousage"]) && union(SV.DIALECT).length === 41);
  assert(same(union(SVT.DIALECT), [...SV.TAGS, "error", "missing", ...SVT.TAGS, "macrousage"]));
  assert(same(union(SVP.DIALECT), [...SV.TAGS, "error", "missing", ...SVP.TAGS, "macrousage"]));
  assert(same(union(SVA.DIALECT), [...SV.TAGS, "error", "missing", ...SVA.TAGS, "distexpression", "macrousage"])); // Constraints' too
  assert(same(union(SVC.DIALECT), [...SV.TAGS, "error", "missing", ...SVC.TAGS, "macrousage", ...SVPP.DIRECTIVES, "comment"]));
  assert(same(union(SVPP.DIALECT), [...SVPP.TAGS, "error", "missing"]) && same([...SV.DIALECT.kinds().keys()], [...SV.TAGS, "error", "missing"]));
  const kinds = new Map<string, any>(store.registered(SVT.DIALECT.union).branches.map((b: any) => [b.name, b.schema])); // each schema in every union that holds it
  const own = new Map<string, any>(store.registered(SV.DIALECT.union).branches.map((b: any) => [b.name, b.schema]));
  assert(SV.TAGS.every((tag) => kinds.get(tag) === own.get(tag)));
  assert(same(store.registered("Expressions.SystemVerilog.Expressions.Name").branches.map((b: any) => b.name), [
    "identifier", "scopedname", "localname", "parameterizedname", "interfacetypename", "interfacemethodname", "unitname",
    "superexpression"]));
  assert(same(store.registered("Expressions.SystemVerilog.Types.Dimension").branches.map((b: any) => b.name), [
    "rangedimension", "sizedimension", "unsizeddimension", "associativedimension", "queuedimension"]));
  assert(same([...kinds.get("binaryexpression").properties.keys()], ["left", "operator", "attributes", "right"])); // in mbse-programs' order
  assert(equal(F._parts(kinds.get("callexpression")), [["callee"], ["arguments"], ["attributes"]]));
  assert(kinds.get("castexpression").properties.get("type").schema === store.registered(SVT.DIALECT.union)); // a sublanguage's position
  assert(kinds.get("callexpression").properties.get("arguments").schema.element === store.registered(SVA.DIALECT.union));
  assert(kinds.get("matchesexpression").properties.get("pattern").schema === store.registered(SVP.DIALECT.union));
  assert(kinds.get("randomizewithexpression").properties.get("items").schema.element === store.registered(SVC.DIALECT.union));
  assert(kinds.get("structtype").properties.get("members").schema.element === store.registered("Expressions.SystemVerilog.Types.StructMember"));
  // Literals (Q3, "(b)"): `literal`, the framework's refined, holds what a scalar schema can, with what its text encodes;
  // `constant` what none can: `null`, a fill, or a 4-state vector, exactly one of them.
  const Literal = kinds.get("literal");
  assert(Literal.refines === store.registered("Expressions.Framework.Literal") && same([...Literal.properties.keys()].slice(3), [
    "base", "signed", "separators", "unit", "triple"]) && same(SV.SPELLING, ["text", "problems", "separators", "triple"]));
  assert(same([...kinds.get("constant").properties.keys()], ["singleton", "fill", "bits", "size", "signed", "base", "text", "problems"]));
  const literals: [unknown, string[]][] = [
    [literal(255n, { base: "h" }), []], [literal(10.0, { unit: "ns" }), []], [literal("s", { triple: true }), []],
    [literal(1n, { base: "q" }), ["a literal's base must be b, o, d or h, got 'q'"]],
    [literal(10n, { unit: "ns" }), ["a time literal's unit must be s, ms, us, ns, ps or fs, its value a float, got 'ns'"]],
    [literal(1.0, { unit: "ks" }), ["a time literal's unit must be s, ms, us, ns, ps or fs, its value a float, got 'ks'"]],
    [constant({ fill: "1" }), []], [constant({ singleton: "null" }), []], [constant({ bits: "1x0z" }), []], [constant({ bits: "10", size: 2n }), []],
    [constant(), ["a constant holds a singleton, a fill or bits, exactly one of them"]],
    [constant({ fill: "1", bits: "1" }), ["a constant holds a singleton, a fill or bits, exactly one of them"]],
    [constant({ text: "8'hzq", problems: ["'q' is not a hexadecimal digit"] }), ["'q' is not a hexadecimal digit"]],
    [constant({ singleton: "nil" }), ["a constant's singleton must be null, got 'nil'"]],
    [constant({ fill: "2" }), ["a constant's fill must be 0, 1, x or z, got '2'"]],
    [constant({ bits: "10", size: 3n }), ["a constant's size must be its number of bits, 2, got 3"]],
  ];
  for (const [spec, problems] of literals) assert(same(validate(spec), problems), globalThis.JSON.stringify([problems, validate(spec)]));
  // mbse-programs' checks, with its messages, within a kind and its helpers alike; and a choice among a few.
  const x = name("x");
  const checks: [unknown, string[]][] = [
    [name("2x"), ["'2x' is not an identifier"]], [name("\\a+b"), []], [name("$x"), ["'$x' is not an identifier"]],
    [name("\\"), ["'\\\\' is not an identifier"]], [name(""), ["nameexpression '' is not bound"]], // an empty one, no check
    [SV.call("$", x), ["'$' is not a system task or function name"]],
    [(u: any) => u.systemcall((r: any) => r.name("")), ["a systemcall needs a name"]],
    [(u: any) => u.rangeselect((r: any) => r.value(x).left(SV.literal(1n)).operator("::").right(SV.literal(0n))), ["RangeSelect.operator cannot be '::'"]],
    [(u: any) => u.valuerange((r: any) => r.left(SV.literal(1n)).operator("+-").right(SV.literal(2n))), ["ValueRange.operator cannot be '+-'"]],
    [(u: any) => u.streamingconcatenation((r: any) => r.operator("<>").items([(i: any) => i.expression(x)])), ["StreamingConcatenation.operator cannot be '<>'"]],
    [(u: any) => u.streamingconcatenation((r: any) => r.operator("<<").items([(i: any) => i.expression(x).left(SV.literal(0n)).operator(":")])),
      ["a StreamItem's `with` has a left, and an operator and a right, or neither"]],
    [(u: any) => u.streamingconcatenation((r: any) => r.operator("<<").items([(i: any) => i.expression(x).left(SV.literal(0n)).operator("::").right(SV.literal(1n))])),
      ["StreamItem.operator cannot be '::'"]],
    [(u: any) => u.assignmentpattern((r: any) => r.count(SV.literal(2n)).items([(i: any) => i.patternitem((p: any) => p.value(x))])),
      ["a repeated AssignmentPattern has positional items only"]],
    [(u: any) => u.predicateexpression((r: any) => r.conditions([x])), ["a PredicateExpression has two conditions or more"]],
    [(u: any) => u.newexpression((r: any) => r.defaulted(true)), ["a defaulted NewExpression is super.new(default)"]],
    [(u: any) => u.newexpression((r: any) => r.scope((s: any) => s.superexpression((e: any) => e)).defaulted(true).arguments([x])),
      ["a defaulted NewExpression is super.new(default)"]],
    [(u: any) => u.binaryexpression((r: any) => r.left(x).operator("+").attributes([(a: any) => a.specs([(s: any) => s.name(spelled("9"))])]).right(x)),
      ["'9' is not an identifier"]],
    [(u: any) => u.macrousage((r: any) => r.name(spelled("1W"))), ["'1W' is not an identifier"]],
    [SV.cast((t: any) => t.integervectortype((v: any) => v.keyword("int")), x), ["type: IntegerVectorType.keyword cannot be 'int'"]],
    [SV.cast((t: any) => t.integeratomtype((v: any) => v.keyword("int").signing("both")), x), ["type: IntegerAtomType.signing cannot be 'both'"]],
    [SV.cast((t: any) => t.integeratomtype((v: any) => v), x), ["type: an integeratomtype needs a keyword"]],
    [SV.cast((t: any) => t.structtype((s: any) => s.keyword("struct").qualifier("tagged")), x), ["type: a StructType with a qualifier is a union"]],
    [SV.cast((t: any) => t.structtype((s: any) => s.keyword("union").members([(m: any) => m.random("maybe").type(
      (y: any) => y.keywordtype((k: any) => k.keyword("string"))).declarators([(d: any) => d.name(spelled("1a"))])])), x),
      ["type: StructMember.random cannot be 'maybe'", "type: '1a' is not an identifier"]],
    [SV.cast((t: any) => t.enumtype((e: any) => e.members([(m: any) => m.name(spelled("A")).right(SV.literal(1n))])), x), ["type: an EnumMember with a right has a left"]],
    [SV.cast((t: any) => t.integervectortype((v: any) => v.keyword("logic").dimensions([(d: any) => d.rangedimension((r: any) => r.left(SV.literal(1n)))])), x),
      ["type: an integervectortype needs a dimensions 0 right"]],
  ];
  for (const [spec, problems] of checks) { // an escaped one, bound too
    assert(same(validate(spec, ["x", "\\a+b"]), problems), globalThis.JSON.stringify([problems, validate(spec, ["x", "\\a+b"])]));
  }
  // Names (Q4, "(b)"): a name used as a value is a nameexpression holding its Name; an identifier's must be bound, any
  // other name is resolved by the scope alone, the terms it holds (a parameterized name's parameters) checked in scope.
  const scoped = (scope: unknown, inner: unknown) => (u: any) => u.nameexpression((n: any) => n.name((s: any) => s.scopedname((c: any) => c.scope(scope).name(inner))));
  const p_x = scoped((y: any) => y.identifier(spelled("p")), (y: any) => y.identifier(spelled("x"))); // p::x
  const c_w = scoped((y: any) => y.parameterizedname((q: any) => q.name(spelled("c")).parameters([name("w")])), (y: any) => y.identifier(spelled("x")));
  assert(validate(p_x).length === 0 && F.nameOf(sv(p_x)) === null && F.nameOf(sv(x)) === "x" && same(names(SY.free(sv(c_w), SV.DIALECT)), ["w"]));
  assert(same(validate(c_w), ["name scope argument 0: nameexpression 'w' is not bound"]) && validate(c_w, ["w"]).length === 0);
  assert(same(validate((u: any) => u.nameexpression((n: any) => n)), ["a nameexpression needs a name"]));
  assert(!SV.DIALECT.kinds().has("variable")); // SystemVerilog writes a name only as a nameexpression, as C++ an idexpression
  // Binders refine Let (Q8, "(a)"): an array method's `with` binds its iterator, its call's argument or `item`, within its
  // call and its `with` expression; a `matches` binds its pattern's variables (Patterns' to write) in its conditional's
  // consequence and the conditions after it.
  const WITH = SV.DIALECT.kinds().get("arraymethodwithexpression")!;
  const summed = sv(SV.iterate(x, "sum", "q", SV.binary("+", SV.identifier("q"), name("r"))));
  assert(same(names(SY.free(summed, SV.DIALECT)), ["r", "x"]) && validate(SV.iterate(x, "sum", "q", SV.identifier("q")), ["x"]).length === 0);
  const implicit = (u: any) => u.arraymethodwithexpression((r: any) => r.value(SV.method(x, "sum")).body(name("item"))); // item
  assert(validate(implicit, ["x"]).length === 0 && same(names(SY.free(sv(implicit), SV.DIALECT)), ["x"]) && F.isBinder(F.kindOf(summed)[1].as_schema()));
  assert(WITH.role === F.BINDING && F.nameOf(summed) === null); // Let's name unset
  assert(equal(WITH.scopes!(F.kindOf(sv((u: any) => u.arraymethodwithexpression((r: any) => r.body(SV.literal(1n)))))[1]).map(names), [[], []])); // no call: nothing bound
  assert(SV.DIALECT.infer(summed, { x: D.Anything, r: D.Anything }) === D.Anything);
  const scopedIterator = WITH.scopes!(F.kindOf(sv((u: any) => u.arraymethodwithexpression(
    (r: any) => r.value(SV.method(x, "sum", p_x)).body(name("item")))))[1]);
  assert(equal(scopedIterator.map(names), [["item"], ["item"]])); // a call's argument that is no identifier names no iterator
  assert(String(SV.DIALECT.infer(sv(constant({ bits: "x01" })))) === "logic [31:0]" && String(SV.DIALECT.infer(sv(constant({ bits: "x01", size: 3n })))) === "logic [2:0]");
  assert(SV.DIALECT.infer(sv(literal(10.0, { unit: "ns" }))) === D.Anything && SV.DIALECT.infer(sv(SV.binary("<<=", x, 1n)), { x: D.Anything }) === D.Anything);
  const matched = (pattern: unknown) => (u: any) => u.matchesexpression((m: any) => m.value(x).pattern(pattern));
  const choice = sv((u: any) => u.conditionalexpression((c: any) => c.condition(matched(SV.literal(1n))).consequence(name("v")).alternative(SV.literal(0n))));
  assert(SVP.boundNames(F.form(F.form(choice).arguments[0]).arguments[1]).size === 0); // a constant pattern binds none
  assert(same(names(SY.free(sv(SV.conditional(x, name("v"), 0n)), SV.DIALECT)), ["v", "x"])); // a plain condition binds none
  assert(same(names(SY.free(choice, SV.DIALECT)), ["v", "x"]) && same(names(SY.free(sv((u: any) => u.predicateexpression((p: any) => p.conditions([
    matched(SV.literal(1n)), name("v")]))), SV.DIALECT)), ["v", "x"]));
  assert(equal(SV.DIALECT.kinds().get("conditionalexpression")!.scopes!(F.kindOf(sv((u: any) => u.conditionalexpression(
    (c: any) => c.consequence(x).alternative(x))))[1]).map(names), [[], [], []])); // no condition: nothing matched
  assert(same(names(SY.free(sv((u: any) => u.conditionalexpression((c: any) => c.condition((p: any) => p.predicateexpression((q: any) => q.conditions([
    matched(SV.literal(1n)), x]))).consequence(name("v")).alternative(SV.literal(0n)))), SV.DIALECT)), ["v", "x"])); // a `&&&`'s matches
  // Each dialect of the family reads the others' terms, where its positions hold them: a cast's type is a Types term,
  // a dimension's bound an Expressions term, a call's argument an Assertions term; another's is refused as before.
  const cast = sv(SV.cast((t: any) => t.integervectortype((v: any) => v.keyword("logic").dimensions([
    (d: any) => d.rangedimension((r: any) => r.left(name("w")).right(SV.literal(0n)))])), x));
  const typed = F.form(cast).arguments[0];
  assert(F.dialectOf(typed) === SVT.DIALECT.union && F.dialectOf(F.form(typed).arguments[0]) === SV.DIALECT.union);
  assert(same(SV.DIALECT.validate(cast, ["x"]), ["type: dimensions 0 left: nameexpression 'w' is not bound"]));
  assert(same(SVT.DIALECT.validate(typed), ["dimensions 0 left: nameexpression 'w' is not bound"]) && same(names(SY.free(cast, SV.DIALECT)), ["w", "x"]));
  const call_ = sv(SV.call("f", x));
  assert(F.dialectOf(F.form(call_).arguments[1]) === SVA.DIALECT.union && SVA.DIALECT.validate(F.form(call_).arguments[1], ["x"]).length === 0);
  assert(FAMILY.every((d) => { const reads = d.reads(), others = FAMILY.filter((o) => o !== d); return reads.length === others.length && reads.every((r, i) => r === others[i]); }));
  assert(SVA.DIALECT.kind(F.form(call_).arguments[1])[2] === SV.DIALECT.kinds().get("nameexpression")); // a kind its Term lists
  assert(SV.DIALECT.infer(cast, { x: D.Anything, w: D.Anything }) === D.Anything && SVT.DIALECT.infer(typed, { w: D.Anything }) === D.Anything);
  raises(ValueError, () => P.DIALECT.kind(typed), "a SystemVerilog Types integervectortype is not a Python3 term");
  assert(same(P.DIALECT.validate(typed), ["a SystemVerilog Types integervectortype is not a Python3 term"]));
  assert(SVP.DIALECT.kindNamed("structtype") === SVT.DIALECT.kinds().get("structtype")); // a kind it reads, which its Term lists not:
  raises(Error, () => store.transaction(() => store.Expressions.SystemVerilog.Patterns.Expression( // the schema refuses it
    (e: any) => e.root(SVP.DIALECT.make(new F.Form("structtype", { keyword: "struct" }))))), "'structtype' is not a member of a union value");
  raises(ValueError, () => SVPP.DIALECT.kindNamed("nope"), "SystemVerilog Preprocessor has no kind 'nope'");
  // Each dialect its holder (Q2, "(a)"), refining the framework's Expression, its root narrowed to its Term, holding its
  // kinds and those its Term lists alone.
  for (const dialect of FAMILY) {
    const Holder = store.registered(`Expressions.${dialect.path}.Expression`);
    assert(Holder.refines === store.registered("Expressions.Framework.Expression") && Holder.properties.get("root").schema === store.registered(dialect.union));
  }
  const held = store.transaction(() => store.Expressions.SystemVerilog.Types.Expression((e: any) => e.root((t: any) => t.integeratomtype((a: any) => a.keyword("int")))));
  assert(SVT.DIALECT.validate(held).length === 0 && SVText.ToText(held) === "int" && F.dialectOf(F.term(held)) === SVT.DIALECT.union);
  raises(Error, () => store.transaction(() => store.Expressions.SystemVerilog.Expressions.Expression( // the schema refuses a kind its root's union lacks
    (e: any) => e.root((t: any) => t.integeratomtype((a: any) => a.keyword("int"))))));
}

<a id="DIA-28"></a>
## DIA-28 · SystemVerilog's printing: mbse-programs' printer ported, its precedence, layout and spellings, for every kind of Expressions and Types and the Preprocessor's macro usage, from any holder of the family

In [ ]:
{
  // Printing ports mbse-programs' printer: IEEE 1800's precedence, its layout and spellings, for every kind of Expressions
  // and Types and the Preprocessor's macro usage, from any holder of the family; the samples' every kind (FRM-02).
  const SVPP = SVPreprocessor.Expressions;
  const svtext = (spec: unknown): string => SVText.ToText(sv(spec));
  const spelled = (text: string) => (i: any) => i.spelling(text); // an identifier, of its spelling
  const name = (text: string) => (u: any) => u.nameexpression((n: any) => n.name((i: any) => i.identifier(spelled(text))));
  const literal = (value: unknown, encoded: Record<string, unknown> = {}) => (u: any) => u.literal((r: any) => // a literal, with what its text encodes
    Object.entries(encoded).reduce((b: any, [key, part]) => b[key](part), r.value((p: any) => p.as_scalar(value))));
  const constant = (parts: Record<string, unknown> = {}) => (u: any) => u.constant((r: any) => // a constant, of its parts
    Object.entries(parts).reduce((b: any, [key, part]) => b[key](part), r));
  const [x, y] = [name("x"), name("y")];
  const attributed = (...names: string[]) => [(a: any) => a.specs(names.map((n) => (s: any) => s.name(spelled(n))))]; // (* a, b *)
  const vector_type = (keyword: string, ...dimensions: unknown[]) => (t: any) => t.integervectortype((v: any) => v.keyword(keyword).dimensions(dimensions));
  const packed = (left: bigint, right: bigint) => (d: any) => d.rangedimension((r: any) => r.left(SV.literal(left)).right(SV.literal(right)));
  const cases: [unknown, string][] = [
      // precedence: an assignment, a tagged value, a `matches` and a `&&&` bind loosest, parenthesized in an operand
      [SV.binary("+", (u: any) => u.assignmentexpression((a: any) => a.target(x).operator("=").value(SV.literal(1n))), y), "(x = 1) + y"],
      [(u: any) => u.assignmentexpression((a: any) => a.target(x).operator("<<=").value(SV.binary("+", y, 1n))), "x <<= y + 1"],
      [SV.unary("-", (u: any) => u.incrementexpression((i: any) => i.operator("++").operand(x))), "-(++x)"],
      [(u: any) => u.incrementexpression((i: any) => i.operator("--").postfix(true).operand(SV.binary("+", x, y))), "(x + y)--"],
      [(u: any) => u.incrementexpression((i: any) => i.operator("++").operand(x).attributes(attributed("a"))), "++ (* a *) x"],
      [(u: any) => u.incrementexpression((i: any) => i.operator("++").postfix(true).operand(x).attributes(attributed("a"))), "x (* a *)++"],
      [(u: any) => u.unaryexpression((i: any) => i.operator("!").operand(x).attributes(attributed("a", "b"))), "! (* a, b *) x"],
      [(u: any) => u.binaryexpression((i: any) => i.left(x).operator("+").attributes(attributed("a")).right(y)), "x + (* a *) y"],
      [(u: any) => u.conditionalexpression((c: any) => c.condition(x).attributes(attributed("a")).consequence(SV.conditional(y, 1n, 2n)).alternative(y)),
       "x ? (* a *) (y ? 1 : 2) : y"],
      [(u: any) => u.conditionalexpression((c: any) => c.condition((p: any) => p.predicateexpression((q: any) => q.conditions([x, y]))).consequence(x).alternative(y)),
       "x &&& y ? x : y"],
      [(u: any) => u.taggedexpression((t: any) => t.member(spelled("a")).value((v: any) => v.taggedexpression((w: any) => w.member(spelled("b")).value(SV.binary("+", x, 1n))))),
       "tagged a tagged b (x + 1)"],
      [(u: any) => u.taggedexpression((t: any) => t.member(spelled("None"))), "tagged None"],
      [(u: any) => u.mintypmaxexpression((m: any) => m.min(x).typ(SV.binary("+", x, 1n)).max(y)), "x:x + 1:y"],
      // selects and calls
      [(u: any) => u.rangeselect((r: any) => r.value(SV.binary("+", x, y)).left(x).operator("+:").right(SV.literal(4n))), "(x + y)[x +: 4]"],
      [(u: any) => u.callexpression((c: any) => c.callee(x)), "x()"],
      [(u: any) => u.systemcall((c: any) => c.name("$finish")), "$finish"],
      [(u: any) => u.systemcall((c: any) => c.name("$finish").attributes(attributed("a"))), "$finish (* a *) ()"],
      [SV.call("$bits", vector_type("bit", packed(3n, 0n))), "$bits(bit [3:0])"],
      [(u: any) => u.newexpression((n: any) => n), "new"], [(u: any) => u.newexpression((n: any) => n.scope((s: any) => s.superexpression((e: any) => e)).arguments([x])), "super.new(x)"],
      [(u: any) => u.newarrayexpression((n: any) => n.size(x)), "new[x]"],
      [(u: any) => u.randomizewithexpression((r: any) => r.call(SV.method(x, "randomize")).items([])), "x.randomize() with {}"],
      [(u: any) => u.randomizewithexpression((r: any) => r.call(SV.method(x, "randomize")).items([SV.binary(">", x, 0n), (i: any) => i.macrousage((m: any) => m.name(spelled("C")))])),
       "x.randomize() with { x > 0; `C }"],
      [(u: any) => u.streamingconcatenation((s: any) => s.operator(">>").items([(i: any) => i.expression(x), (i: any) => i.expression(y).left(SV.literal(0n))])),
       "{>>{x, y with [0]}}"],
      [(u: any) => u.streamingconcatenation((s: any) => s.operator("<<").slice(SV.literal(4n)).items([(i: any) => i.expression(x).left(SV.literal(0n)).operator(":").right(SV.literal(3n))])),
       "{<< 4 {x with [0:3]}}"],
      [(u: any) => u.assignmentpattern((a: any) => a.type(vector_type("logic", packed(1n, 0n))).count(SV.binary("+", x, 1n)).items([x, y])), "logic [1:0]'{(x + 1){x, y}}"],
      [(u: any) => u.assignmentpattern((a: any) => a.items([(i: any) => i.patternitem((p: any) => p.key((k: any) => k.integeratomtype((t: any) => t.keyword("int"))).value(SV.literal(0n)))])),
       "'{int: 0}"],
      [SV.cast(SV.binary("+", x, 1n), y), "(x + 1)'(y)"], [SV.cast((t: any) => t.typereference((r: any) => r.operand(x)), y), "type(x)'(y)"],
      [(u: any) => u.namedconnection((c: any) => c.name(spelled("a"))), ".a()"],
      [SV.inside(x, (u: any) => u.valuerange((r: any) => r.left(SV.literal(1n)).operator("+/-").right(SV.literal(2n)))), "x inside {[1 +/- 2]}"],
      [(u: any) => u.streamingconcatenation((s: any) => s.operator("<<").items([(i: any) => i.expression(x).left(SV.literal(0n)).operator("+:").right(SV.literal(4n))])),
       "{<<{x with [0 +: 4]}}"],
      // literals in their spelling, or their value's default one; constants in their base where it writes their bits
      [literal(255n, { base: "h" }), "'hff"], [literal(5n, { base: "d", signed: true }), "'sd5"], [literal(-5n, { base: "o" }), "-'o5"], [literal(5n, { base: "b" }), "'b101"],
      [literal(10.0, { unit: "ns" }), "10ns"], [literal(1.5, { unit: "us" }), "1.5us"], [literal('a"\n', { triple: true }), '"""a"\n"""'],
      [literal(8n, { text: "4'h8" }), "4'h8"], [constant({ fill: "z" }), "'z"], [constant({ singleton: "null" }), "null"],
      [constant({ bits: "xxxxzzzz", base: "h" }), "'hxz"], [constant({ bits: "x1", base: "h" }), "'bx1"], [constant({ bits: "1010", size: 4n, base: "d" }), "4'd10"],
      [constant({ bits: "0x", size: 2n, base: "d" }), "2'b0x"], [constant({ text: "8'hFF", bits: "11111111", size: 8n }), "8'hFF"],
      [SV.unary("-", literal(255n, { base: "h" })), "-'hff"], [SV.unary("-", literal(-5n, { base: "o" })), "-(-'o5)"],
  ];

  for (const [spec, source] of cases) assert(svtext(spec) === source, globalThis.JSON.stringify([source, svtext(spec)]));
  // Types: data types, their dimensions and members, on one line, written from a Types holder too
  const typetext = (spec: unknown): string => SVText.ToText(term(spec, SVT.DIALECT));
  const types: [unknown, string][] = [
      [(t: any) => t.namedtype((n: any) => n.name((i: any) => i.identifier(spelled("t")))), "t"],
      [vector_type("reg"), "reg"], [(t: any) => t.integeratomtype((a: any) => a.keyword("int").signing("unsigned")), "int unsigned"],
      [(t: any) => t.keywordtype((k: any) => k.keyword("chandle")), "chandle"], [(t: any) => t.implicittype((i: any) => i.dimensions([packed(7n, 0n)])), "[7:0]"],
      [(t: any) => t.implicittype((i: any) => i.signing("unsigned").dimensions([packed(7n, 0n)])), "unsigned [7:0]"],
      [(t: any) => t.virtualinterfacetype((v: any) => v.interface(spelled("bus"))), "virtual bus"],
      [(t: any) => t.virtualinterfacetype((v: any) => v.interface_keyword(true).interface(spelled("bus"))), "virtual interface bus"],
      [(t: any) => t.structtype((s: any) => s.keyword("struct")), "struct {  }"],
      [(t: any) => t.structtype((s: any) => s.keyword("union").qualifier("soft").members([(m: any) => m.type(vector_type("bit")).declarators([
          (d: any) => d.name(spelled("a")), (d: any) => d.name(spelled("b")).dimensions([packed(1n, 0n)])])]).dimensions([packed(3n, 0n)])),
       "union soft { bit a, b[1:0]; } [3:0]"],
      [(t: any) => t.enumtype((e: any) => e.members([(m: any) => m.name(spelled("A")).value(SV.literal(1n))])), "enum {A = 1}"],
      [(t: any) => t.namedtype((n: any) => n.name((i: any) => i.parameterizedname((q: any) => q.name(spelled("c")).parameters([
          SV.literal(8n), (p: any) => p.integeratomtype((a: any) => a.keyword("int")), (p: any) => p.namedconnection((c: any) => c.name(spelled("T")).implicit(true))])))),
       "c #(8, int, .T)"],
      [(t: any) => t.namedtype((n: any) => n.name((i: any) => i.parameterizedname((q: any) => q.name(spelled("c"))))), "c"], // no parameters
      [(t: any) => t.literal((r: any) => r.value((p: any) => p.as_scalar(8n))), "8"], // Types lists Expressions' kinds
  ];

  for (const [spec, source] of types) assert(typetext(spec) === source, globalThis.JSON.stringify([source, typetext(spec)]));
  // Invalid syntax is written as written, wherever it stands, by the dialect whose union holds it; one with no text has
  // nothing to print, refused by name.
  assert(svtext(SV.cast((t: any) => t.error((e: any) => e.text("int int")), x)) === "int int'(x)" && typetext((t: any) => t.missing((m: any) => m)) === "");
  raises(ValueError, () => svtext(SV.cast((t: any) => t.error((e: any) => e), x)), "SystemVerilog Types error has no text to print");
  raises(ValueError, () => svtext((u: any) => u.literal((r: any) => r.problems(["bad"]))), "SystemVerilog literal has no value to print");
  raises(ValueError, () => svtext(constant({ problems: ["bad"] })), "SystemVerilog constant has no value to print");
  // A macro usage, the Preprocessor's, printed from its holder.
  assert(SVText.ToText(term((u: any) => u.macrousage((m: any) => m.name(spelled("W")).arguments("a, b")), SVPP.DIALECT)) === "`W(a, b)");
}

<a id="DIA-29"></a>
## DIA-29 · SystemVerilog is complete: each of slang's syntax kinds, kept as data, is a kind or a helper of the family's six dialects, or listed as not an expression with its reason

In [ ]:
import { readFileSync as readSlang } from "node:fs";

{
  const grammar = globalThis.JSON.parse(readSlang("../../grammars/systemverilog-slang.json", "utf8")); // slang's syntax kinds, as data
  const DIALECTS = new Map<string, any>([["Expressions", SV], ["Types", SVT], ["Patterns", SVPatterns.Expressions],
    ["Assertions", SVAssertions.Expressions], ["Constraints", SVConstraints.Expressions], ["Preprocessor", SVPreprocessor.Expressions]]);
  const EXEMPT = new Map([["Expressions", ["missing"]], ["Preprocessor", ["disabledtext", "comment"]]]); // no slang syntax kind: a missing token, trivia
  const branches = (name: string): string[] => store.registered(name).branches.map((b: any) => b.name);
  assert(grammar.nodes.length === new Set(grammar.nodes.map((node: any) => node.node)).size && grammar.nodes.length === 533); // every syntax kind, once
  const mapped = new Map([...DIALECTS.keys()].map((name): [string, Set<string>] => [name, new Set()]));
  for (const node of grammar.nodes) { // each a kind or a helper of one of the six, or both, or not an expression, with its reason
    const ways = ["kinds", "helpers", "not"].filter((way) => way in node);
    assert(ways.length > 0 && (!ways.includes("not") || ways.length === 1) && Object.keys(node).length === ways.length + 1, node.node);
    assert(!("not" in node) || (typeof node.not === "string" && node.not !== ""), node.node);
    for (const way of ["kinds", "helpers"]) {
      for (const item of node[way] ?? []) {
        const [dialect, tag] = [item.slice(0, item.indexOf(".")), item.slice(item.indexOf(".") + 1)];
        assert(DIALECTS.has(dialect) && tag !== "" && item.includes("."), node.node);
        mapped.get(dialect)!.add(tag);
        const union = branches(DIALECTS.get(dialect).DIALECT.union);
        if (way === "kinds") { // a branch of its dialect's Term
          assert(union.includes(tag), node.node);
        } else { // a value object schema its dialect registers
          const schema = store.registered(`Expressions.SystemVerilog.${dialect}.${tag}`);
          assert(schema !== null && schema !== undefined && !schema.ref && !union.includes(tag.toLowerCase()), node.node);
        }
      }
    }
  }
  // slang's one kind read as either a kind or a helper, where mbse-programs' reader reads it as either
  assert(same(grammar.nodes.filter((node: any) => "kinds" in node && "helpers" in node).map((node: any) => node.node), ["IdentifierName", "ScopedName"]));
  for (const [name, module] of DIALECTS) { // each dialect's kinds and helpers are the ones slang's kinds map to, but those
    const own = new Set<string>([...module.TAGS, ...module.HELPERS, ...(name === "Expressions" ? ["error", "missing"] : [])]); // no syntax kind has
    const covered = new Set([...mapped.get(name)!, ...(EXEMPT.get(name) ?? [])]);
    assert(same([...covered].sort(), [...own].sort()), name);
  }
}

<a id="DIA-30"></a>
## DIA-30 · Every binder refines `Let`: Let's `name`, `value` and `body` first, renamed where the grammar named them otherwise, its role and scopes saying what it binds

In [ ]:
{
  // Every kind that binds names refines the framework's `Let`, Let's `name`, `value` and `body` first, renamed to them where
  // the grammar named them otherwise; those it has no name or term for left unset, and optional.
  const E = Patterns.Expressions;
  const CPPE = CppDialect.Expressions;
  const SVC = SVConstraints.Expressions;
  const LET: any = store.registered("Expressions.Framework.Let");
  const COMPREHENSION = ["name", "value", "body", "generators"];
  const BINDERS = new Map<F.Dialect, Record<string, string[]>>([ // each binder of every dialect, its schema's properties in order
    [E.DIALECT, { quantifier: ["name", "value", "body", "quantifier"] }], // its collection Let's value
    [M.DIALECT, { arrayfun: ["name", "value", "body"], import: ["name", "value", "body"] }], // an array Let's value
    [X.DIALECT, { map: ["name", "value", "body"] }],
    [P.DIALECT, { [P.LISTCOMP]: COMPREHENSION, [P.SETCOMP]: COMPREHENSION, [P.GENERATOREXP]: COMPREHENSION, // an element Let's body
      [P.DICTCOMP]: ["name", "value", "body", "key", "generators"], [P.LAMBDA]: ["name", "value", "body", "args"],
      [P.NAMEDEXPR]: ["name", "value", "body", "target"], [P.IMPORT]: ["name", "value", "body", "module", "alias"],
      [P.IMPORTFROM]: ["name", "value", "body", "module", "alias"] }],
    [TS.DIALECT, { arrowfunctionexpression: ["name", "value", "body", "isAsync", "typeParameters", "params", "returnType"],
      functionexpression: ["name", "value", "body", "isAsync", "generator", "id", "typeParameters", "params", "returnType"],
      tsemptybodyfunctionexpression: ["name", "value", "body", "isAsync", "generator", "typeParameters", "params", "returnType"],
      classexpression: ["name", "value", "body", "decorators", "id", "typeParameters", "superClass", "superTypeArguments",
        "implements"] }],
    [CPPE.DIALECT, { lambdaexpression: ["name", "value", "body", "captures", "template_parameters", "template_requires",
      "attributes", "declarator"], requiresexpression: ["name", "value", "body", "parameters", "requirements"] }],
    [SV.DIALECT, { arraymethodwithexpression: ["name", "value", "body"] }],
    [SVC.DIALECT, { foreachconstraint: ["name", "value", "body", "variables"] }],
  ]);
  for (const [dialect, binders] of BINDERS) {
    const union = new Map((store.registered(dialect.union).branches as any[]).map((b) => [b.name, b.schema]));
    for (const [tag, properties] of Object.entries(binders)) {
      const schema: any = union.get(tag);
      assert(schema.refines === LET && F.isBinder(schema) && same([...schema.properties.keys()], properties), `${dialect.name()} ${tag}`);
    }
    assert((union.get("let") ?? LET) === LET); // a dialect's own let is the framework's
  }
  // A binder keeps its role, which evaluation and inference read: a quantifier and an import bind by it, not as a let.
  const roles = ([[E.DIALECT, "quantifier"], [M.DIALECT, "arrayfun"], [M.DIALECT, "import"], [X.DIALECT, "map"],
    [P.DIALECT, P.LAMBDA], [P.DIALECT, P.IMPORT]] as [F.Dialect, string][]).map(([d, tag]) => (d.kinds().get(tag) as F.Kind).role);
  assert(same(roles, [F.QUANTIFIER, F.QUANTIFIER, F.IMPORT, F.QUANTIFIER, F.APPLICATION, F.IMPORT]));
  // What each binds, as its scopes or its role say: Patterns' quantifier its name in its body, of the collection's items'
  // domain, giving a bool; MATLAB's arrayfun and Excel's map their name in their body.
  const [xs, x, y] = [E.variable("xs"), E.variable("x"), E.variable("y")];
  const quantified = term(E.quantifier("any", "x", xs, x.gt(y)), E.DIALECT);
  assert(same([...SY.free(quantified, E.DIALECT)].sort(), ["xs", "y"]) && E.DIALECT.validate(quantified, ["xs", "y"]).length === 0);
  assert(same(E.DIALECT.validate(quantified, ["xs"]), ["body: argument 1: variable 'y' is not bound"]));
  assert(E.DIALECT.infer(quantified, { xs: D.Anything, y: D.Anything }).name() === "bool");
  const mapped = term(M.arrayfun("p", M.identifier("xs"), M.binary("+", M.identifier("p"), M.identifier("q"))), M.DIALECT);
  assert(same([...SY.free(mapped, M.DIALECT)].sort(), ["q", "xs"]) && MText.ToText(mapped) === "arrayfun(@(p) p + q, xs)");
  const lambdas = term(X.map_("p", X.name("xs"), X.infix("+", X.name("p"), X.name("q"))), X.DIALECT);
  assert(same([...SY.free(lambdas, X.DIALECT)].sort(), ["q", "xs"]) && XText.ToText(lambdas) === "=MAP(xs, LAMBDA(p, p + q))");
  // An import leaves Let's value empty, its body its last argument: printed after its line, and evaluated within it.
  const imported = term(M.import_("pkg.*", M.identifier("x")), M.DIALECT);
  assert(F.form(imported).arguments[0] === null && same([...SY.free(imported, M.DIALECT)], ["x"])
    && MText.ToText(imported) === "import pkg.*\nx");
  // Python3: a comprehension's element and a dict comprehension's value are its body, after Let's empty value; a lambda's
  // body its parameters bind; a named expression's value; an import's body.
  const n = P.name;
  const listed = term(P.listcomp(n("p"), P.comprehension("p", n("xs"))), P.DIALECT);
  assert(F.form(listed).arguments[0] === null && F.nameOf(F.form(listed).arguments[1]) === "p");
  assert(same([...SY.free(listed, P.DIALECT)], ["xs"]) && PText.ToText(listed) === "[p for p in xs]");
  const keyed = term(P.dictcomp(n("k"), n("v"), P.comprehension(P.tuple_(n("k"), n("v")), n("d"))), P.DIALECT);
  assert(same(F.form(keyed).arguments.slice(1, 3).map((a: any) => F.nameOf(a)), ["v", "k"])); // its value, its body, then its key
  assert(same([...SY.free(keyed, P.DIALECT)], ["d"]) && PText.ToText(keyed) === "{k: v for k, v in d}");
  const fn = term(P.lambda_(P.binop("+", n("a"), n("b")), ["a"]), P.DIALECT);
  assert(same([...SY.free(fn, P.DIALECT)], ["b"]) && PText.ToText(fn) === "lambda a: a + b");
  const walrus = term(P.namedexpr("y", n("x")), P.DIALECT); // it binds in the enclosing scope, which a term does not see
  assert(same([...SY.free(walrus, P.DIALECT)], ["x"]) && F.form(walrus).arguments[1] === null && PText.ToText(walrus) === "(y := x)");
  const floor = term(P.importfrom("math", "floor", P.call(n("floor"), 2.5)), P.DIALECT);
  assert(SY.free(floor, P.DIALECT).size === 0 && PText.ToText(floor) === "from math import floor\nfloor(2.5)");
  assert(P.DIALECT.validate(floor).length === 0 && same(P.DIALECT.validate(term(P.importfrom("math", "floor", n("ceil")), P.DIALECT)), [
    "body: variable 'ceil' is not bound"]));
  // TypeScript5: an arrow's expression body is Let's body, its parameters bound within it.
  const ident = (name: string) => (u: any) => u.identifier((r: any) => r.name(name));
  const arrow = term((u: any) => u.arrowfunctionexpression((r: any) => r.params([ident("a")]).body(
    (b: any) => b.binaryexpression((e: any) => e.left(ident("a")).operator("+").right(ident("b"))))), TS.DIALECT);
  assert(F.form(arrow).arguments[0] === null && same([...SY.free(arrow, TS.DIALECT)], ["b"]));
  assert(TSText.ToText(arrow) === "(a) => a + b");
  // C++: a lambda and a requires expression leave Let's name, value and body unset, a body of statements not part of the term.
  const captured = term((u: any) => u.lambdaexpression((r: any) => r), CPPE.DIALECT);
  assert(same(F.form(captured).arguments.slice(0, 3), [null, null, null]) && CPPE.DIALECT.validate(captured).length === 0);
}